# pawn covers
This notebook shows how to create a pawn cover so that queens can cross the board.

In [ ]:
#| default_exp game/mechanics/coverage

In [ ]:
#| export
import sys
import math
from fastcore.basics import patch
from fasthtml.common import *
from fasthtml.jupyter import *
import pandas as pd
import numpy as np
from dataclasses import dataclass, field
from collections import deque
from monsterui.all import TableT, Card, CardHeader, CardTitle

In [ ]:
#| export
from HexMagic.primitives import MapPath, MapSize, MapRect, MapCord 
from HexMagic.primitives import HexGrid, HexPosition ,  HexRegion, GosperCurve, windy_edge , unique_windy_edge

import numpy as np

from HexMagic.terrain import Terrain
from HexMagic.climate import ClimatePreset, Climate, TerraDemo, TerrainFactory
from HexMagic.geology import Geology, DrainageBasins, SoilSystem
from HexMagic.overlay import  TerrainDisplay, TerrainOverlay, ClimateOverlay, TerraDemo, DrainageBasins, OverlaySpec
from HexMagic.overlay import TerrainDisplay, CreamOverlay, RiverOverlay, ClimateOverlay

In [ ]:
from HexMagic.game.globals import appRoutes,  webMe, globalStore, ensure_user,new_game_page, create_game, create_world, invalidate_cache, showUsers, logging
from HexMagic.game.data import Settlement, Kingdom, Piece, TradeRoute, GameBoard, ActiveGame, CountryFlag, _map_point
from HexMagic.game.data import Piece, PieceType, Instruction, InstructionList, Squad
from HexMagic.game.data import RANK_ORDER 
from HexMagic.game.piece import CountryFlag, DiagramGlyphs,PieceBoard, PieceStep, _move_cost,PieceBoardPlan
from HexMagic.game.piece import SquadPlanOverlay, PieceList, FormationType, SurroundMode, TroopPath, _facing_toward
from HexMagic.game.piece import GroupedPieceList, SquadPlanOverlay, piece_plan_overlay, SquadSymbolOverlay #, SquadVisionOverlay
from HexMagic.game.piece import FormationType, _formation_offsets

from HexMagic.game.piece import pieces_center, StatBar, SquadVisionOverlay, GameParts,SettlementOverlay, KingdomNamesOverlay, PieceOverlay

In [ ]:
from HexMagic.game.food import FoodOverlay, FoodYield, PieceOverlay, PieceDataOverlay, PieceArrowOverlay, SquadSymbolOverlay, food_triage, pre_march_distribute, compose_callbacks

In [ ]:
from HexMagic.game.mechanics.simulator import TurnReport, SimEventType, FoodSimulator, squad_chart, SquadFoodTrailOverlay, STARVE_MULT

In [ ]:
import pyomo.environ as pyo
import optuna

## Food

Clean results:

    Tier-sum ≥ 7 rule holds perfectly — every sum-7 combo passes, sum-6 fails
    Tight margin at sum=7 — only 0.7 surplus (7.7 vs 7.0), so it works but there's no slack. Good tension.
    Solo T7 pawn feeds the queen — strategic chokepoint mechanic works
    Pawn survives on T1 — pawns never starve, even on the worst land
    10× diet ratio — queen is very expensive, which makes escort logistics the core game

The numbers are all round and easy for a player to reason about: "I need my two pawns' tiers to add up to 7." That's a great mental model.

One thing to note: we don't have knight/rook profiles in V2 yet. Worth adding those when you're ready, but for now this is solid for testing the PCR. Want to rerun the route with this profile and get the overlay working?

In [ ]:
#| export
FOOD_PROFILE_V2 = {
    'pawn_diet':       0.7,
    'pawn_harvest':    1.3,
    'pawn_capacity':  14.0,
    'pawn_sight':      4,
    'queen_diet':      7.0,
    'queen_harvest':   0.2,
    'queen_capacity': 24.0,
    'bishop_diet':     3.5,
    'bishop_harvest':  0.7,
    'bishop_capacity': 140.0,
}

def apply_food_profile(pieces, profile=None):
    """Apply food profile to a list of pieces."""
    p = profile or FOOD_PROFILE_V2
    for piece in pieces:
        t = piece.piece_type.name.lower()
        for attr, field in [('diet','diet'), ('harvest','harvest_strength'),
                            ('capacity','food_capacity'), ('sight','sight')]:
            key = f'{t}_{attr}'
            if key in p:
                setattr(piece, field, p[key])
        # Fill to capacity
        piece.food = piece.food_capacity

In [ ]:
@patch
def apply_food_profile(self: GameParts, soil_scale: float = 1.0):
    """Set food stats on all pieces based on balanced profile.
    
    Design intent:
      - Queen is expensive to feed (3.5× pawn), can't self-sustain
      - Pawns are field workers: decent harvest, small pack
      - Bishops are supply wagons: bad harvest, huge capacity
      - Knights are fast but hungry scouts
      - Rooks are heavy but carry well
      - King eats like a queen (royalty!)
    """
    PROFILES = {
        #                  diet  harvest  capacity
        PieceType.PAWN:   (2.0,  2.0,     12),
        PieceType.BISHOP: (2.5,  1.0,    100),
        PieceType.KNIGHT: (3.0,  1.5,     20),
        PieceType.ROOK:   (3.5,  1.0,     40),
        PieceType.QUEEN:  (7.0,  0.5,     30),
        PieceType.KING:   (6.0,  0.5,     25),
    }
    
    count = {t.name: 0 for t in PieceType}
    pieces = []
    for country in self.board.kingdoms:
        for city in country.settlements:
            for piece in city.citizens:
                profile = PROFILES.get(piece.piece_type)
                if profile is None:
                    continue
                piece.diet, piece.harvest_strength, piece.food_capacity = profile
                piece.food = piece.food_capacity  # start full
                count[piece.piece_type.name] += 1
                pieces.append(piece)

    apply_food_profile(pieces,FOOD_PROFILE_V2)
    
    # Apply soil scaling if needed

    fy = FoodYield(self.terr, self.basin)
    if soil_scale != 1.0:
        fy.soil_mult = [s * soil_scale for s in fy.soil_mult]
    fy.compute()

    
    print(f"Food profile applied:")
    for ptype, (d, h, c) in PROFILES.items():
        n = count.get(ptype.name, 0)
        net_t5 = 0.5 * h * 5 - d  # net at tier 5
        label = "✅" if net_t5 > 0 else "❌"
        print(f"  {ptype.name:8s} ×{n:2d}: diet={d:.1f} harvest={h:.1f} "
              f"cap={c:3d}  net@tier5={net_t5:+.1f} {label}")
    
    queen_d = PROFILES[PieceType.QUEEN][0]
    pawn_net = 0.5 * PROFILES[PieceType.PAWN][1] * 5 - PROFILES[PieceType.PAWN][0]
    bish_gap = PROFILES[PieceType.BISHOP][2] / (PROFILES[PieceType.BISHOP][0] + queen_d)
    print(f"\n  Pawns to sustain queen @tier5: {math.ceil(queen_d / max(pawn_net, 0.01))}")
    print(f"  Bishop gap crossing: ~{int(bish_gap)} barren hexes")
    return fy

In [ ]:
p = FOOD_PROFILE_V2
h, d, Q = p['pawn_harvest'], p['pawn_diet'], p['queen_diet']

print("Tier-sum rule (2 pawns, need sum ≥ 7):")
for ta, tb in [(4,4), (3,6), (5,2), (1,5), (3,4), (7,0)]:
    s = ta + tb
    exp = h*s - 2*d if tb > 0 else h*ta - d
    label = f"T{ta}+T{tb}" if tb > 0 else f"Solo T{ta}"
    print(f"  {label:10s} sum={s:2d}  export={exp:.1f}  {'✅' if exp >= Q else '❌'}")
print(f"\n  Pawn on T1: {h*1:.1f} vs diet {d:.1f} → {'survives' if h >= d else 'starves'}")
print(f"  Queen/pawn ratio: {Q/d:.1f}×")

## Piece Shadow

In [ ]:
@dataclass
class PieceShadow:
    "Lightweight piece proxy for planning simulations."
    id:               str
    name:             str
    piece_type:       PieceType
    location:         int
    facing:           int
    birth_year:       int
    food:             float
    food_capacity:    float
    diet:             float
    health:           float
    max_health:       float
    move_strength:    int
    harvest_strength: float
    sight:            int
    owner_id:         int = 0
    flag:             CountryFlag = None
    instructions:     InstructionList = field(default_factory=lambda: InstructionList([], 0, False))
    # stubbed combat — planner doesn't fight
    attack_strength:  float = 0.0
    defense:          float = 1.0
    temperment:       int = 100

    @classmethod
    def from_piece(cls, piece):
        return cls(**{f: getattr(piece, f) for f in [
            'id','name','piece_type','birth_year','location','facing',
            'food','food_capacity','diet','health','max_health',
            'move_strength','harvest_strength','sight','owner_id','flag']})


In [ ]:
@patch
def hexes_in_sight(self: PieceShadow, grid: HexGrid, elevations: np.ndarray,
                   elevation_mult: float = 0.005,
                   facing_only: bool = False) -> set[int]:
    """All hex indices visible from this piece's location."""
    if self.location is None or self.location < 0:
        return set()

    elev = max(0, elevations[self.location])
    effective_sight = int(self.sight + elev * elevation_mult)

    if facing_only:
        facing_dir = HexPosition.directions()[self.facing % 6]
        hex_positions = HexPosition.origin().field_of_view( facing_dir, effective_sight)
        return {
            idx for hp in hex_positions
            if (idx := grid.hexposition_to_index(hp, self.location)) >= 0
        }
    else:
        return set(grid.indices_in_range(self.location, effective_sight))

## coverage

In [ ]:
#| export
def _solver_ok(result):
    return (result.solver.status == pyo.SolverStatus.ok and
            result.solver.termination_condition == pyo.TerminationCondition.optimal)

In [ ]:
#| export
def _find_hungry(alive, grid, piece, safety_turns=2):
    """Find the most desperate piece, sorted by urgency."""
    hungry = []
    for p in alive:
        if p is piece: continue
        if p.health <= 0: continue
        turns_left = p.food / max(p.diet, 0.1)
        if turns_left < safety_turns * 3:  # worth helping
            hungry.append((p, turns_left))
    hungry.sort(key=lambda h: h[1])  # most desperate first
    return hungry


def _adjacent(grid, loc_a, loc_b):
    """True if same hex or neighbors."""
    if loc_a == loc_b: return True
    return loc_b in grid.neighborsOf(loc_a)


def _donor_surplus(piece, safety_turns=2):
    """How much food this piece can spare."""
    keep = piece.diet * safety_turns
    return max(0, piece.food - keep)


def food_redistribute(report: TurnReport):
    """Squad-level food sharing — three tiers of urgency.
    
    Tier 1: Direct transfer (adjacent, no instruction cost)
    Tier 2: Rotate toward hungry (prepend ROT, then GIVE next turn)
    Tier 3: Move toward hungry (prepend FORWARD steps toward them)
    
    Only donors with surplus above their own safety margin participate.
    """
    alive = [p for p in report.squad.pieces if p.health > 0]
    if len(alive) < 2: return
    grid = report.grid
    elevations = report.elevations
    
    # ── Tier 1: Direct transfers (free, every turn) ──
    by_need = sorted(alive, key=lambda p: p.food / max(p.diet, 0.1))
    
    for receiver in by_need:
        need = receiver.food_capacity - receiver.food
        if need < 0.1: continue
        
        for donor in alive:
            if donor is receiver: continue
            surplus = _donor_surplus(donor)
            if surplus < 0.1: continue
            if not _adjacent(grid, donor.location, receiver.location):
                continue
            
            transfer = min(need, surplus)
            donor.food -= transfer
            receiver.food += transfer
            need -= transfer
            if need < 0.1: break
    
    # ── Tier 2 & 3: Only for pieces with surplus that aren't feeding ──
    # Check if anyone is still desperate after direct transfers
    desperate = [p for p in alive 
                 if p.food / max(p.diet, 0.1) < 3.0]
    if not desperate: return
    
    most_desperate = min(desperate, key=lambda p: p.food / max(p.diet, 0.1))
    
    for donor in alive:
        if donor is most_desperate: continue
        surplus = _donor_surplus(donor)
        if surplus < 1.0: continue
        if _adjacent(grid, donor.location, most_desperate.location):
            continue  # already handled in tier 1
        
        # ── Tier 2: Rotate to face the hungry piece ──
        target_facing = _facing_toward(grid, donor.location, 
                                        most_desperate.location)
        if donor.facing != target_facing:
            diff = (target_facing - donor.facing) % 6
            if diff <= 3:
                rot = Instruction.ROT_R.value
            else:
                rot = Instruction.ROT_L.value
            
            cursor = donor.instructions.cursor
            # Only insert if not already rotating toward them
            rules = donor.instructions.rules
            if cursor < len(rules) and rules[cursor] in (
                    Instruction.ROT_R.value, Instruction.ROT_L.value):
                continue  # already rotating, don't stack
            
            donor.instructions.rules.insert(cursor, rot)
            continue  # rotate this turn, give next turn
        
        # ── Tier 3: Facing correct but not adjacent — move toward ──
        # Only move if the receiver is truly desperate (< 2 turns)
        recv_turns = most_desperate.food / max(most_desperate.diet, 0.1)
        if recv_turns > 2.0: continue
        
        # Check if forward movement gets us closer
        hp_donor = grid.index_to_hexposition(donor.location)
        hp_recv  = grid.index_to_hexposition(most_desperate.location)
        dist_now = hp_donor.distance(hp_recv)
        
        # Peek at hex in front
        fwd_hp = hp_donor.neighbor(donor.facing)
        fwd_idx = grid.hexposition_to_index(fwd_hp, donor.location)
        
        if (fwd_idx >= 0 and 
            fwd_idx not in grid.invalidRegion and
            elevations[fwd_idx] > 0):
            
            fwd_dist = fwd_hp.distance(hp_recv)
            if fwd_dist < dist_now:
                cursor = donor.instructions.cursor
                rules = donor.instructions.rules
                if cursor < len(rules) and rules[cursor] == Instruction.FORWARD.value:
                    continue  # already moving
                
                donor.instructions.rules.insert(cursor, Instruction.FORWARD.value)


def food_harvest_check(report: TurnReport):
    """Prepend HARVEST when a piece is on a food tile and hungry.
    
    Capped at 1 insert per turn — won't corrupt instruction lists.
    """
    alive = [p for p in report.squad.pieces if p.health > 0]
    
    for piece in alive:
        loc = piece.location
        if loc is None or loc < 0: continue
        if piece.food >= piece.food_capacity * 0.8: continue
        
        tier = int(report.food_tiers[loc]) if 0 <= loc < len(report.food_tiers) else 0
        if tier < 1: continue
        
        rules = piece.instructions.rules
        cursor = piece.instructions.cursor
        if cursor < len(rules) and rules[cursor] == Instruction.HARVEST.value:
            continue
        
        piece.instructions.rules.insert(cursor, Instruction.HARVEST.value)


def food_triage_v2(report: TurnReport):
    """Combined: harvest opportunities, then redistribute/rescue."""
    food_harvest_check(report)
    food_redistribute(report)

In [ ]:
#| export
def food_redistribute_adjacent(report: TurnReport):
    """Tier 1: Direct adjacent transfers. No instructions touched.
    
    Safe to call every turn, even during march.
    Moves food from anyone with surplus to the most desperate neighbor.
    """
    alive = [p for p in report.squad.pieces if p.health > 0]
    if len(alive) < 2: return
    grid = report.grid
    
    # Most desperate first
    by_need = sorted(alive, key=lambda p: p.food / max(p.diet, 0.1))
    
    for receiver in by_need:
        need = receiver.food_capacity - receiver.food
        if need < 0.1: continue
        
        # Donors: adjacent pieces with surplus above safety margin
        for donor in alive:
            if donor is receiver: continue
            surplus = _donor_surplus(donor)
            if surplus < 0.1: continue
            if not _adjacent(grid, donor.location, receiver.location):
                continue
            
            transfer = min(need, surplus)
            donor.food -= transfer
            receiver.food += transfer
            need -= transfer
            if need < 0.1: break


def food_redistribute_rescue(report: TurnReport):
    """Tiers 2-3: Rotate/move toward hungry. Edits instructions.
    
    Only call during CAMP or emergency. Will disrupt march orders.
    """
    alive = [p for p in report.squad.pieces if p.health > 0]
    if len(alive) < 2: return
    grid = report.grid
    elevations = report.elevations
    
    # Anyone still desperate after tier-1?
    desperate = [p for p in alive 
                 if p.food / max(p.diet, 0.1) < 3.0]
    if not desperate: return
    
    most_desperate = min(desperate, key=lambda p: p.food / max(p.diet, 0.1))
    
    for donor in alive:
        if donor is most_desperate: continue
        surplus = _donor_surplus(donor)
        if surplus < 1.0: continue
        if _adjacent(grid, donor.location, most_desperate.location):
            continue  # tier 1 already handled
        
        # ── Tier 2: Rotate toward hungry ──
        target_facing = _facing_toward(grid, donor.location, 
                                        most_desperate.location)
        if donor.facing != target_facing:
            diff = (target_facing - donor.facing) % 6
            rot = Instruction.ROT_R.value if diff <= 3 else Instruction.ROT_L.value
            
            cursor = donor.instructions.cursor
            rules = donor.instructions.rules
            if cursor < len(rules) and rules[cursor] in (
                    Instruction.ROT_R.value, Instruction.ROT_L.value):
                continue
            
            rules.insert(cursor, rot)
            continue
        
        # ── Tier 3: Move toward hungry (only if truly dying) ──
        recv_turns = most_desperate.food / max(most_desperate.diet, 0.1)
        if recv_turns > 2.0: continue
        
        hp_donor = grid.index_to_hexposition(donor.location)
        hp_recv  = grid.index_to_hexposition(most_desperate.location)
        dist_now = hp_donor.distance(hp_recv)
        
        fwd_hp = hp_donor.neighbor(donor.facing)
        fwd_idx = grid.hexposition_to_index(fwd_hp, donor.location)
        
        if (fwd_idx >= 0 and 
            fwd_idx not in grid.invalidRegion and
            elevations[fwd_idx] > 0):
            
            fwd_dist = fwd_hp.distance(hp_recv)
            if fwd_dist < dist_now:
                cursor = donor.instructions.cursor
                rules = donor.instructions.rules
                if cursor < len(rules) and rules[cursor] == Instruction.FORWARD.value:
                    continue
                rules.insert(cursor, Instruction.FORWARD.value)

## layout

In [ ]:
#| export
def optimal_camp_layout(squad, center, grid, elevations, food_tiers,
                        countries=None, max_ring=3, elevation_mult=0.005):
    """Network-flow camp placement MIP.
    
    The hex neighborhood is a directed graph:
      - Nodes: candidate hexes (occupied by pieces)
      - Arcs: adjacency edges (or sight-cone edges for longer range)
      - Source nodes: pawns on food hexes (production = harvest × tier)
      - Relay nodes: bishops (high throughput, buffer capacity)
      - Sink nodes: queen/king (high diet, low harvest)
    
    Flow conservation at each occupied hex:
      production + inflow ≥ consumption + outflow
    
    Placement decides WHO goes WHERE.
    Flow decides HOW MUCH food moves along each arc.
    Objective: maximize net surplus (total prod - total diet),
               tiebreak by keeping high-combat pieces off food duty.
    """
    alive = squad.alive
    N = len(alive)
    
    # ── Candidate hexes (ring 0..max_ring around center) ──
    cands = []
    seen = set()
    for ring in range(0, max_ring + 1):
        ring_hps = [HexPosition.origin()] if ring == 0 else HexPosition.origin().ring(ring)
        for hp in ring_hps:
            idx = grid.hexposition_to_index(hp, center)
            if idx < 0 or idx in grid.invalidRegion or idx in seen: continue
            if elevations[idx] <= 0: continue
            if countries is not None and countries[idx] < 0: continue
            seen.add(idx)
            cands.append(idx)
    
    H = len(cands)
    tier = [int(food_tiers[c]) if 0 <= c < len(food_tiers) else 0
            for c in cands]
    hex_to_local = {c: i for i, c in enumerate(cands)}
    
    # ── Directed arcs: adjacency-based ──
    # (For sight-cone arcs, expand this with FOV precomputation)
    arcs = []
    for i in range(H):
        for nb in grid.neighborsOf(cands[i]):
            if nb in hex_to_local:
                j = hex_to_local[nb]
                arcs.append((i, j))
    
    # Piece parameters
    diet_p = [p.diet for p in alive]
    harv_p = [p.harvest_strength for p in alive]
    combat_p = [getattr(p, 'attack_strength', 0) for p in alive]
    
    # ── Pyomo model ──
    m = pyo.ConcreteModel()
    m.P = pyo.RangeSet(0, N - 1)
    m.H = pyo.RangeSet(0, H - 1)
    m.A = pyo.Set(initialize=arcs)
    
    # ── Decision variables ──
    m.place = pyo.Var(m.P, m.H, domain=pyo.Binary)    # piece p on hex h
    m.flow  = pyo.Var(m.A, domain=pyo.NonNegativeReals) # food along arc
    
    # ── Placement constraints ──
    m.one_hex = pyo.Constraint(m.P, rule=lambda m, p:
        sum(m.place[p, h] for h in m.H) == 1)
    
    m.one_piece = pyo.Constraint(m.H, rule=lambda m, h:
        sum(m.place[p, h] for p in m.P) <= 1)
    
    # ── Production / consumption at each hex ──
    def _prod(m, h):
        """Harvest output: Σ_p place[p,h] × harv_p × tier[h]"""
        return sum(m.place[p, h] * harv_p[p] * tier[h] for p in m.P)
    
    def _cons(m, h):
        """Diet demand: Σ_p place[p,h] × diet_p"""
        return sum(m.place[p, h] * diet_p[p] for p in m.P)
    
    # ── Flow conservation at every hex ──
    # production + inflow ≥ consumption + outflow
    def balance(m, h):
        inflow  = sum(m.flow[j, h] for (j, hh) in arcs if hh == h)
        outflow = sum(m.flow[h, j] for (hh, j) in arcs if hh == h)
        return _prod(m, h) + inflow >= _cons(m, h) + outflow
    m.balance = pyo.Constraint(m.H, rule=balance)
    
    # ── Arc capacity: can only send from an occupied hex ──
    M_big = max(harv_p) * max(tier + [1]) * 2
    m.arc_send = pyo.Constraint(m.A, rule=lambda m, i, j:
        m.flow[i, j] <= M_big * sum(m.place[p, i] for p in m.P))
    m.arc_recv = pyo.Constraint(m.A, rule=lambda m, i, j:
        m.flow[i, j] <= M_big * sum(m.place[p, j] for p in m.P))
    
    # ── Objective ──
    # Primary: maximize total surplus (production - consumption)
    # Secondary: minimize combat value on food hexes (keep fighters free)
    α = 100  # surplus is much more important than combat tiebreak
    
    # A piece is "on food duty" if on a hex with tier > 0
    food_duty_combat = sum(
        combat_p[p] * m.place[p, h] * (1 if tier[h] > 0 else 0)
        for p in range(N) for h in range(H))
    
    m.obj = pyo.Objective(
        expr=α * (sum(_prod(m, h) for h in m.H) 
                - sum(_cons(m, h) for h in m.H))
             - food_duty_combat,
        sense=pyo.maximize)
    
    # ── Solve ──
    solver = pyo.SolverFactory('appsi_highs')
    result = solver.solve(m, tee=False)
    if not _solver_ok(result):
        return None
    
    # ── Extract results ──
    placements = {}  # piece → hex_idx
    for p in m.P:
        for h in m.H:
            if pyo.value(m.place[p, h]) > 0.5:
                placements[alive[p].id] = cands[h]
                break
    
    flows = {}  # (hex_i, hex_j) → flow amount
    for (i, j) in arcs:
        v = pyo.value(m.flow[i, j])
        if v > 0.01:
            flows[(cands[i], cands[j])] = round(v, 2)
    
    total_prod = sum(pyo.value(_prod(m, h)) for h in m.H)
    total_cons = sum(pyo.value(_cons(m, h)) for h in m.H)
    
    return {
        'placements': placements,
        'flows': flows,
        'production': total_prod,
        'consumption': total_cons,
        'surplus': total_prod - total_cons,
        'cands': cands,
        'tiers': tier,
    }

In [ ]:
def smart_placement(settlement, squad, terrain, food_tiers, grid,
                       max_ring=10, camp_ring=3, top_k=5):
    """Place entire squad using optimal_camp_layout network-flow MIP.
    
    Phase 1: Score candidate centers cheaply (sum of nearby food tiers)
    Phase 2: Run full MIP on top-k candidates
    Phase 3: Apply best layout
    
    Returns the best result dict from optimal_camp_layout, or None.
    """
    center = settlement.location
    elevs = terrain.elevations
    
    # ── Phase 1: Cheap scoring of candidate centers ──
    candidates = []
    for hp in HexPosition.origin().spiral(max_ring):
        idx = grid.hexposition_to_index(hp, origin_index=center)
        if idx < 0 or elevs[idx] <= 0: continue
        
        # Quick heuristic: sum food tiers in the camp neighborhood
        score = 0
        land_count = 0
        for r in range(0, camp_ring + 1):
            ring_hps = [HexPosition.origin()] if r == 0 else HexPosition.origin().ring(r)
            for nhp in ring_hps:
                nidx = grid.hexposition_to_index(nhp, origin_index=idx)
                if nidx < 0 or elevs[nidx] <= 0: continue
                land_count += 1
                score += int(food_tiers[nidx]) if 0 <= nidx < len(food_tiers) else 0
        
        # Need enough land hexes for the squad
        if land_count >= len(squad.alive):
            candidates.append((idx, score))
    
    if not candidates:
        return None
    
    # Top-k by cheap score
    candidates.sort(key=lambda x: -x[1])
    shortlist = candidates[:top_k]
    
    # ── Phase 2: Full MIP on shortlist ──
    best_result = None
    best_surplus = -float('inf')
    best_center = center
    
    for cand_idx, cheap_score in shortlist:
        result = optimal_camp_layout(
            squad, cand_idx, grid, elevs, food_tiers,
            max_ring=camp_ring)
        
        if result is None: continue
        if result['surplus'] > best_surplus:
            best_surplus = result['surplus']
            best_result = result
            best_center = cand_idx
    
    if best_result is None:
        return None
    
    # ── Phase 3: Apply placements + facings ──
    for piece in squad.alive:
        hex_idx = best_result['placements'].get(piece.id)
        if hex_idx is not None:
            piece.location = hex_idx
            # Face toward camp center (for GIVE arcs)
            if hex_idx != best_center:
                piece.facing = _facing_toward(grid, hex_idx, best_center)
    
    best_result['center'] = best_center
    print(f"Smart placement v2: center={best_center}, "
          f"surplus={best_surplus:.1f} "
          f"(prod={best_result['production']:.1f}, "
          f"cons={best_result['consumption']:.1f}), "
          f"checked {len(shortlist)} candidates")
    
    return best_result

In [ ]:
#| export
@patch
def loadGeology(self: GameParts, geo: Geology):
    self.terr = geo.terrain
    self.grid = self.terr.hexGrid
    self.builder = self.grid.builder

    self.board = GameBoard(self.terr, 2)
    self.basin = geo.basins

    self.country = self.board.kingdoms[0]
    self.home = self.country.settlements[0]
    self.squads = Squad.squads(6, self.country.flag)
    self.queenGuard = self.squads[0]
    for pType in [PieceType.PAWN, PieceType.PAWN, PieceType.PAWN, PieceType.PAWN,
                  PieceType.PAWN, PieceType.BISHOP, PieceType.BISHOP, PieceType.QUEEN]:
        piece = self.home.recruit(pType, flag=self.queenGuard.flag)
        self.queenGuard.pieces.append(piece)

    # Initial placement so pieces have locations
    self.home.place_squad_spiral(self.queenGuard, self.terr)
    self.queen = self.queenGuard.by_rank()[0]

    # Set diet/harvest/capacity on all pieces
    self.apply_food_profile()

    # Compute food tiers, then re-place using sustainability logic
    fy = FoodYield(self.terr, self.basin)
    fy.compute()
    best_cost = smart_placement(self.home, self.queenGuard, self.terr, fy.tiers, self.grid)
    print(f"Smart placement: queen at {self.queen.location}, sustain cost = {best_cost} pawns")

In [ ]:
### Show demo

In [ ]:
showDemo = True

In [ ]:
showDemo = False

In [ ]:
myStuff = GameParts()

In [ ]:
def rebootWorld():

    world = TerrainFactory.create_world(
        bounds= MapRect(MapCord(0, 0), MapSize(300, 300)),
        preset='temperate',
        name='Maiden Lane',
        radius=15,
        lon_span=10.0,
        num_plates=8,
        subdivisions=3,
        ocean_fraction=0.3,
        oceanic_sides=['N'],  # Ocean on east and west
        terrain_age='young',  # Sharp, dramatic features
        formation_type='ridge',  # Creates ridge formations
        elevation_scale=1.5,  # Exaggerate the heights
        erosion_age=0.1,  # Minimal erosion for sharp peaks
        num_lakes=0,
        seed=23,
        debug=True
    )

    sampleMap = world.terrain
    sampleMap.repalette()

    for i in range(len(sampleMap.hexGrid.hexes)):
        sampleMap.hexGrid.hexes[i].label = str(i)

    myStuff.loadGeology(world)
    myStuff.grid.adjustRadius(40)
    for i in range(len(myStuff.grid.hexes)):
        myStuff.grid.hexes[i].label = str(i)

    return myStuff

In [ ]:
rebootWorld()

In [ ]:
TerrainDisplay(
    TerrainOverlay(),
    SettlementOverlay(),
    SquadVisionOverlay([myStuff.queenGuard]),
    #SustainOverlay(queen=myStuff.queen),
    RiverOverlay(max_width=5),
    terrain=myStuff.terr,
    board=myStuff.board,
    basins=myStuff.basin,
    debug=not showDemo
    )

In [ ]:
# Run the simulator with food_triage so the squad stops to feed
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()
food_tiers = fy.tiers

sim = FoodSimulator(
    grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=food_tiers,
    pieces=myStuff.queenGuard.alive,
    countries=myStuff.terr.fields.get('country'),
    squads=[myStuff.queenGuard],
    callbacks={myStuff.queenGuard.id: food_triage_v2},
)

sim.run(num_turns=40)
sim.summary()

## Routes

In [ ]:
import heapq

def food_aware_route(start_hex, end_hex, grid, elevations, food_tiers,
                     cone_range=4, pawn_harvest=1.3, pawn_diet=0.7,
                     queen_diet=7.0, barren_penalty=10.0, countries=None):
    """Dijkstra pathfinding that prefers hexes near good food sources.
    
    Cost model for each step:
      base_cost = 1.0  (one hex of movement)
      
      For the neighbor hex, look at the best pawn-pair export 
      available within cone_range. The richer the neighborhood,
      the cheaper the step.
      
      cost = base_cost + barren_penalty × (1 - pair_sustainability)
      
      where pair_sustainability = min(1, best_pair_export / queen_diet)
    
    This routes the queen through corridors where 2 pawns can feed her,
    and avoids barren stretches even if they're shorter.
    """
    N = len(elevations)
    
    # ── Precompute: for each hex, best pawn-pair export within cone_range ──
    # First, get per-hex export (what a single pawn stationed there produces)
    hex_export = [0.0] * N
    for i in range(N):
        if i in grid.invalidRegion or elevations[i] <= 0: continue
        tier = int(food_tiers[i]) if 0 <= i < len(food_tiers) else 0
        net = pawn_harvest * tier - pawn_diet
        hex_export[i] = max(0, net)
    
    # For each hex, find the best pair of food sources within cone_range
    # (cached so Dijkstra doesn't recompute)
    _pair_cache = {}
    
    def best_pair_export(hex_idx):
        if hex_idx in _pair_cache: return _pair_cache[hex_idx]
        
        # Collect all food sources within cone_range
        hp_center = grid.index_to_hexposition(hex_idx)
        sources = []
        for ring in range(1, int(cone_range) + 1):
            for hp in hp_center.ring(ring):
                nidx = grid.hexposition_to_index(hp, hex_idx)
                if nidx < 0 or nidx >= N: continue
                if nidx in grid.invalidRegion: continue
                if hex_export[nidx] > 0:
                    sources.append(hex_export[nidx])
        
        # Best pair
        sources.sort(reverse=True)
        if len(sources) >= 2:
            result = sources[0] + sources[1]
        elif len(sources) == 1:
            result = sources[0]
        else:
            result = 0.0
        
        _pair_cache[hex_idx] = result
        return result
    
    # ── Dijkstra ──
    INF = float('inf')
    dist = {start_hex: 0.0}
    parent = {start_hex: None}
    pq = [(0.0, start_hex)]
    
    while pq:
        d, cur = heapq.heappop(pq)
        if cur == end_hex: break
        if d > dist.get(cur, INF): continue
        
        for nb in grid.neighborsOf(cur):
            if nb < 0 or nb >= N: continue
            if nb in grid.invalidRegion: continue
            if elevations[nb] <= 0: continue
            if countries is not None and countries[nb] < 0: continue
            
            # Cost: base + penalty for poor food neighborhood
            pair_exp = best_pair_export(nb)
            sustainability = min(1.0, pair_exp / queen_diet)
            step_cost = 1.0 + barren_penalty * (1.0 - sustainability)
            
            new_dist = d + step_cost
            if new_dist < dist.get(nb, INF):
                dist[nb] = new_dist
                parent[nb] = cur
                heapq.heappush(pq, (new_dist, nb))
    
    if end_hex not in parent:
        print(f"No route ⬡{start_hex} → ⬡{end_hex}"); return None
    
    # Reconstruct
    route = []
    cur = end_hex
    while cur is not None:
        route.append(cur)
        cur = parent[cur]
    route.reverse()
    
    # Stats
    total_cost = dist[end_hex]
    min_pair = min(best_pair_export(h) for h in route)
    avg_pair = sum(best_pair_export(h) for h in route) / len(route)
    barren_count = sum(1 for h in route if best_pair_export(h) < queen_diet)
    
    print(f"Food-aware route: {len(route)} hexes, cost={total_cost:.1f}")
    print(f"  Pair export — min={min_pair:.1f}, avg={avg_pair:.1f}, "
          f"barren={barren_count}/{len(route)}")
    
    return route


In [ ]:
import pyomo.environ as pyo

def _solver_ok(result):
    """Check if solver found a feasible solution."""
    tc = result.solver.termination_condition
    return tc in (pyo.TerminationCondition.optimal, 
                  pyo.TerminationCondition.feasible)

def pawn_covered_route(start_hex, end_hex, grid, elevations, food_tiers,
                       pawn_harvest=1.3, pawn_diet=0.7, queen_diet=7.0,
                       cone_range=4, barren_penalty=10.0, countries=None,
                       n_bishops=1, bishop_capacity=140.0, bishop_diet=2.5,
                       n_queens=1):
    """Find food-aware route + optimal pawn placements with bishop gap support.
    
    Soft coverage: gaps are allowed if available bishops can ferry the queen
    across them. Slack per route hex represents food deficit that bishops
    must supply. Total slack is bounded by bishop budget.
    
    Args:
        n_bishops:        bishops available for gap crossings
        bishop_capacity:  food capacity per bishop
        bishop_diet:      bishop food per turn while escorting
        n_queens:         number of queens (scales food demand)
    
    Returns dict with route, placements, gaps, queen_diet, bishop_fuel_needed
    """
    effective_diet = queen_diet * n_queens
    
    # ── 1. Food-aware route (Dijkstra with barren penalty) ──
    route = food_aware_route(start_hex, end_hex, grid, elevations,
                             food_tiers, barren_penalty)
    if not route:
        print(f"No route found ⬡{start_hex} → ⬡{end_hex}")
        return None
    
    W = len(route)
    route_set = set(route)
    
    # Route stats
    pair_exp = []
    barren_count = sum(1 for h in route 
                       if h >= len(food_tiers) or food_tiers[h] == 0)
    print(f"Food-aware route: {W} hexes")
    
    # ── 2. Hex distance helper (O(1) via cube coords) ──
    def hex_dist(a, b):
        pa = grid.index_to_hexposition(a)
        pb = grid.index_to_hexposition(b)
        return pa.distance(pb)
    
    # ── 3. Candidate pawn positions ──
    # Find all hexes within cone_range of any route hex, with food > 0
    seen_cands = set()
    candidates = []
    
    for rh in route:
        # BFS outward from route hex to find candidate sites
        frontier = {rh}
        visited_bfs = {rh}
        for _ in range(cone_range + 1):
            nxt = set()
            for h in frontier:
                for nb in grid.neighborsOf(h):
                    if nb < 0 or nb >= len(elevations): continue
                    if nb in visited_bfs: continue
                    if elevations[nb] <= 0: continue
                    visited_bfs.add(nb)
                    nxt.add(nb)
            frontier = nxt
        
        for h in visited_bfs:
            if h in seen_cands or h in route_set: continue
            if h in grid.invalidRegion: continue
            
            tier = int(food_tiers[h]) if h < len(food_tiers) else 0
            if tier <= 0: continue
            
            export = tier * pawn_harvest - pawn_diet
            if export <= 0: continue
            
            # Coverage: route indices this candidate can feed
            coverage = set()
            for w, rw in enumerate(route):
                if hex_dist(h, rw) <= cone_range:
                    coverage.add(w)
            if not coverage: continue
            
            candidates.append({
                'hex': h, 'tier': tier, 'export': export, 'coverage': coverage,
            })
            seen_cands.add(h)
    
    if not candidates:
        print("No pawn candidates found!")
        return None
    
    S = range(len(candidates))
    W_range = range(W)
    
    # Pre-compute (candidate, route_hex) → export
    export_at = {}
    for s in S:
        for w in candidates[s]['coverage']:
            export_at[s, w] = candidates[s]['export']
    
    # Identify fully uncoverable hexes (no candidate reaches them at all)
    uncoverable = set()
    for w in W_range:
        if not any((s, w) in export_at for s in S):
            uncoverable.add(w)
    
    if uncoverable:
        print(f"  {len(uncoverable)} route hexes unreachable by any pawn "
              f"(indices: {sorted(uncoverable)[:10]}{'...' if len(uncoverable) > 10 else ''})")
    
    # ── 4. MIP with soft constraints ──
    m = pyo.ConcreteModel()
    m.S = pyo.Set(initialize=list(S))
    m.W = pyo.Set(initialize=list(W_range))
    
    # Binary: place this candidate?
    m.placed = pyo.Var(m.S, domain=pyo.Binary)
    
    # Continuous: food deficit at each route hex (bishop must supply)
    m.deficit = pyo.Var(m.W, domain=pyo.NonNegativeReals,
                      bounds=(0, effective_diet))
    
    # Binary: is this route hex a gap? (slack > 0)
    m.is_gap = pyo.Var(m.W, domain=pyo.Binary)
    
    # ── Coverage: placed pawn exports + slack >= queen demand ──
    m.cover = pyo.ConstraintList()
    for w in m.W:
        pawn_supply = sum(export_at[s, w] * m.placed[s]
                          for s in m.S if (s, w) in export_at)
        m.cover.add(pawn_supply + m.deficit[w] >= effective_diet)
    
    # ── Link slack ↔ gap indicator ──
    m.gap_link = pyo.ConstraintList()
    for w in m.W:
        m.gap_link.add(m.deficit[w] <= effective_diet * m.is_gap[w])
    
    # ── Bishop budget: food to supply + bishop self-consumption ≤ capacity ──
    # sum(slack) = total food bishops must give the queen
    # bishop_diet × num_gap_hexes = food bishops eat while in gaps
    bishop_budget = n_bishops * bishop_capacity
    m.bishop_budget = pyo.Constraint(
        expr=(sum(m.deficit[w] for w in m.W)
              + bishop_diet * sum(m.is_gap[w] for w in m.W)
              <= bishop_budget))
    
    # ── Objective: minimize pawns, then minimize slack ──
    # Pawns are expensive (weight=1), slack is cheap (weight=0.01)
    # → solver prefers to use bishops for gaps rather than adding marginal pawns
    slack_weight = 0.01
    m.obj = pyo.Objective(
        expr=(sum(m.placed[s] for s in m.S)
              + slack_weight * sum(m.deficit[w] for w in m.W)),
        sense=pyo.minimize)
    
    # ── Solve ──
    solver = pyo.SolverFactory('appsi_highs')
    result = solver.solve(m)
    
    if not _solver_ok(result):
        print(f"MIP infeasible — {len(uncoverable)} uncoverable, "
              f"bishop budget={bishop_budget:.0f}")
        return None
    
    # ── 5. Extract results ──
    placements = []
    for s in m.S:
        if pyo.value(m.placed[s]) > 0.5:
            placements.append(candidates[s])
    placements.sort(key=lambda p: min(p['coverage']))
    
    gaps = sorted(w for w in W_range if pyo.value(m.deficit[w]) > 0.01)
    gap_hexes = [route[w] for w in gaps]
    bishop_fuel = sum(pyo.value(m.deficit[w]) for w in W_range)
    bishop_turns = sum(1 for w in W_range if pyo.value(m.is_gap[w]) > 0.5)
    
    # ── Per-hex export summary ──
    min_exp = float('inf')
    total_exp = 0
    pawn_covered = 0
    for w in W_range:
        exp = sum(p['export'] for p in placements if w in p['coverage'])
        total_exp += exp
        if exp >= effective_diet: pawn_covered += 1
        min_exp = min(min_exp, exp)
    avg_exp = total_exp / W if W > 0 else 0
    
    print(f"  Pair export — min={min_exp:.1f}, avg={avg_exp:.1f}, "
          f"barren={W - pawn_covered}/{W}")
    print(f"PCR: route {W} hexes, {len(placements)} pawns, "
          f"{pawn_covered}/{W} pawn-covered"
          + (f", {len(gaps)} gap hexes → bishop" if gaps else " ✅"))
    
    for p in placements:
        cmin, cmax = min(p['coverage']), max(p['coverage'])
        print(f"  ⬡{p['hex']} T{p['tier']} export={p['export']:.1f}/t  "
              f"path[{cmin}..{cmax}]")
    
    if gaps:
        print(f"  🐪 Bishop gap: route{gaps} (⬡{gap_hexes})")
        print(f"     fuel={bishop_fuel:.1f} + bishop_eat={bishop_diet * bishop_turns:.1f} "
              f"= {bishop_fuel + bishop_diet * bishop_turns:.1f}"
              f"/{bishop_budget:.0f} "
              f"({'✅' if bishop_fuel + bishop_diet * bishop_turns <= bishop_budget else '❌'})")
    
    return {
        'route': route,
        'placements': placements,
        'gaps': gaps,
        'gap_hexes': gap_hexes,
        'queen_diet': effective_diet,
        'bishop_fuel_needed': bishop_fuel,
        'bishop_turns': bishop_turns,
    }


In [ ]:
cities = []
for country in myStuff.board.kingdoms:
    for city in country.settlements:
        cities.append(city)
print(len(cities))

In [ ]:
start = cities[0].location
finish = cities[1].location
start = cities[0].location
finish = cities[1].location
print(f"Route: ⬡{start} → ⬡{finish}")

pcr = pawn_covered_route(
    start, finish,
    grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=fy.tiers,
    pawn_harvest=FOOD_PROFILE_V2['pawn_harvest'],
    pawn_diet=FOOD_PROFILE_V2['pawn_diet'],
    queen_diet=FOOD_PROFILE_V2['queen_diet'],
    cone_range=FOOD_PROFILE_V2['pawn_sight'],
)

In [ ]:
rebootWorld()

In [ ]:
# ── Step 1: Create pawn pieces at PCR placements ──
from HexMagic.game.data import Piece, PieceType, Instruction, InstructionList

# Fresh squad for the PCR
pcr_squad = Squad.squads(1, myStuff.country.flag)[0]

# Add the queen
queen = myStuff.queen
queen.location = pcr['route'][0]  # start of route
queen.food = FOOD_PROFILE_V2['queen_capacity']
pcr_squad.pieces.append(queen)

# Create pawns at each placement hex
for i, pl in enumerate(pcr['placements']):
    pawn = myStuff.home.recruit(PieceType.PAWN, flag=pcr_squad.flag)
    pawn.location = pl['hex']
    
    # Face toward nearest route hex
    hp_p = myStuff.grid.index_to_hexposition(pl['hex'])
    mid_w = sorted(pl['coverage'])[len(pl['coverage']) // 2]
    route_hex = pcr['route'][mid_w]
    pawn.facing = _facing_toward(myStuff.grid, pl['hex'], route_hex)
    
    # Instructions: harvest then give, on patrol (loop forever)
    pawn.instructions = InstructionList(
        [Instruction.HARVEST.value, Instruction.GIVE.value],
        cursor=0, patrol=True)
    
    pcr_squad.pieces.append(pawn)

# Apply V2 profile to all pieces in the squad
apply_food_profile(pcr_squad.pieces)

print(f"Squad: {len(pcr_squad.pieces)} pieces "
      f"({len(pcr_squad.alive)} alive)")
for p in pcr_squad.pieces:
    print(f"  {p.name:10s} {p.piece_type.name:6s} ⬡{p.location} "
          f"facing={p.facing} food={p.food:.1f}")

In [ ]:
def route_to_instructions(route, grid, start_facing):
    """Convert a list of adjacent hex indices into exact FORWARD/ROT instructions."""
    rules = []
    facing = start_facing
    dirs = HexPosition.directions()
    
    for i in range(len(route) - 1):
        # Which of the 6 directions goes from route[i] to route[i+1]?
        target_facing = None
        for d in range(6):
            nb_idx = grid.hexposition_to_index(dirs[d], route[i])
            if nb_idx == route[i + 1]:
                target_facing = d
                break
        
        if target_facing is None:
            print(f"⚠️ {route[i]}→{route[i+1]} not adjacent! Skipping.")
            continue
        
        # Rotate shortest direction
        while facing != target_facing:
            diff = (target_facing - facing) % 6
            if diff <= 3:
                rules.append(Instruction.ROT_R.value)
                facing = (facing + 1) % 6
            else:
                rules.append(Instruction.ROT_L.value)
                facing = (facing - 1) % 6
        
        rules.append(Instruction.FORWARD.value)
    return rules, facing

rules, _ = route_to_instructions(pcr['route'], myStuff.grid, queen.facing)
queen.instructions = InstructionList(rules, cursor=0, patrol=False)
print(f"Route instructions: {len(rules)} steps for {len(pcr['route'])} hexes")

In [ ]:
#| export
# ── Step 3: Run the simulator ──
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()

sim = FoodSimulator(
    grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=fy.tiers,
    pieces=pcr_squad.alive,
    countries=myStuff.terr.fields.get('country'),
    squads=[pcr_squad],
    callbacks={pcr_squad.id: food_redistribute_adjacent},
)

sim.run(num_turns=30)
sim.summary()

In [ ]:
#| export
#show(sim)

In [ ]:
def find_stations(pcr_result, pawns_per_station=2):
    """Group PCR placements into ordered stations of pawn-pairs.
    
    Each station is 2 pawn slots that a leapfrog pair occupies.
    Stations are ordered by their coverage midpoint along the route.
    """
    placements = sorted(pcr_result['placements'], 
                        key=lambda p: min(p['coverage']))
    
    stations = []
    i = 0
    while i < len(placements):
        group = placements[i:i + pawns_per_station]
        covers = set()
        for p in group:
            covers |= p['coverage']
        stations.append({
            'hexes': [p['hex'] for p in group],
            'tiers': [p['tier'] for p in group],
            'covers': covers,
            'min_w': min(covers),
            'max_w': max(covers),
        })
        i += pawns_per_station
    
    return stations


def optimal_lighthouse_placement(gap_indices, route, grid, elevations,
                                 n_bishops=1, bishop_sight=4,
                                 queen_diet=7.0, bishop_diet=3.5,
                                 bishop_capacity=140.0,
                                 max_ring=2, profile=None):
    """MIP: place bishops as lighthouses to cover route gap hexes.

    Facility location problem:
      - Candidates: hexes within max_ring of any gap hex, on land
      - Each bishop parks at a candidate and covers gap hexes in sight range
      - Fuel constraint: bishop burns (queen_diet + bishop_diet) per turn
        the queen is in its cone, bounded by bishop_capacity
      - Objective: minimize bishops, tiebreak by elevation (higher = better)

    Args:
        gap_indices:  list of route indices that need bishop coverage
        route:        full route hex list
        grid:         HexGrid
        elevations:   elevation array
        n_bishops:    max bishops available
        bishop_sight: base sight range
        queen_diet, bishop_diet, bishop_capacity: food profile values
        max_ring:     how far from gap hexes to search for lighthouse sites
        profile:      optional FOOD_PROFILE_V2 override

    Returns dict with placements, coverage, fuel info, or None if infeasible.
    """
    if profile:
        queen_diet = profile.get('queen_diet', queen_diet)
        bishop_diet = profile.get('bishop_diet', bishop_diet)
        bishop_capacity = profile.get('bishop_capacity', bishop_capacity)
        bishop_sight = profile.get('pawn_sight', bishop_sight)  # or a bishop_sight key

    burn_per_turn = queen_diet + bishop_diet
    gap_set = set(gap_indices)
    gap_hexes = [route[w] for w in gap_indices]
    route_set = set(route)
    G = len(gap_indices)

    if G == 0:
        return {'placements': [], 'fuel': {}, 'total_fuel': 0, 'feasible': True}

    # ── Candidate lighthouse hexes ──
    seen = set()
    candidates = []

    for gh in gap_hexes:
        hp_g = grid.index_to_hexposition(gh)
        for ring in range(0, bishop_sight + max_ring + 1):
            ring_hps = [HexPosition.origin()] if ring == 0 else HexPosition.origin().ring(ring)
            for hp in ring_hps:
                idx = grid.hexposition_to_index(hp, gh)
                if idx < 0 or idx in seen: continue
                if idx in grid.invalidRegion: continue
                if idx >= len(elevations) or elevations[idx] <= 0: continue
                seen.add(idx)

                # Which gap hexes can this candidate see?
                hp_c = grid.index_to_hexposition(idx)
                can_see = []
                for wi, w in enumerate(gap_indices):
                    hp_r = grid.index_to_hexposition(route[w])
                    if hp_c.distance(hp_r) <= bishop_sight:
                        can_see.append(wi)  # index into gap_indices

                if not can_see: continue

                candidates.append({
                    'hex': idx,
                    'can_see': can_see,  # indices into gap_indices
                    'elevation': elevations[idx],
                    'on_route': idx in route_set,
                })

    if not candidates:
        print("No lighthouse candidates found!")
        return None

    C = len(candidates)
    B = n_bishops

    # ── Pyomo MIP ──
    m = pyo.ConcreteModel()
    m.C = pyo.RangeSet(0, C - 1)
    m.G = pyo.RangeSet(0, G - 1)
    m.B = pyo.RangeSet(0, B - 1)

    # place[b, c]: bishop b placed at candidate c?
    m.place = pyo.Var(m.B, m.C, domain=pyo.Binary)

    # serve[b, g]: bishop b feeds queen at gap index g?
    m.serve = pyo.Var(m.B, m.G, domain=pyo.Binary)

    # ── Each bishop placed at most one candidate ──
    m.one_site = pyo.Constraint(m.B, rule=lambda m, b:
        sum(m.place[b, c] for c in m.C) <= 1)

    # ── Every gap hex must be served by at least one bishop ──
    m.covered = pyo.Constraint(m.G, rule=lambda m, g:
        sum(m.serve[b, g] for b in m.B) >= 1)

    # ── Can only serve from a placed bishop within sight ──
    # Pre-compute: which (candidate, gap_index) pairs are valid
    valid_cg = set()
    for c_idx, cand in enumerate(candidates):
        for g_local in cand['can_see']:
            valid_cg.add((c_idx, g_local))

    m.sight = pyo.ConstraintList()
    for b in m.B:
        for g in m.G:
            # serve[b,g] ≤ Σ place[b,c] for candidates c that can see gap g
            covers = [c for c in m.C if (c, g) in valid_cg]
            if not covers:
                m.sight.add(m.serve[b, g] == 0)
            else:
                m.sight.add(m.serve[b, g] <= sum(m.place[b, c] for c in covers))

    # ── Fuel constraint: each bishop's total service ≤ capacity ──
    # Each gap hex served costs burn_per_turn (1 turn per hex at queen speed)
    m.fuel = pyo.Constraint(m.B, rule=lambda m, b:
        sum(m.serve[b, g] for g in m.G) * burn_per_turn <= bishop_capacity)

    # ── Objective: minimize bishops used, tiebreak by elevation ──
    # "bishop used" = placed anywhere
    m.used = pyo.Var(m.B, domain=pyo.Binary)
    m.link_used = pyo.Constraint(m.B, rule=lambda m, b:
        m.used[b] >= sum(m.place[b, c] for c in m.C))

    # Elevation bonus (normalized small so it's just a tiebreak)
    max_elev = max(c['elevation'] for c in candidates) or 1
    elev_bonus = sum(
        m.place[b, c] * (candidates[c]['elevation'] / max_elev) * 0.01
        for b in m.B for c in m.C
    )
    # Prefer off-route (small penalty for on-route)
    route_penalty = sum(
        m.place[b, c] * (0.001 if candidates[c]['on_route'] else 0)
        for b in m.B for c in m.C
    )

    m.obj = pyo.Objective(
        expr=sum(m.used[b] for b in m.B) - elev_bonus + route_penalty,
        sense=pyo.minimize)

    # ── Solve ──
    solver = pyo.SolverFactory('appsi_highs')

    # In optimal_lighthouse_placement, change the solve call to:
    result = solver.solve(m, tee=False, load_solutions=False)

    if not _solver_ok(result):
        print(f"Lighthouse MIP infeasible: {G} gap hexes, {C} candidates, {B} bishops")
        return None

    m.solutions.load_from(result)


    if not _solver_ok(result):
        print(f"Lighthouse MIP infeasible: {G} gap hexes, {C} candidates, {B} bishops")
        return None

    # ── Extract results ──
    placements = []
    fuel_usage = {}

    for b in m.B:
        for c in m.C:
            if pyo.value(m.place[b, c]) > 0.5:
                served = [g for g in m.G if pyo.value(m.serve[b, g]) > 0.5]
                served_route_w = [gap_indices[g] for g in served]
                fuel = len(served) * burn_per_turn

                placements.append({
                    'bishop': b,
                    'hex': candidates[c]['hex'],
                    'elevation': candidates[c]['elevation'],
                    'on_route': candidates[c]['on_route'],
                    'serves_gap': served,
                    'serves_route_w': served_route_w,
                    'fuel_used': round(fuel, 1),
                    'fuel_remaining': round(bishop_capacity - fuel, 1),
                })
                fuel_usage[b] = fuel

    total_fuel = sum(fuel_usage.values())
    bishops_used = len(placements)

    # ── Report ──
    print(f"Lighthouse MIP: {bishops_used}/{B} bishops cover {G} gap hexes")
    for p in placements:
        w_range = f"route[{min(p['serves_route_w'])}..{max(p['serves_route_w'])}]"
        print(f"  🔦 Bishop {p['bishop']}: ⬡{p['hex']} "
              f"(elev={p['elevation']:.0f}"
              f"{', ON route' if p['on_route'] else ''}) "
              f"serves {len(p['serves_gap'])} hexes {w_range} "
              f"fuel={p['fuel_used']}/{bishop_capacity:.0f} "
              f"(margin={p['fuel_remaining']})")

    return {
        'placements': placements,
        'fuel': fuel_usage,
        'total_fuel': round(total_fuel, 1),
        'bishops_used': bishops_used,
        'burn_per_turn': burn_per_turn,
        'feasible': True,
    }


In [ ]:
def test_lighthouse_arc_traversal():
    """Test that a queen can walk through excluded hexes 
    while always staying inside at least one bishop's sight arc."""
    
    fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()
    route = pcr['route']
    gaps = pcr['gaps']  # route indices that are uncovered by pawns
    
    if len(gaps) < 2:
        print("⚠️ Need ≥2 gap hexes to test arc traversal"); return

    res = optimal_lighthouse_placement(
        gaps, route, myStuff.grid, myStuff.terr.elevations,
        n_bishops=2, profile=FOOD_PROFILE_V2)
    assert res and res['feasible'], "MIP must be feasible first"

    # ── Build: for each gap index g, which bishops cover it? ──
    bishop_at_gap = {g: set() for g in range(len(gaps))}
    for p in res['placements']:
        for g in p['serves_gap']:
            bishop_at_gap[g].add(p['bishop'])

    # ── Find contiguous gap segments along the route ──
    segments = []
    seg = [0]
    for i in range(1, len(gaps)):
        if gaps[i] == gaps[i-1] + 1:  # consecutive route indices
            seg.append(i)
        else:
            segments.append(seg)
            seg = [i]
    segments.append(seg)
    
    print(f"Gap segments: {len(segments)} "
          f"(sizes: {[len(s) for s in segments]})")

    # ── Core check: within each segment, consecutive gap hexes 
    #    share at least one bishop (continuous arc) ──
    for si, seg in enumerate(segments):
        for j in range(len(seg) - 1):
            g_cur, g_nxt = seg[j], seg[j+1]
            shared = bishop_at_gap[g_cur] & bishop_at_gap[g_nxt]
            assert shared, (
                f"❌ Arc break in segment {si}: "
                f"gap[{g_cur}] (⬡{route[gaps[g_cur]]}) → "
                f"gap[{g_nxt}] (⬡{route[gaps[g_nxt]]}) "
                f"have no bishop in common! "
                f"Bishops: {bishop_at_gap[g_cur]} vs {bishop_at_gap[g_nxt]}")
        print(f"  ✅ Segment {si}: route[{gaps[seg[0]]}..{gaps[seg[-1]]}] "
              f"— continuous arc ({len(seg)} hexes)")

    # ── Verify queen is never "in the dark" ──
    for g in range(len(gaps)):
        assert bishop_at_gap[g], \
            f"❌ Gap {g} (⬡{route[gaps[g]]}) has no bishop!"

    # ── Check: excluded hexes that are NOT on the route 
    #    shouldn't break anything ──
    # Simulate excluding some extra hexes from pawn coverage
    extra_excluded = []
    for w, h in enumerate(route):
        tier = int(fy.tiers[h]) if h < len(fy.tiers) else 0
        if tier <= 2 and w not in gaps:  # low-tier route hexes
            extra_excluded.append(w)
    
    if extra_excluded:
        all_gaps = sorted(set(gaps + extra_excluded))
        res2 = optimal_lighthouse_placement(
            all_gaps, route, myStuff.grid, myStuff.terr.elevations,
            n_bishops=2, profile=FOOD_PROFILE_V2)
        if res2 and res2['feasible']:
            print(f"  ✅ Expanded gaps ({len(all_gaps)} hexes): "
                  f"{res2['bishops_used']} bishops, "
                  f"fuel={res2['total_fuel']:.0f}")
        else:
            print(f"  ⚠️ Expanded gaps ({len(all_gaps)}) infeasible "
                  f"— may need more bishops")

    print("\n🎉 Arc traversal tests passed!")

test_lighthouse_arc_traversal()


In [ ]:
def find_stations_v3(pcr_result, n_pawns=4, n_bishops=1,
                     queen_diet=None, profile=None):
    """MIP: optimally assign pawns to cover the route, gaps go to bishops.
    
    Unlike v2 (greedy pair enumeration), this directly optimizes:
      - Which PCR candidates to activate (up to n_pawns)
      - Every route hex is either pawn-covered or a bishop gap
      - Minimize gaps (maximize pawn coverage)
    
    Handles 1-pawn, 2-pawn, or 3-pawn coverage per hex naturally.
    
    Returns list of segments in route order:
      {'type': 'station', ...}  — pawn(s) cover this stretch
      {'type': 'caravan', ...}  — bishop lighthouse gap
    """
    p = profile or FOOD_PROFILE_V2
    queen_diet = queen_diet or pcr_result.get('queen_diet', p['queen_diet'])
    b_cap = p.get('bishop_capacity', 140)
    b_diet = p.get('bishop_diet', 3.5)
    burn_rate = queen_diet + b_diet
    
    placements = pcr_result['placements']
    route = pcr_result['route']
    W = len(route)
    N = len(placements)
    
    if N == 0:
        print("No PCR candidates!"); return []
    
    # Pre-compute: for each (candidate, route_hex), the export
    export_at = {}
    for c in range(N):
        for w in placements[c]['coverage']:
            export_at[c, w] = placements[c]['export']
    
    # ── MIP ──
    m = pyo.ConcreteModel()
    m.C = pyo.RangeSet(0, N - 1)
    m.W = pyo.RangeSet(0, W - 1)
    
    # Place pawn at candidate c?
    m.place = pyo.Var(m.C, domain=pyo.Binary)
    
    # Route hex w is a gap (not pawn-covered)?
    m.gap = pyo.Var(m.W, domain=pyo.Binary)
    
    # ── Pawn budget ──
    m.budget = pyo.Constraint(
        expr=sum(m.place[c] for c in m.C) <= n_pawns)
    
    # ── Coverage: placed pawn exports + gap slack ≥ queen_diet ──
    # If gap[w]=0 (covered): Σ exports ≥ queen_diet
    # If gap[w]=1 (gap): constraint relaxed by big-M
    big_M = queen_diet * 2
    m.cover = pyo.ConstraintList()
    for w in m.W:
        supply = sum(export_at[c, w] * m.place[c]
                     for c in m.C if (c, w) in export_at)
        m.cover.add(supply + big_M * m.gap[w] >= queen_diet)
    
    # ── Bishop fuel budget: total gap hexes bounded ──
    max_gap_hexes = int(b_cap * n_bishops / burn_rate) if n_bishops > 0 else 0
    m.bishop_budget = pyo.Constraint(
        expr=sum(m.gap[w] for w in m.W) <= max_gap_hexes)
    
    # ── Objective: minimize gaps, tiebreak minimize pawns ──
    m.obj = pyo.Objective(
        expr=sum(m.gap[w] for w in m.W) * 100
             + sum(m.place[c] for c in m.C),
        sense=pyo.minimize)
    
    # ── Solve ──
    solver = pyo.SolverFactory('appsi_highs')
    result = solver.solve(m, load_solutions=False)
    
    if not _solver_ok(result):
        print(f"Stations v3 MIP infeasible: {N} candidates, "
              f"{n_pawns} pawns, {n_bishops} bishops")
        return []
    
    m.solutions.load_from(result)
    
    # ── Extract placed pawns ──
    placed = []
    for c in m.C:
        if pyo.value(m.place[c]) > 0.5:
            placed.append(placements[c])
    
    # ── Classify each route hex ──
    hex_type = []  # 'covered' or 'gap'
    for w in range(W):
        if pyo.value(m.gap[w]) > 0.5:
            hex_type.append('gap')
        else:
            hex_type.append('covered')
    
    # ── Group into contiguous segments ──
    segments = []
    i = 0
    while i < W:
        kind = hex_type[i]
        j = i
        while j < W and hex_type[j] == kind:
            j += 1
        # i..j-1 is a contiguous run of the same type
        
        if kind == 'covered':
            # Find which placed pawns serve this stretch
            stretch = set(range(i, j))
            serving = [p for p in placed
                       if p['coverage'] & stretch]
            segments.append({
                'type': 'station',
                'hexes': [p['hex'] for p in serving],
                'tiers': [p['tier'] for p in serving],
                'exports': [p['export'] for p in serving],
                'total_export': sum(p['export'] for p in serving),
                'min_w': i,
                'max_w': j - 1,
                'n_pawns': len(serving),
            })
        else:
            gap_len = j - i
            est_turns = max(1, (gap_len + 1) // 2)
            fuel_needed = est_turns * burn_rate
            load_w = i - 1 if i > 0 else 0
            dock_w = j if j < W else W - 1
            segments.append({
                'type': 'caravan',
                'gap_start': i,
                'gap_end': j - 1,
                'gap_hexes': route[i:j],
                'gap_len': gap_len,
                'load_w': load_w,
                'dock_w': dock_w,
                'load_hex': route[load_w],
                'dock_hex': route[dock_w],
                'est_turns': est_turns,
                'fuel_needed': round(fuel_needed, 1),
                'burn_rate': burn_rate,
            })
        i = j
    
    # ── Report ──
    n_sta = sum(1 for s in segments if s['type'] == 'station')
    n_car = sum(1 for s in segments if s['type'] == 'caravan')
    gap_total = sum(s['gap_len'] for s in segments if s['type'] == 'caravan')
    pawns_used = len(placed)
    
    print(f"Stations v3: {pawns_used}/{n_pawns} pawns → "
          f"{n_sta} stations + {n_car} caravans ({W} hex route)")
    
    for seg in segments:
        if seg['type'] == 'station':
            exp_str = '+'.join(f"{e:.1f}" for e in seg['exports'])
            print(f"  🏕  Station: ⬡{seg['hexes']} T{seg['tiers']} "
                  f"({seg['n_pawns']} pawns) "
                  f"export={exp_str}={seg['total_export']:.1f} "
                  f"route[{seg['min_w']}..{seg['max_w']}]")
        else:
            print(f"  🐪 Caravan: route[{seg['gap_start']}..{seg['gap_end']}] "
                  f"({seg['gap_len']}hex ~{seg['est_turns']}t) "
                  f"fuel={seg['fuel_needed']:.1f}")
    
    if n_car > 0:
        total_fuel = sum(s['fuel_needed'] for s in segments if s['type'] == 'caravan')
        budget = n_bishops * b_cap
        print(f"\n  Gap total: {gap_total} hexes, "
              f"fuel={total_fuel:.1f}/{budget:.0f} "
              f"{'✅' if total_fuel <= budget else '❌ over budget'}")
    else:
        print(f"  Full pawn coverage ✅")
    
    return segments


In [ ]:
def optimal_lighthouse_placement(gap_indices, route, grid, elevations,
                                 n_bishops=1, bishop_sight=4,
                                 queen_diet=7.0, bishop_diet=3.5,
                                 bishop_capacity=140.0,
                                 max_ring=2, profile=None):
    """MIP: place bishops as lighthouses to cover route gap hexes.
    
    Facility location problem:
      - Candidates: hexes within max_ring of any gap hex, on land
      - Each bishop parks at a candidate and covers gap hexes in sight range
      - Fuel constraint: bishop burns (queen_diet + bishop_diet) per turn
        the queen is in its cone, bounded by bishop_capacity
      - Objective: minimize bishops, tiebreak by elevation (higher = better)
    
    Args:
        gap_indices:  list of route indices that need bishop coverage
        route:        full route hex list
        grid:         HexGrid
        elevations:   elevation array
        n_bishops:    max bishops available
        bishop_sight: base sight range
        queen_diet, bishop_diet, bishop_capacity: food profile values
        max_ring:     how far from gap hexes to search for lighthouse sites
        profile:      optional FOOD_PROFILE_V2 override
    
    Returns dict with placements, coverage, fuel info, or None if infeasible.
    """
    if profile:
        queen_diet = profile.get('queen_diet', queen_diet)
        bishop_diet = profile.get('bishop_diet', bishop_diet)
        bishop_capacity = profile.get('bishop_capacity', bishop_capacity)
        bishop_sight = profile.get('pawn_sight', bishop_sight)  # or a bishop_sight key
    
    burn_per_turn = queen_diet + bishop_diet
    gap_set = set(gap_indices)
    gap_hexes = [route[w] for w in gap_indices]
    route_set = set(route)
    G = len(gap_indices)
    
    if G == 0:
        return {'placements': [], 'fuel': {}, 'total_fuel': 0, 'feasible': True}
    
    # ── Candidate lighthouse hexes ──
    seen = set()
    candidates = []
    
    for gh in gap_hexes:
        hp_g = grid.index_to_hexposition(gh)
        for ring in range(0, bishop_sight + max_ring + 1):
            ring_hps = [HexPosition.origin()] if ring == 0 else HexPosition.origin().ring(ring)
            for hp in ring_hps:
                idx = grid.hexposition_to_index(hp, gh)
                if idx < 0 or idx in seen: continue
                if idx in grid.invalidRegion: continue
                if idx >= len(elevations) or elevations[idx] <= 0: continue
                seen.add(idx)
                
                # Which gap hexes can this candidate see?
                hp_c = grid.index_to_hexposition(idx)
                can_see = []
                for wi, w in enumerate(gap_indices):
                    hp_r = grid.index_to_hexposition(route[w])
                    if hp_c.distance(hp_r) <= bishop_sight:
                        can_see.append(wi)  # index into gap_indices
                
                if not can_see: continue
                
                candidates.append({
                    'hex': idx,
                    'can_see': can_see,  # indices into gap_indices
                    'elevation': elevations[idx],
                    'on_route': idx in route_set,
                })
    
    if not candidates:
        print("No lighthouse candidates found!")
        return None
    
    C = len(candidates)
    B = n_bishops
    
    # ── Pyomo MIP ──
    m = pyo.ConcreteModel()
    m.C = pyo.RangeSet(0, C - 1)
    m.G = pyo.RangeSet(0, G - 1)
    m.B = pyo.RangeSet(0, B - 1)
    
    # place[b, c]: bishop b placed at candidate c?
    m.place = pyo.Var(m.B, m.C, domain=pyo.Binary)
    
    # serve[b, g]: bishop b feeds queen at gap index g?
    m.serve = pyo.Var(m.B, m.G, domain=pyo.Binary)
    
    # ── Each bishop placed at most one candidate ──
    m.one_site = pyo.Constraint(m.B, rule=lambda m, b:
        sum(m.place[b, c] for c in m.C) <= 1)
    
    # ── Every gap hex must be served by at least one bishop ──
    m.covered = pyo.Constraint(m.G, rule=lambda m, g:
        sum(m.serve[b, g] for b in m.B) >= 1)
    
    # ── Can only serve from a placed bishop within sight ──
    # Pre-compute: which (candidate, gap_index) pairs are valid
    valid_cg = set()
    for c_idx, cand in enumerate(candidates):
        for g_local in cand['can_see']:
            valid_cg.add((c_idx, g_local))
    
    m.sight = pyo.ConstraintList()
    for b in m.B:
        for g in m.G:
            # serve[b,g] ≤ Σ place[b,c] for candidates c that can see gap g
            covers = [c for c in m.C if (c, g) in valid_cg]
            if not covers:
                m.sight.add(m.serve[b, g] == 0)
            else:
                m.sight.add(m.serve[b, g] <= sum(m.place[b, c] for c in covers))
    
    # ── Fuel constraint: each bishop's total service ≤ capacity ──
    # Each gap hex served costs burn_per_turn (1 turn per hex at queen speed)
    m.fuel = pyo.Constraint(m.B, rule=lambda m, b:
        sum(m.serve[b, g] for g in m.G) * burn_per_turn <= bishop_capacity)
    
    # ── Objective: minimize bishops used, tiebreak by elevation ──
    # "bishop used" = placed anywhere
    m.used = pyo.Var(m.B, domain=pyo.Binary)
    m.link_used = pyo.Constraint(m.B, rule=lambda m, b:
        m.used[b] >= sum(m.place[b, c] for c in m.C))
    
    # Elevation bonus (normalized small so it's just a tiebreak)
    max_elev = max(c['elevation'] for c in candidates) or 1
    elev_bonus = sum(
        m.place[b, c] * (candidates[c]['elevation'] / max_elev) * 0.01
        for b in m.B for c in m.C
    )
    # Prefer off-route (small penalty for on-route)
    route_penalty = sum(
        m.place[b, c] * (0.001 if candidates[c]['on_route'] else 0)
        for b in m.B for c in m.C
    )
    
    m.obj = pyo.Objective(
        expr=sum(m.used[b] for b in m.B) - elev_bonus + route_penalty,
        sense=pyo.minimize)
    
    # ── Solve ──
    solver = pyo.SolverFactory('appsi_highs')
    result = solver.solve(m, tee=False)
    
    if not _solver_ok(result):
        print(f"Lighthouse MIP infeasible: {G} gap hexes, {C} candidates, {B} bishops")
        return None
    
    # ── Extract results ──
    placements = []
    fuel_usage = {}
    
    for b in m.B:
        for c in m.C:
            if pyo.value(m.place[b, c]) > 0.5:
                served = [g for g in m.G if pyo.value(m.serve[b, g]) > 0.5]
                served_route_w = [gap_indices[g] for g in served]
                fuel = len(served) * burn_per_turn
                
                placements.append({
                    'bishop': b,
                    'hex': candidates[c]['hex'],
                    'elevation': candidates[c]['elevation'],
                    'on_route': candidates[c]['on_route'],
                    'serves_gap': served,
                    'serves_route_w': served_route_w,
                    'fuel_used': round(fuel, 1),
                    'fuel_remaining': round(bishop_capacity - fuel, 1),
                })
                fuel_usage[b] = fuel
    
    total_fuel = sum(fuel_usage.values())
    bishops_used = len(placements)
    
    # ── Report ──
    print(f"Lighthouse MIP: {bishops_used}/{B} bishops cover {G} gap hexes")
    for p in placements:
        w_range = f"route[{min(p['serves_route_w'])}..{max(p['serves_route_w'])}]"
        print(f"  🔦 Bishop {p['bishop']}: ⬡{p['hex']} "
              f"(elev={p['elevation']:.0f}"
              f"{', ON route' if p['on_route'] else ''}) "
              f"serves {len(p['serves_gap'])} hexes {w_range} "
              f"fuel={p['fuel_used']}/{bishop_capacity:.0f} "
              f"(margin={p['fuel_remaining']})")
    
    return {
        'placements': placements,
        'fuel': fuel_usage,
        'total_fuel': round(total_fuel, 1),
        'bishops_used': bishops_used,
        'burn_per_turn': burn_per_turn,
        'feasible': True,
    }


So the goal is to make something so that queens can move about the board. We need an object like

```
class QueenCoverage

    coverage:[HexPostion:list of pawns and bishops whose sight cone includes the hex]

    def foodLevel(self,hex):
        the amount of food that pieces can give a hex based upon yield

    def turns(self,hex):
        the amount of turns that the bishop can remain at the spot for that hex

    def shift(sourceHexes,destHexes):
        pieces that were covering the source hexes need to move to optimally cover the dest hexes

```

So in the larger picture we want a travelling wherehose optimiztion so that pieces can move across the board in either a leapfrog fashion or we have traderoutes since we have enough pieces



## `HexMagic.game.mechanics.coverage` — Design

### Purpose
A logistics planning layer that lets queens move across the board by managing the pawn/bishop infrastructure needed to feed them. Supports both permanent strategic corridors and temporary what-if planning.

---

### Core Dataclasses

**`CorridorType`**
```python
class CorridorType(Enum):
    PERMANENT   # established route, kept between turns (e.g. oasis-to-oasis)
    TEMPORARY   # what-if / planning, discardable
```

**`Corridor`** — lightweight route descriptor
```python
@dataclass class Corridor:
    hexes:           list[int]       # ordered route hex indices
    kind:            CorridorType
    segments:        list[dict]      # stations + caravans from find_stations_v3
    # derived metrics
    n_stations:      int             # pawn-covered hex count
    n_gaps:          int             # bishop-dependent hex count
    fuel_needed:     float           # total bishop fuel required
    bishop_min_fuel: float           # weakest bishop's remaining fuel
    duration:        int             # turns corridor can be sustained
    congestion:      int             # pieces currently on the route
```

**`Target`** — why a queen wants to move somewhere
```python
@dataclass class Target:
    hex:      int
    purpose:  Literal['attack', 'settle', 'harvest', 'explore']
    priority: float           # player-set or auto-scored
    deadline: int | None      # turns until it matters (None = no urgency)
    payoff:   float           # expected value if reached
```

**`Assignment`** — output of the planning MIP
```python
@dataclass class Assignment:
    queen:     Piece
    target:    Target
    corridor:  Corridor
    turns_to_arrive: int
```

---

### `QueenCoverage` — the heavy object

```python
@dataclass class QueenCoverage:
    board:      GameBoard
    grid:       HexGrid
    elevations: list[float]
    food_tiers: list[float]
    profile:    dict              # FOOD_PROFILE_V2

    corridors:   list[Corridor]   # all known corridors
    targets:     list[Target]     # all known targets
    queens:      list[Piece]      # queens to route
    assignments: list[Assignment] # current plan (output of plan())
```

**Key methods:**

- `build_corridor(start, end, kind)` → runs `pawn_covered_route` + `find_stations_v3`, computes derived metrics, appends to `corridors`
- `add_target(hex, purpose, priority, deadline, payoff)` → appends to `targets`
- `plan()` → MIP queen↔target assignment; for each assignment finds/builds best corridor; populates `assignments`
- `food_level(hex)` → pawn+bishop export available at a given hex
- `shift(source_hexes, dest_hexes)` → rebalance piece assignments when a corridor extends or moves
- `__ft__()` → debug summary table (see below)

---

### `__ft__` Debug Table

One row per corridor:

| start→end | type | length | stations | gaps | fuel_needed | bishop_min_fuel | duration | congestion | ✅/❌ |
|---|---|---|---|---|---|---|---|---|---|
| ⬡158→⬡345 | PERM | 19 | 9 | 10 | 105.0 | 35.0 | 4t | 3 | ✅ |
| ⬡210→⬡448 | TEMP | 12 | 12 | 0 | 0.0 | — | ∞ | 0 | ✅ |

- `type` column distinguishes PERMANENT vs TEMPORARY
- `bishop_min_fuel` shows `—` when no gaps (no bishops needed)
- `duration` shows `∞` when fully pawn-covered
- ✅/❌ = corridor is currently feasible given available bishop fuel

---

### Planning MIP (inside `plan()`)

**Inputs:**
- Queens (pieces), Targets (hexes + metadata), Corridors (existing + buildable)

**Decision:**
- Which queen goes to which target via which corridor

**Objective:**
- Maximise `Σ (payoff × priority)` across assignments
- Penalise by corridor cost (fuel, gaps) and turns-to-arrive

**Constraints:**
- Each queen assigned at most one target
- Each target claimed by at most one queen
- Bishop fuel budget respected across all active corridors

---

### Key Design Decisions

1. **Corridor owns its route, `QueenCoverage` owns piece allocation** — avoids corridors fighting over the same bishop
2. **Hybrid corridor lifecycle** — PERMANENT corridors persist across turns; TEMPORARY ones are rebuilt each planning cycle
3. **`shift()` is the hard problem** — leapfrog vs trade-route semantics differ; initial impl can be greedy, optimised later
4. **`__ft__` doubles as player UI** — same data, different skin when we get to rendering


## Corridor

In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field
from enum import Enum
from typing import Literal
import heapq, math

import pyomo.environ as pyo
from fasthtml.common import *
from HexMagic.primitives import HexGrid, HexPosition
from HexMagic.game.data import Piece, PieceType, GameBoard

# ── Types ──────────────────────────────────────────────────────────────────

class CorridorType(Enum):
    PERMANENT = 'PERM'   # established route, kept between turns
    TEMPORARY = 'TEMP'   # what-if / planning, discardable


@dataclass
class Target:
    "Where a queen wants to go and why."
    hex:      int
    purpose:  str          # 'attack' | 'settle' | 'harvest' | 'explore'
    priority: float        = 1.0
    deadline: int | None   = None   # turns until it matters
    payoff:   float        = 0.0    # expected value if reached

    def __ft__(self):
        dl = f'{self.deadline}t' if self.deadline else '—'
        return Tr(
            Td(f'⬡{self.hex}'), Td(self.purpose),
            Td(f'{self.priority:.1f}'), Td(dl), Td(f'{self.payoff:.1f}'),
        )


@dataclass
class Assignment:
    "Queen → Target → Corridor — output of plan()."
    queen:           Piece
    target:          Target
    corridor:        Corridor
    turns_to_arrive: int = 0

    def __ft__(self):
        return Tr(
            Td(self.queen.name),
            Td(f'⬡{self.target.hex}'),
            Td(self.target.purpose),
            Td(f'{self.turns_to_arrive}t'),
            Td(f'⬡{self.corridor.start}→⬡{self.corridor.end}'),
            Td('✅' if self.corridor.feasible else '❌'),
        )


In [ ]:
@dataclass
class Corridor:
    hexes:           list[int]
    kind:            CorridorType
    segments:        list   = field(default_factory=list)
    n_stations:      int    = 0
    n_gaps:          int    = 0
    fuel_needed:     float  = 0.0
    bishop_min_fuel: float  = 0.0
    duration:        int    = 0
    congestion:      int    = 0
    valid:           bool   = True   # False = planning failed (no candidates etc)

    @property
    def feasible(self):
        if not self.valid: return False          # ← new
        if self.n_gaps == 0: return True
        return self.bishop_min_fuel >= self.fuel_needed

    @property
    def start(self):  return self.hexes[0]  if self.hexes else None
    @property
    def end(self):    return self.hexes[-1] if self.hexes else None
    @property
    def length(self): return len(self.hexes)
   

    


In [ ]:
@patch
def __ft__(self:Corridor):
    "Single table row — intended for use inside QueenCoverage.__ft__"
    if not self.valid:
        return Tr(
            Td(f'⬡{self.start}→⬡{self.end}'),
            Td(self.kind.value),
            Td(str(self.length)),
            Td('—'), Td('—'), Td('—'), Td('—'), Td('—'),
            Td(str(self.congestion)),
            Td('❌ no cover'),
        )

    dur = '∞'                        if self.n_gaps == 0 else f'{self.duration}t'
    bmf = '—'                        if self.n_gaps == 0 else f'{self.bishop_min_fuel:.0f}'
    ok  = '✅'                       if self.feasible   else '❌'
    return Tr(
        Td(f'⬡{self.start}→⬡{self.end}'),
        Td(self.kind.value),
        Td(str(self.length)),
        Td(str(self.n_stations)),
        Td(str(self.n_gaps)),
        Td(f'{self.fuel_needed:.1f}'),
        Td(bmf),
        Td(dur),
        Td(str(self.congestion)),
        Td(ok),
    )

In [ ]:
# Bridge existing public functions to the private names QueenCoverage expects
_solver_ok           = _solver_ok            # already private
_pawn_covered_route  = pawn_covered_route
_find_stations_v3    = find_stations_v3


In [ ]:
@dataclass
class CrossingPlan:
    """Complete resource budget + schedule for a queen to walk a corridor."""
    corridor:         Corridor
    roster:           list[dict]       # [{piece_type, hex, facing, instructions, role}]
    queen_start_turn: int
    schedule:         pd.DataFrame     # validated budget schedule
    feasible:         bool
    queen_arrival:    int              # turn queen reaches end
    food_margin:      float            # min food queen has at any point
    bishop_fuel_used: float
    
    def pieces_needed(self):
        """Summary: how many of each type."""
        from collections import Counter
        return Counter(r['piece_type'] for r in self.roster if r['role'] != 'queen')
    
    def __ft__(self):
        needed = self.pieces_needed()
        summary = ', '.join(f"{v}× {k.name}" for k, v in needed.items())
        status = '✅' if self.feasible else '❌'
        
        info = Table(Tbody(
            Tr(Td(B('Route')),    Td(f'⬡{self.corridor.start}→⬡{self.corridor.end} ({self.corridor.length} hexes)')),
            Tr(Td(B('Roster')),   Td(summary or 'queen only')),
            Tr(Td(B('Queen go')), Td(f'turn {self.queen_start_turn}')),
            Tr(Td(B('Arrival')),  Td(f'turn {self.queen_arrival}')),
            Tr(Td(B('Food min')), Td(f'{self.food_margin:.1f}')),
            Tr(Td(B('Bishop fuel')), Td(f'{self.bishop_fuel_used:.0f}')),
            Tr(Td(B('Status')),   Td(status)),
        ), cls=TableT.sm)
        
        timetable = schedule_timetable(self.schedule, title='Crossing Schedule')
        return Div(
            Card(info, header=CardTitle(f'Crossing Plan {status}')),
            timetable,
        )


## QueenCoverage

In [ ]:
@dataclass
class QueenCoverage:
    "Logistics planning layer: corridors, targets, and queen assignments."
    board:      GameBoard
    grid:       object        # HexGrid
    elevations: list
    food_tiers: list
    profile:    dict          # FOOD_PROFILE_V2

    corridors:   list = field(default_factory=list)   # list[Corridor]
    targets:     list = field(default_factory=list)   # list[Target]
    queens:      list = field(default_factory=list)   # list[Piece]
    assignments: list = field(default_factory=list)   # list[Assignment]

    # ── Internal helpers ──────────────────────────────────────────────────

    def _countries(self):
        return getattr(self.board, 'terrain', None) and \
               self.board.terrain.fields.get('country')

    def _find_existing_corridor(self, start: int, end: int) -> Corridor | None:
        "Return first corridor whose endpoints match, or None."
        return next((c for c in self.corridors
                     if c.start == start and c.end == end), None)


 

    def add_target(self, hex: int, purpose: str,
                   priority: float = 1.0,
                   deadline: int | None = None,
                   payoff: float = 0.0) -> Target:
        "Register a target hex."
        t = Target(hex=hex, purpose=purpose, priority=priority,
                   deadline=deadline, payoff=payoff)
        self.targets.append(t)
        return t

    def food_level(self, hex: int) -> float:
        "Net pawn export available at a given hex (single pawn, no pair)."
        tier = int(self.food_tiers[hex]) if 0 <= hex < len(self.food_tiers) else 0
        return max(0.0, self.profile.get('pawn_harvest', 1.3) * tier
                      - self.profile.get('pawn_diet', 0.7))

    def assign_bishop(self, corridor: Corridor, bishop: Piece):
        "Update bishop_min_fuel when a bishop is assigned to a corridor."
        fuel = getattr(bishop, 'food',
                       self.profile.get('bishop_capacity', 140.0))
        prev = corridor.bishop_min_fuel
        corridor.bishop_min_fuel = (fuel if prev == self.profile.get('bishop_capacity', 140.0)
                                    else min(prev, fuel))

    def update_congestion(self):
        "Recount pieces on each corridor's hexes."
        for c in self.corridors:
            rs = set(c.hexes)
            c.congestion = sum(
                1 for country in self.board.kingdoms
                for settlement in country.settlements
                for piece in settlement.citizens
                if piece.location in rs)

    def shift(self, source_hexes: list[int], dest_hexes: list[int]):
        "Rebalance pieces from source to dest hexes. (stub — TBD)"
        print(f"shift(): {len(source_hexes)}→{len(dest_hexes)} hexes "
              f"(not yet implemented)")

    


In [ ]:
@patch
def plan(self: QueenCoverage) -> list:
    "MIP: assign queens to targets via corridors. Returns assignments."
    if not self.queens or not self.targets:
        print("plan(): need queens and targets first"); return []

    Q, T = len(self.queens), len(self.targets)

    # ── Build/find corridors for all pairs ──
    pair_corridors = {}
    for qi, queen in enumerate(self.queens):
        for ti, target in enumerate(self.targets):
            # ① Prefer an existing feasible corridor that ends at this target
            existing = next(
                (c for c in self.corridors
                 if c.end == target.hex and c.feasible),
                None)
            if existing:
                print(f"  plan(): reusing {existing.kind.value} corridor "
                      f"⬡{existing.start}→⬡{existing.end} "
                      f"for {queen.name}")
                pair_corridors[qi, ti] = existing
                continue

            # ② No existing — build a TEMP from queen's current position
            c = self.build_corridor(
                queen.location, target.hex,
                kind=CorridorType.TEMPORARY)
            if c:
                pair_corridors[qi, ti] = c

    feasible_pairs = [(qi, ti) for (qi, ti), c in pair_corridors.items()
                      if c.feasible]
    if not feasible_pairs:
        print("plan(): no feasible assignments found"); return []

    # ── Assignment MIP ──
    m = pyo.ConcreteModel()
    m.Q = pyo.RangeSet(0, Q - 1)
    m.T = pyo.RangeSet(0, T - 1)
    m.assign = pyo.Var(m.Q, m.T, domain=pyo.Binary)

    m.infeas = pyo.ConstraintList()
    for q in range(Q):
        for t in range(T):
            if (q, t) not in pair_corridors or not pair_corridors[q, t].feasible:
                m.infeas.add(m.assign[q, t] == 0)

    m.one_target = pyo.Constraint(m.Q, rule=lambda m, q:
        sum(m.assign[q, t] for t in m.T) <= 1)
    m.one_queen  = pyo.Constraint(m.T, rule=lambda m, t:
        sum(m.assign[q, t] for q in m.Q) <= 1)

    cost_w = 0.01
    m.obj = pyo.Objective(
        expr=sum(
            m.assign[q, t] * (
                self.targets[t].payoff * self.targets[t].priority
                - cost_w * pair_corridors[q, t].fuel_needed
            )
            for q, t in feasible_pairs),
        sense=pyo.maximize)

    result = pyo.SolverFactory('appsi_highs').solve(m)
    if not _solver_ok(result):
        print("plan(): MIP infeasible"); return []

    self.assignments = []
    for q in range(Q):
        for t in range(T):
            if pyo.value(m.assign[q, t]) > 0.5:
                corr = pair_corridors[q, t]
                # turns = distance queen→corridor entry + corridor length
                queen = self.queens[q]
                hp_q  = self.grid.index_to_hexposition(queen.location)
                hp_s  = self.grid.index_to_hexposition(corr.start)
                entry_dist = hp_q.distance(hp_s)
                turns = entry_dist + corr.length
                self.assignments.append(
                    Assignment(queen=queen, target=self.targets[t],
                               corridor=corr, turns_to_arrive=turns))

    print(f"plan(): {len(self.assignments)}/{min(Q,T)} assignments")
    for a in self.assignments:
        print(f"  {a.queen.name} → ⬡{a.target.hex} "
              f"({a.target.purpose}) via ⬡{a.corridor.start}→⬡{a.corridor.end} "
              f"~{a.turns_to_arrive}t")
    return self.assignments


In [ ]:
def _split_contiguous(indices):
    """Split a sorted list of ints into contiguous runs."""
    if not indices: return []
    segs, cur = [], [indices[0]]
    for g in indices[1:]:
        if g == cur[-1] + 1: cur.append(g)
        else: segs.append(cur); cur = [g]
    segs.append(cur)
    return segs


@patch
def _make_bishop_shadow(self: QueenCoverage, park_hex, served_ws, route,
                        profile, queen, shadow_id, shadow_name):
    """Create a bishop shadow that sprints to park_hex, faces the gap, parks.

    Returns (PieceShadow, arrival_turn) or (None, 0) if no path.
    """
    bp = simple_hex_path(route[0], park_hex, self.grid, self.elevations)
    if not bp: return None, 0

    sprint, end_f = route_to_instructions(bp, self.grid, 0)

    # Face toward the middle of the served gap for max sight coverage
    mid_w = served_ws[len(served_ws) // 2]
    want_f = _facing_toward(self.grid, park_hex, route[mid_w])
    d = (want_f - end_f) % 6
    rot = ([Instruction.ROT_R.value] * d if d <= 3
           else [Instruction.ROT_L.value] * (6 - d))

    park = [Instruction.HARVEST.value, Instruction.GIVE.value] * 60

    bs = PieceShadow(
        id=shadow_id, name=shadow_name,
        piece_type=PieceType.BISHOP, birth_year=0,
        location=route[0], facing=0,
        food=profile['bishop_capacity'],
        food_capacity=profile['bishop_capacity'],
        diet=profile.get('bishop_diet', 3.5),
        health=100, max_health=100, move_strength=3,
        harvest_strength=profile.get('bishop_harvest', 0.7),
        sight=profile['pawn_sight'],
        owner_id=queen.owner_id, flag=queen.flag)
    bs.instructions = InstructionList(sprint + rot + park, cursor=0, patrol=False)

    arrival = math.ceil(len(bp) / 3) + 2  # bishop speed 3, +safety
    return bs, arrival


In [ ]:
@patch
def propose_crossing(self: QueenCoverage, corridor: Corridor,
                     queen: Piece,
                     available_pawns: int = 5,
                     available_bishops: int = 2,
                     max_retries: int = 3,
                     n_turns: int = 60) -> CrossingPlan:
    """Propose a validated crossing plan with origin + destination camps.

    Pawn roles (allocated from available_pawns budget):
      origin_camp:    park near route start, feed queen while she waits
      advance_party:  sprint to dest camp hexes BEFORE queen departs
      station:        mid-route food stations from corridor segments

    Queen departs only after bishops are parked AND advance party has
    arrived + harvested at least once. Origin camp keeps her full until then.
    """
    route = corridor.hexes
    segments = corridor.segments
    profile = self.profile
    grid = self.grid
    elevs = self.elevations
    food_tiers = self.food_tiers

    # ── Find best food hexes near an endpoint ──────────────────────────
    def _find_camp_hexes(center, n=2, max_ring=3):
        route_set = set(route)
        cands = []
        for ring in range(1, max_ring + 1):
            for hp in HexPosition.origin().ring(ring):
                idx = grid.hexposition_to_index(hp, center)
                if idx < 0 or idx in grid.invalidRegion: continue
                if idx >= len(elevs) or elevs[idx] <= 0: continue
                if idx in route_set: continue
                tier = int(food_tiers[idx]) if idx < len(food_tiers) else 0
                if tier > 0:
                    cands.append((idx, tier))
        cands.sort(key=lambda x: -x[1])
        return cands[:n]

    origin_camps = _find_camp_hexes(route[0])
    dest_camps   = _find_camp_hexes(route[-1])

    # ── Pawn budget allocation ─────────────────────────────────────────
    # Count mid-station slots from corridor segments
    station_hexes = []
    for seg in segments:
        if seg['type'] == 'station':
            station_hexes.extend(seg['hexes'])

    n_origin  = min(len(origin_camps), 2)
    n_dest    = min(len(dest_camps), 2)
    n_station = min(len(station_hexes), available_pawns - n_origin - n_dest)
    # Clamp to budget
    total_pawns = n_origin + n_dest + n_station
    if total_pawns > available_pawns:
        # Trim stations first, then dest, then origin
        n_station = max(0, available_pawns - n_origin - n_dest)
        if n_origin + n_dest > available_pawns:
            n_dest = max(0, available_pawns - n_origin)

    print(f"  Camp plan: {n_origin} origin + {n_station} station "
          f"+ {n_dest} advance → {n_origin + n_station + n_dest}/{available_pawns} pawns")
    print(f"  Origin camps: {[(h, f'T{t}') for h, t in origin_camps[:n_origin]]}")
    print(f"  Dest camps:   {[(h, f'T{t}') for h, t in dest_camps[:n_dest]]}")

    best_plan = None

    for attempt in range(max_retries + 1):
        extra_wait = attempt * 2
        roster, shadows = [], []
        pawn_n = 0

        # ── 1. Queen shadow (instructions set in step 6) ──────────────
        qs = PieceShadow.from_piece(queen)
        qs.location = route[0]
        qs.food = profile['queen_capacity']
        roster.append(dict(piece_type=PieceType.QUEEN, hex=route[0],
                           facing=qs.facing, role='queen'))

        # ── 2. Origin camp pawns (park + HARVEST+GIVE) ────────────────
        for hx, tier in origin_camps[:n_origin]:
            facing = _facing_toward(grid, hx, route[0])
            ps = PieceShadow(
                id=f'camp_o_{pawn_n}', name=f'OriginPawn_{pawn_n}',
                piece_type=PieceType.PAWN, birth_year=0,
                location=hx, facing=facing,
                food=profile['pawn_capacity'],
                food_capacity=profile['pawn_capacity'],
                diet=profile['pawn_diet'], health=100, max_health=100,
                move_strength=2, harvest_strength=profile['pawn_harvest'],
                sight=profile['pawn_sight'],
                owner_id=queen.owner_id, flag=queen.flag)
            ps.instructions = InstructionList(
                [Instruction.HARVEST.value, Instruction.GIVE.value],
                cursor=0, patrol=True)
            shadows.append(ps)
            roster.append(dict(piece_type=PieceType.PAWN, hex=hx,
                               facing=facing, role='origin_camp'))
            pawn_n += 1

        # ── 3. Advance party (sprint to dest camp, then park) ─────────
        advance_arrival = 0
        for hx, tier in dest_camps[:n_dest]:
            sprint_path = simple_hex_path(route[0], hx, grid, elevs)
            if not sprint_path:
                continue
            sprint_rules, end_facing = route_to_instructions(
                sprint_path, grid, 0)

            want_f = _facing_toward(grid, hx, route[-1])
            diff = (want_f - end_facing) % 6
            rot = ([Instruction.ROT_R.value] * diff if diff <= 3
                   else [Instruction.ROT_L.value] * (6 - diff))

            park = [Instruction.HARVEST.value, Instruction.GIVE.value] * 80

            ps = PieceShadow(
                id=f'camp_d_{pawn_n}', name=f'DestPawn_{pawn_n}',
                piece_type=PieceType.PAWN, birth_year=0,
                location=route[0], facing=0,
                food=profile['pawn_capacity'],
                food_capacity=profile['pawn_capacity'],
                diet=profile['pawn_diet'], health=100, max_health=100,
                move_strength=2, harvest_strength=profile['pawn_harvest'],
                sight=profile['pawn_sight'],
                owner_id=queen.owner_id, flag=queen.flag)
            ps.instructions = InstructionList(
                sprint_rules + rot + park, cursor=0, patrol=False)
            shadows.append(ps)
            roster.append(dict(piece_type=PieceType.PAWN, hex=hx,
                               facing=want_f, role='advance_party'))
            pawn_n += 1

            # Estimate arrival: path_len / speed + rotate + 1 harvest
            arr = math.ceil(len(sprint_path) / 2) + len(rot) + 2
            advance_arrival = max(advance_arrival, arr)

        # ── 4. Mid-route station pawns ────────────────────────────────
        for seg in segments:
            if seg['type'] != 'station': continue
            for hx in seg['hexes']:
                if pawn_n >= n_origin + n_dest + n_station: break
                mid_w = (seg['min_w'] + seg['max_w']) // 2
                facing = _facing_toward(grid, hx, route[mid_w])
                ps = PieceShadow(
                    id=f'station_{pawn_n}', name=f'StationPawn_{pawn_n}',
                    piece_type=PieceType.PAWN, birth_year=0,
                    location=hx, facing=facing,
                    food=profile['pawn_capacity'],
                    food_capacity=profile['pawn_capacity'],
                    diet=profile['pawn_diet'], health=100, max_health=100,
                    move_strength=2, harvest_strength=profile['pawn_harvest'],
                    sight=profile['pawn_sight'],
                    owner_id=queen.owner_id, flag=queen.flag)
                ps.instructions = InstructionList(
                    [Instruction.HARVEST.value, Instruction.GIVE.value],
                    cursor=0, patrol=True)
                shadows.append(ps)
                roster.append(dict(piece_type=PieceType.PAWN, hex=hx,
                                   facing=facing, role='station'))
                pawn_n += 1

        # ── 5. Bishops (lighthouse MIP or escort fallback) ────────────
        all_gaps = []
        for seg in segments:
            if seg['type'] == 'caravan':
                all_gaps.extend(range(seg['gap_start'], seg['gap_end'] + 1))

        bishop_ready = 0
        bishop_mode = 'none'

        if all_gaps and available_bishops > 0:
            lh = optimal_lighthouse_placement(
                all_gaps, route, grid, elevs,
                n_bishops=available_bishops, profile=profile)

            if lh and lh['feasible']:
                bishop_mode = 'lighthouse'
                for pl in lh['placements']:
                    bs, arr = self._make_bishop_shadow(
                        pl['hex'], pl['serves_route_w'], route,
                        profile, queen, f'blh_{pl["bishop"]}',
                        f'Bishop_{pl["bishop"]}')
                    if bs:
                        shadows.append(bs)
                        roster.append(dict(piece_type=PieceType.BISHOP,
                                           hex=pl['hex'], facing=bs.facing,
                                           role='lighthouse'))
                        bishop_ready = max(bishop_ready, arr)
            else:
                bishop_mode = 'escort'
                gap_segs = _split_contiguous(all_gaps)
                for bi, gseg in enumerate(gap_segs[:available_bishops]):
                    mid_w = gseg[len(gseg) // 2]
                    park_hex = route[mid_w]
                    bs, arr = self._make_bishop_shadow(
                        park_hex, gseg, route, profile, queen,
                        f'besc_{bi}', f'Escort_{bi}')
                    if bs:
                        shadows.append(bs)
                        roster.append(dict(piece_type=PieceType.BISHOP,
                                           hex=park_hex, facing=bs.facing,
                                           role='escort'))
                        bishop_ready = max(bishop_ready, arr)

        # ── 6. Queen timing: depart after bishops + advance party ─────
        queen_start = max(bishop_ready, advance_arrival) + extra_wait
        wait  = [Instruction.PAUSE.value] * queen_start
        march, _ = route_to_instructions(route, grid, qs.facing)
        qs.instructions = InstructionList(wait + march, cursor=0, patrol=False)
        shadows.insert(0, qs)

        # ── 7. Ownership + squad ──────────────────────────────────────
        for s in shadows:
            s.owner_id = queen.owner_id
            s.flag = queen.flag

        shadow_squad = Squad.squads(1, queen.flag)[0]
        shadow_squad.pieces = list(shadows)

        initial = {s.id: dict(location=s.location, facing=s.facing,
                               food=s.food, health=s.health)
                   for s in shadows}

        # ── 8. Simulate ──────────────────────────────────────────────
        sim = FoodSimulator(
            grid=grid, elevations=elevs,
            food_tiers=food_tiers, pieces=shadows,
            countries=self._countries(),
            squads=[shadow_squad],
            callbacks={shadow_squad.id: food_redistribute_adjacent},
        )
        sim.run(num_turns=n_turns)

        # ── 9. Evaluate ──────────────────────────────────────────────
        q_snap = [s['piece_states'].get(qs.id) for s in sim.snapshots]
        arrived = any(s and s['location'] == route[-1] for s in q_snap)
        alive   = q_snap[-1] and q_snap[-1]['health'] > 0 if q_snap else False
        min_food = min((s['food'] for s in q_snap if s and s['health'] > 0),
                       default=0)

        arrival_turn = n_turns
        for i, s in enumerate(q_snap):
            if s and s['location'] == route[-1]:
                arrival_turn = i + 1; break

        bishop_fuel = 0
        if sim.snapshots:
            for s in shadows:
                if s.piece_type == PieceType.BISHOP:
                    ls = sim.snapshots[-1]['piece_states'].get(s.id)
                    if ls:
                        bishop_fuel += profile['bishop_capacity'] - ls.get('food', 0)

        adapter = SimAdapter(sim, corridors=[corridor])
        schedule = adapter.extract(schedule_id=f'crossing_v{attempt}',
                                    initial_states=initial)

        feasible = arrived and alive
        plan = CrossingPlan(
            corridor=corridor, roster=roster,
            queen_start_turn=queen_start, schedule=schedule,
            feasible=feasible, queen_arrival=arrival_turn,
            food_margin=min_food, bishop_fuel_used=bishop_fuel)
        best_plan = plan

        if feasible:
            roles = {}
            for r in roster:
                roles[r['role']] = roles.get(r['role'], 0) + 1
            print(f"✅ Crossing (attempt {attempt}, {bishop_mode}): "
                  f"roles={roles}, start=t{queen_start}, "
                  f"arrive=t{arrival_turn}, min_food={min_food:.1f}")
            return plan

        reason = 'died' if not alive else 'never arrived'
        last_hex = None
        for st in reversed(q_snap):
            if st and st['health'] > 0: last_hex = st['location']; break
        print(f"❌ Attempt {attempt} ({bishop_mode}): {reason}, "
              f"last=⬡{last_hex}, min_food={min_food:.1f}, "
              f"wait={queen_start}t")

    print(f"⚠️ All {max_retries + 1} attempts failed")
    return best_plan


Can you fix propose_cross so that pieces have  the same settlement id and they will feed each other.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

The problem is that `propose_crossing` creates a `FoodSimulator` without a squad or callbacks, so the food redistribution logic never fires — pieces don't feed each other even though the real sim uses `squads=[...]` and `callbacks={...: food_redistribute_adjacent}`.

Two things to fix:

1. **Create a shadow squad** containing all the shadow pieces
2. **Pass it + a callback** to the `FoodSimulator`

Here's the corrected sim setup section (step 5 inside `propose_crossing`):

```python
        # ── 5. Run shadow sim ──
        initial = {s.id: dict(location=s.location, facing=s.facing,
                               food=s.food, health=s.health)
                   for s in shadows}

        # Wrap shadows in a squad so food sharing works
        shadow_squad = Squad.squads(1, queen.flag)[0]
        shadow_squad.pieces = list(shadows)

        sim = FoodSimulator(
            grid=self.grid, elevations=self.elevations,
            food_tiers=self.food_tiers, pieces=shadows,
            countries=self._countries(),
            squads=[shadow_squad],
            callbacks={shadow_squad.id: food_redistribute_adjacent},
        )
        sim.run(num_turns=n_turns)
```

Also make sure the `owner_id` on every shadow matches so that any ownership checks pass. Add this right before the sim block (after all shadows are built):

```python
        # Ensure all shadows share the same owner so they cooperate
        for s in shadows:
            s.owner_id = qs.owner_id
            s.flag = qs.flag
```

The key insight: the existing sim calls (`_63fbaa4d`) pass `squads=[pcr_squad]` and `callbacks={pcr_squad.id: food_redistribute_adjacent}`, which is what triggers adjacent food transfers. Without those two args the pieces just sit next to each other and starve independently.

can you write the full function?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->



```python
@patch
def propose_crossing(self: QueenCoverage, corridor: Corridor,
                     queen: Piece,
                     available_pawns: int = 4,
                     available_bishops: int = 2,
                     max_retries: int = 3,
                     n_turns: int = 50) -> CrossingPlan:
    """Propose a resource budget that gets the queen across alive.
    
    Flow:
      1. Allocate pawns to stations (from corridor segments)
      2. Allocate bishops to lighthouse gaps
      3. Compute queen start turn (after bishops are parked)
      4. Build shadow pieces with instructions
      5. Wrap in a squad so pieces feed each other
      6. Run throwaway sim to verify
      7. If queen dies: retry with +1 bishop or +1 wait turn
      8. Return validated CrossingPlan
    """
    route = corridor.hexes
    segments = corridor.segments
    profile = self.profile
    
    for attempt in range(max_retries + 1):
        roster = []
        shadows = []
        
        # ── 1. Queen shadow (instructions set later) ──
        qs = PieceShadow.from_piece(queen)
        qs.location = route[0]
        qs.food = profile['queen_capacity']
        roster.append(dict(piece_type=PieceType.QUEEN, hex=route[0],
                           facing=qs.facing, role='queen'))
        
        # ── 2. Pawns at stations ──
        pawn_id = 0
        for seg in segments:
            if seg['type'] != 'station': continue
            for hex_idx in seg['hexes'][:available_pawns - pawn_id]:
                mid_w = (seg['min_w'] + seg['max_w']) // 2
                facing = _facing_toward(self.grid, hex_idx, route[mid_w])
                
                ps = PieceShadow(
                    id=f'plan_pawn_{pawn_id}', name=f'Pawn_{pawn_id}',
                    piece_type=PieceType.PAWN, birth_year=0,
                    location=hex_idx, facing=facing,
                    food=profile['pawn_capacity'], food_capacity=profile['pawn_capacity'],
                    diet=profile['pawn_diet'], health=100, max_health=100,
                    move_strength=2, harvest_strength=profile['pawn_harvest'],
                    sight=profile['pawn_sight'],
                    owner_id=queen.owner_id, flag=queen.flag,
                )
                ps.instructions = InstructionList(
                    [Instruction.HARVEST.value, Instruction.GIVE.value],
                    cursor=0, patrol=True)
                
                shadows.append(ps)
                roster.append(dict(piece_type=PieceType.PAWN, hex=hex_idx,
                                   facing=facing, role='station'))
                pawn_id += 1
        
        # ── 3. Bishops at lighthouses ──
        all_gap_indices = []
        for seg in segments:
            if seg['type'] == 'caravan':
                all_gap_indices.extend(range(seg['gap_start'], seg['gap_end'] + 1))
        
        n_bishops = min(available_bishops, max(1, attempt + 1))  # retry adds bishops
        bishop_ready_turn = 0
        
        if all_gap_indices:
            lh = optimal_lighthouse_placement(
                all_gap_indices, route, self.grid, self.elevations,
                n_bishops=n_bishops, profile=profile)
            
            if lh and lh['feasible']:
                for pl in lh['placements']:
                    lh_hex = pl['hex']
                    bishop_path = simple_hex_path(route[0], lh_hex,
                                                  self.grid, self.elevations)
                    if not bishop_path:
                        continue
                    
                    sprint_rules, end_facing = route_to_instructions(
                        bishop_path, self.grid, 0)
                    
                    gap_mid = route[(min(pl['serves_route_w']) + max(pl['serves_route_w'])) // 2]
                    face_route = _facing_toward(self.grid, lh_hex, gap_mid)
                    diff = (face_route - end_facing) % 6
                    rot = ([Instruction.ROT_R.value] * diff if diff <= 3
                           else [Instruction.ROT_L.value] * (6 - diff))
                    
                    park = [Instruction.HARVEST.value, Instruction.GIVE.value] * 60
                    
                    bs = PieceShadow(
                        id=f'plan_bishop_{pl["bishop"]}',
                        name=f'Bishop_{pl["bishop"]}',
                        piece_type=PieceType.BISHOP, birth_year=0,
                        location=route[0], facing=0,
                        food=profile['bishop_capacity'],
                        food_capacity=profile['bishop_capacity'],
                        diet=profile['bishop_diet'], health=100, max_health=100,
                        move_strength=3, harvest_strength=profile.get('bishop_harvest', 0.7),
                        sight=profile['pawn_sight'],
                        owner_id=queen.owner_id, flag=queen.flag,
                    )
                    bs.instructions = InstructionList(
                        sprint_rules + rot + park, cursor=0, patrol=False)
                    
                    shadows.append(bs)
                    roster.append(dict(piece_type=PieceType.BISHOP, hex=lh_hex,
                                       facing=face_route, role='lighthouse'))
                    
                    arrival = math.ceil(len(bishop_path) / 3) + 2  # +rot +safety
                    bishop_ready_turn = max(bishop_ready_turn, arrival)
        
        # ── 4. Queen start turn + instructions ──
        queen_start = bishop_ready_turn + attempt  # retry adds wait
        wait = [Instruction.PAUSE.value] * queen_start
        march, _ = route_to_instructions(route, self.grid, qs.facing)
        qs.instructions = InstructionList(wait + march, cursor=0, patrol=False)
        shadows.insert(0, qs)
        
        # ── 5. Ensure all shadows share owner/flag so they cooperate ──
        for s in shadows:
            s.owner_id = queen.owner_id
            s.flag = queen.flag
        
        # ── 6. Build a squad wrapper so food sharing triggers ──
        shadow_squad = Squad.squads(1, queen.flag)[0]
        shadow_squad.pieces = list(shadows)
        
        initial = {s.id: dict(location=s.location, facing=s.facing,
                               food=s.food, health=s.health)
                   for s in shadows}
        
        sim = FoodSimulator(
            grid=self.grid, elevations=self.elevations,
            food_tiers=self.food_tiers, pieces=shadows,
            countries=self._countries(),
            squads=[shadow_squad],
            callbacks={shadow_squad.id: food_redistribute_adjacent},
        )
        sim.run(num_turns=n_turns)
        
        # ── 7. Check queen survived + arrived ──
        q_snap = [s['piece_states'].get(qs.id) for s in sim.snapshots]
        arrived = any(s and s['location'] == route[-1] for s in q_snap)
        alive = q_snap[-1] and q_snap[-1]['health'] > 0 if q_snap else False
        min_food = min((s['food'] for s in q_snap if s and s['health'] > 0),
                       default=0)
        
        # Find arrival turn
        arrival_turn = n_turns
        for i, s in enumerate(q_snap):
            if s and s['location'] == route[-1]:
                arrival_turn = i + 1
                break
        
        # Bishop fuel used
        bishop_fuel = 0
        if sim.snapshots:
            for s in shadows:
                if s.piece_type == PieceType.BISHOP:
                    last_state = sim.snapshots[-1]['piece_states'].get(s.id)
                    if last_state:
                        bishop_fuel += profile['bishop_capacity'] - last_state.get('food', 0)
        
        # Extract schedule
        adapter = SimAdapter(sim, corridors=[corridor])
        schedule = adapter.extract(schedule_id=f'crossing_v{attempt}',
                                    initial_states=initial)
        
        feasible = arrived and alive
        plan = CrossingPlan(
            corridor=corridor, roster=roster,
            queen_start_turn=queen_start, schedule=schedule,
            feasible=feasible, queen_arrival=arrival_turn,
            food_margin=min_food, bishop_fuel_used=bishop_fuel,
        )
        
        if feasible:
            print(f"✅ Crossing plan (attempt {attempt}): "
                  f"{plan.pieces_needed()}, queen starts t{queen_start}, "
                  f"arrives t{arrival_turn}, min_food={min_food:.1f}")
            return plan
        else:
            reason = 'died' if not alive else 'never arrived'
            print(f"❌ Attempt {attempt}: {reason} "
                  f"(min_food={min_food:.1f}, start=t{queen_start})")
    
    print(f"⚠️ All {max_retries + 1} attempts failed")
    return plan  # return last attempt even if infeasible
```

Changes from the original:

1. **`owner_id` and `flag` set on creation** — pawns and bishops now get `owner_id=queen.owner_id, flag=queen.flag` in their `PieceShadow(...)` constructors instead of inheriting the default `0` / `None`.

2. **Explicit ownership pass after all shadows built** (step 5) — `for s in shadows: s.owner_id = queen.owner_id; s.flag = queen.flag` — belt-and-suspenders in case any piece was built without it.

3. **Shadow squad wraps all pieces** (step 6) — `shadow_squad.pieces = list(shadows)` so the sim treats them as one cooperative unit.

4. **`FoodSimulator` gets `squads` and `callbacks`** — `squads=[shadow_squad]` and `callbacks={shadow_squad.id: food_redistribute_adjacent}` so adjacent food transfers actually fire each turn.

5. **Bishop fuel calc simplified** — the original had a broken nested comprehension; now it's a straightforward loop over bishop shadows checking the last snapshot.

In [ ]:
@patch
def _metrics_from_segments(self:QueenCoverage, segments: list, pcr: dict | None = None) -> dict:
    "Derive Corridor metrics from _find_stations_v3 output."
    p = self.profile
    b_cap = p.get('bishop_capacity', 140.0)

    # Empty segments with a PCR that had gaps = planning failed, not free
    if not segments:
        has_gaps = pcr and len(pcr.get('gaps', [])) > 0
        return dict(n_stations=0, n_gaps=0, fuel_needed=0.0,
                    bishop_min_fuel=0.0, duration=0,
                    valid=not has_gaps)   # ← False if gaps existed but unplanned

    n_stations = sum(s['max_w'] - s['min_w'] + 1
                    for s in segments if s['type'] == 'station')
    gap_segs   = [s for s in segments if s['type'] == 'caravan']
    n_gaps     = sum(s['gap_len']     for s in gap_segs)
    fuel_needed= sum(s['fuel_needed'] for s in gap_segs)
    duration   = (0 if n_gaps == 0
                else int(b_cap / max(fuel_needed, 0.1)))
    return dict(n_stations=n_stations, n_gaps=n_gaps,
                fuel_needed=fuel_needed,
                bishop_min_fuel=b_cap,
                duration=duration,
                valid=True)

In [ ]:
# ── Public API ────────────────────────────────────────────────────────
@patch
def build_corridor(self:QueenCoverage, start: int, end: int,
                    kind: CorridorType = CorridorType.TEMPORARY,
                    n_pawns: int = 4,
                    n_bishops: int = 1) -> Corridor | None:
    "Build a corridor from start→end, compute metrics, register it."
    # Don't rebuild if we already have this route
    existing = self._find_existing_corridor(start, end)
    if existing:
        print(f"build_corridor: reusing ⬡{start}→⬡{end}")
        return existing

    p = self.profile
    pcr = _pawn_covered_route(
        start, end,
        grid=self.grid, elevations=self.elevations,
        food_tiers=self.food_tiers,
        pawn_harvest=p.get('pawn_harvest', 1.3),
        pawn_diet=p.get('pawn_diet', 0.7),
        queen_diet=p.get('queen_diet', 7.0),
        cone_range=p.get('pawn_sight', 4),
        n_bishops=n_bishops,
        bishop_capacity=p.get('bishop_capacity', 140.0),
        bishop_diet=p.get('bishop_diet', 2.5),
        countries=self._countries(),
    )
    if pcr is None:
        print(f"build_corridor: no route ⬡{start}→⬡{end}")
        return None

    segments = _find_stations_v3(pcr, n_pawns=n_pawns,
                                    n_bishops=n_bishops, profile=p)
    metrics = self._metrics_from_segments(segments, pcr=pcr)   # ← pass pcr
    corridor = Corridor(hexes=pcr['route'], kind=kind,
                    segments=segments, **metrics)
    self.corridors.append(corridor)
    print(f"Corridor ⬡{start}→⬡{end} ({kind.value}): "
            f"{corridor.length} hexes, "
            f"{corridor.n_stations} covered, {corridor.n_gaps} gaps, "
            f"fuel={corridor.fuel_needed:.1f} "
            f"{'✅' if corridor.feasible else '❌'}")
    return corridor

In [ ]:
@patch
def __ft__(self: QueenCoverage):
    

    c_heads = ['route','type','length','stations','gaps',
               'fuel_needed','bishop_min_fuel','duration','congestion','ok']
    a_heads = ['queen','target','purpose','turns','corridor','ok']

    corridor_table = (
        Table(
            Thead(Tr(*[Th(h) for h in c_heads])),
            Tbody(*[c.__ft__() for c in self.corridors]),
            cls=TableT.hover,
        ) if self.corridors else P('No corridors yet.')
    )

    assign_table = (
        Table(
            Thead(Tr(*[Th(h) for h in a_heads])),
            Tbody(*[a.__ft__() for a in self.assignments]),
            cls=TableT.hover,
        ) if self.assignments else P('No assignments yet.')
    )

    return Div(Style("table { border-collapse: collapse; width: 100%; } "
      "th, td { padding: 4px 10px; text-align: left; border-bottom: 1px solid #ddd; } "
      "th { background: #f2f2f2; font-weight: bold; }"),
        Card(corridor_table,
             header=CardTitle(f'Corridors ({len(self.corridors)})')),
        Card(assign_table,
             header=CardTitle(f'Assignments ({len(self.assignments)})')),
    )

In [ ]:
# ── Recompute food tiers ──
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()

# ── Build coverage object ──
coverage = QueenCoverage(
    board=myStuff.board,
    grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=fy.tiers,
    profile=FOOD_PROFILE_V2,
    queens=[myStuff.queen],
)

# ── Register the opposing settlement as a target ──
coverage.add_target(
    hex=cities[1].location,
    purpose='attack',
    priority=1.5,
    deadline=20,
    payoff=10.0,
)

# ── Build a permanent corridor between the two settlements ──
c = coverage.build_corridor(
    cities[0].location,
    cities[1].location,
    kind=CorridorType.PERMANENT,
    n_pawns=4,
    n_bishops=1,
)

# ── Run the assignment MIP ──
coverage.plan()

# ── Update congestion count ──
coverage.update_congestion()

# ── Display debug table ──
show(coverage)


We found a corridor

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

🎉 It works! And the output is already telling a real story:

- **19 hex route** — 9 pawn-covered, 10 bishop-dependent
- **Betty needs ~30 turns** to arrive — 11 hex gap at the start of the corridor explains why (she has to cross unsupported before reaching the T7 pawn at ⬡248)
- **Bishop has 140 fuel, needs 63** — comfortable margin, `duration=2t` means that bishop can service the corridor twice before needing resupply
- **Congestion=2** — two pieces already on the route (from the earlier squad placement)

The interesting design tension this reveals: Betty starts *inside* the gap, not at the pawn station. So the first order of business for `shift()` will be getting a bishop to her starting position before she marches.

That's a natural next step — want to tackle the `shift()` stub, or first add a second target (say the fertile valley) to test the multi-queen assignment MIP?

In [ ]:
!cat ../../../HexMagic/game/mechanics/simulator.py

---

## `HexMagic.game.mechanics.schedule` — Design

### Purpose
A common **timetable** representation for piece movements across the board. Both the planner (budget) and the simulator (actual) produce Schedules in the same format, enabling variance analysis to detect when and where plans go off-track.

---

### Core Concept: Budget vs Actual

```
Planner (QueenCoverage.plan())  ──→  Schedule (budget)
                                                        ──→  VarianceReport
Simulator (FoodSimulator.run()) ──→  Schedule (actual)
```

- **Budget** — prescriptive: where pieces *should* be, with expected food/facing
- **Actual** — descriptive: where pieces *are*, pulled from simulator snapshots
- **Variance** — diagnostic: the diff, flagging delays, food shortfalls, and facing mismatches

---

### Schedule DataFrame Schema

One row per **(piece, turn)**. Shared by both budget and actual.

| column | type | description |
|---|---|---|
| `schedule_id` | str | links to the plan/sim that created this |
| `piece_id` | str | piece UUID |
| `piece_name` | str | human-readable name |
| `turn` | int | game turn |
| `hex` | int | hex index |
| `facing` | int | 0–5 |
| `action` | str | primary action that turn (`FORWARD`, `HARVEST`, `GIVE`, `ROT_L`, `ROT_R`, `PAUSE`, `—`) |
| `food` | float | food level at end of turn |
| `health` | float | health at end of turn |
| `segment_id` | int \| None | which corridor segment this serves (station/caravan index) |
| `corridor_id` | int \| None | which corridor, if any |
| `note` | str | free text (`arrived`, `blocked`, `starving`, etc.) |

Stored as a `pd.DataFrame`. Both adapters write to this exact schema.

---

### Adapters

**`PlanAdapter`** — budget schedule from a coverage plan
```python
@dataclass class PlanAdapter:
    coverage: QueenCoverage
```

- `build(assignment: Assignment) → pd.DataFrame`
  - Walks the corridor route hex by hex
  - Estimates facing changes (inserts `ROT_*` turns where needed)
  - Projects food consumption along the route using profile values
  - Tags each row with `segment_id` / `corridor_id`
  - Returns a Schedule DataFrame for that queen's planned march

- `build_all() → pd.DataFrame`
  - Calls `build()` per assignment, concatenates into one DataFrame

**`SimAdapter`** — actual schedule from simulator output
```python
@dataclass class SimAdapter:
    sim: FoodSimulator
    corridors: list[Corridor]   # for segment/corridor tagging
```

- `extract() → pd.DataFrame`
  - Reads `sim.snapshots` for per-turn piece state (hex, facing, food, health)
  - Reads `sim.df` for the primary action each turn
  - Reverse-maps each piece's location to corridor/segment (if on a known corridor)
  - Returns a Schedule DataFrame

---

### VarianceReport

```python
@dataclass class VarianceReport:
    budget: pd.DataFrame   # Schedule
    actual: pd.DataFrame   # Schedule
```

**Key computed columns (from join on `piece_id, turn`):**

| column | meaning |
|---|---|
| `hex_match` | bool — actual hex == budget hex |
| `hex_delta` | int — hex distance between actual and budget position |
| `facing_match` | bool — actual facing == budget facing |
| `food_delta` | float — actual food − budget food (negative = hungrier than expected) |
| `turns_behind` | int — how many turns late vs budget (0 = on schedule) |
| `status` | `on_track` \| `delayed` \| `off_route` \| `starving` \| `dead` |

**Key methods:**

- `compute() → pd.DataFrame` — the joined variance table
- `critical_path()` — which piece is the bottleneck (furthest behind schedule)
- `__ft__()` — two views:
  1. **Timetable grid** — turns across top, pieces down side, cells show hex with colour coding (green = on track, yellow = delayed, red = off route)
  2. **Summary card** — per-piece: planned arrival vs projected arrival, food margin

---

### `__ft__` Views

**Timetable (budget or actual):**

| | t0 | t1 | t2 | t3 | t4 | ... |
|---|---|---|---|---|---|---|
| Betty (Q) | ⬡158 | ⬡157 | ⬡156 | ⬡155 | ⬡154 | ... |
| Mary (P) | ⬡248 | ⬡248 | ⬡248 | ⬡248 | ⬡248 | ... |
| Bishop | ⬡190 | ⬡190 | ⬡170 | ⬡170 | ⬡151 | ... |

**Variance (diff):**

Same grid, but cells show:
- 🟢 on track
- 🟡 1–2 turns behind
- 🔴 3+ turns behind or off-route
- 💀 dead (not in budget)

**Summary card:**

| piece | planned arrival | projected arrival | delay | food margin | status |
|---|---|---|---|---|---|
| Betty | t19 | t22 | +3 | -4.2 | 🟡 delayed |

---

### Key Design Decisions

1. **Schedule is a DataFrame, not a class** — keeps it query-friendly and avoids another object hierarchy
2. **Adapters are thin** — they just translate existing data into the shared schema; no new simulation logic
3. **Corridor/segment tagging is reverse-mapped** — neither the planner nor the simulator "knows" about schedules; the adapters do the mapping
4. **Variance is a join, not a diff** — inner join on `(piece_id, turn)` gives you aligned rows; missing rows in actual = piece not yet moved
5. **`turns_behind` is the key metric** — computed by finding the budget turn where the piece was at the actual hex, vs the current turn

---

### Dependencies

- **Schedule** depends on: `pd.DataFrame`, corridor/segment IDs from `coverage`
- **PlanAdapter** depends on: `QueenCoverage`, `Assignment`, `Corridor`
- **SimAdapter** depends on: `FoodSimulator` (with `facing` added to snapshots), corridor list
- **VarianceReport** depends on: two Schedule DataFrames

---

### Open Questions

1. Should the timetable `__ft__` truncate to show only every Nth turn if the march is long (e.g. 30+ turns)? We are probably going to need to have to audit this if it gets chatty. We first need tools to see if what we are buidling is ok
2. The planAdapter needs to coordinate all pieces. if a bishop needs to be put in place or returned to a feeding area then it needs to think about it.
3. Should variance trigger **alerts** (e.g. "bishop will run out of fuel in 3 turns at current pace")? There is a question of how often we need to replan. (The proveriable hit in the face). we want to start with examples that stay on plan. Later we will have hooks into the food simular to say we are fine or we need to replan.

In [ ]:
# ── Schedule schema + helpers ─────────────────────────────────────────────

SCHEDULE_COLUMNS = [
    'schedule_id', 'piece_id', 'piece_name', 'turn', 'hex', 'facing',
    'action', 'food', 'health', 'segment_id', 'corridor_id', 'note',
]

def _exact_facing(grid, from_hex, to_hex):
    "Exact facing from from_hex to adjacent to_hex, or None."
    dirs = HexPosition.directions()
    for d in range(6):
        if grid.hexposition_to_index(dirs[d], from_hex) == to_hex:
            return d
    return None

def _route_w_to_seg(corridor, w):
    "Map route index w → segment index, or None."
    for i, seg in enumerate(corridor.segments):
        if seg['type'] == 'station' and seg['min_w'] <= w <= seg['max_w']:
            return i
        if seg['type'] == 'caravan' and seg.get('gap_start', -1) <= w <= seg.get('gap_end', -1):
            return i
    return None

def _hex_to_route_w(corridor, hex_idx):
    "Map hex index → route position, or None."
    for w, h in enumerate(corridor.hexes):
        if h == hex_idx: return w
    return None

def empty_schedule(schedule_id='empty'):
    return pd.DataFrame(columns=SCHEDULE_COLUMNS)

def schedule_timetable(df, title='Schedule', max_cols=25):
    "Render a Schedule DataFrame as a pieces×turns timetable."
    if df.empty: return Card(P('Empty schedule'), header=CardTitle(title))

    pieces = df.piece_name.unique()
    turns  = sorted(df.turn.unique())
    if len(turns) > max_cols:
        step = max(1, len(turns) // max_cols)
        turns = turns[::step]

    header = Tr(Th('piece'), *[Th(f't{int(t)}') for t in turns])
    rows = []
    for pname in pieces:
        pdata = df[df.piece_name == pname].set_index('turn')
        cells = [Th(pname)]
        for t in turns:
            if t in pdata.index:
                r = pdata.loc[t]
                if isinstance(r, pd.DataFrame): r = r.iloc[0]
                h = int(r['hex']); f = int(r['facing']); fd = r['food']
                tip = f"f={f} food={fd:.1f} {r.get('action','—')}"
                cells.append(Td(f'⬡{h}', title=tip, style='font-size:0.8em'))
            else:
                cells.append(Td('—'))
        rows.append(Tr(*cells))

    return Card(
        Table(Thead(header), Tbody(*rows), cls=TableT.sm),
        header=CardTitle(title),
    )


## Adapters

In [ ]:
@dataclass
class PlanAdapter:
    """Builds budget schedules from corridor routes.
    
    Decoupled from QueenCoverage — takes only grid + profile.
    Assumes: 1 hex/turn, free rotations, flat terrain cost.
    """
    grid:    object   # HexGrid
    profile: dict     # FOOD_PROFILE_V2
    elevations: list = None   # ← add this
    food_tiers: list = None    # ← add
    countries:  list = None    # ← add

    def build_all(self, assignments, schedule_id='budget'):
        "Build budget for every assignment, concatenated."
        dfs = [self.build(a.queen, a.corridor,
                          schedule_id=f"{schedule_id}_{a.queen.name}")
               for a in assignments]
        return pd.concat(dfs, ignore_index=True) if dfs else empty_schedule()


In [ ]:
@patch
def build3(self: PlanAdapter, piece, corridor, schedule_id='budget', start_turn=0):
    "Project piece position/food along corridor, using terrain move costs."
    route       = corridor.hexes
    corridor_id = f"{corridor.start}→{corridor.end}"
    elevations  = getattr(self, 'elevations', None)
    
    facing = piece.facing
    food   = piece.food
    health = piece.health
    diet   = piece.diet
    speed  = getattr(piece, 'move_strength', 1)
    turn   = start_turn
    w      = 0
    rows   = []

    def _row(turn, w, facing, action, food, health, note=''):
        return dict(
            schedule_id=schedule_id, piece_id=piece.id,
            piece_name=piece.name, turn=turn,
            hex=route[w], facing=facing, action=action,
            food=round(food, 1), health=round(health, 1),
            segment_id=_route_w_to_seg(corridor, w),
            corridor_id=corridor_id, note=note)

    rows.append(_row(turn, 0, facing, '—', food, health, 'start'))

    while w < len(route) - 1:
        turn += 1
        budget = float(speed)

        while budget >= 1.0 and w < len(route) - 1:
            tf = _exact_facing(self.grid, route[w], route[w + 1])
            if tf is not None: facing = tf
            
            # Use actual terrain cost if elevations available
            if elevations is not None:
                cost = _move_cost(elevations, route[w], route[w + 1])
            else:
                cost = 1.0
            
            if cost > budget: break
            w += 1
            budget -= cost

        food -= diet
        if food < 0:
            health = max(0, health + food * STARVE_MULT)
            food = 0

        note = ''
        if w == len(route) - 1: note = '✅ arrived'
        if health <= 0: note = '💀 dead'

        rows.append(_row(turn, w, facing, 'FORWARD', food, health, note))
        if health <= 0: break

    return pd.DataFrame(rows, columns=SCHEDULE_COLUMNS)


In [ ]:
@patch
def build_simulated(self: PlanAdapter, pieces, corridor,
                    schedule_id='budget_sim', n_turns=40,
                    callbacks=None):
    """Run a throwaway sim to produce a realistic budget schedule.
    
    Pieces are cloned as PieceShadows so originals aren't mutated.
    Handles blocking, terrain costs, eating — everything the real sim does.
    """
    # 1. Clone pieces
    shadows = [PieceShadow.from_piece(p) for p in pieces]
    
    # 2. Assign instructions from corridor route
    for shadow in shadows:
        w = _hex_to_route_w(corridor, shadow.location)
        if w is None: w = 0
        sub_route = corridor.hexes[w:]
        if len(sub_route) > 1:
            rules, _ = route_to_instructions(sub_route, self.grid, shadow.facing)
            shadow.instructions = InstructionList(rules, cursor=0, patrol=False)
    
    # 3. Capture initial states
    initial = {s.id: dict(location=s.location, facing=s.facing,
                          food=s.food, health=s.health)
               for s in shadows}
    
    # 4. Run throwaway sim
    sim = FoodSimulator(
        grid=self.grid,
        elevations=self.elevations,
        food_tiers=self.food_tiers,
        pieces=shadows,
        countries=self.countries,
    )
    sim.run(num_turns=n_turns)
    
    # 5. Extract schedule via SimAdapter
    adapter = SimAdapter(sim, corridors=[corridor])
    return adapter.extract(schedule_id=schedule_id,
                           initial_states=initial,
                           piece_ids={s.id for s in shadows})


In [ ]:
_EVENT_TO_ACTION = {
    'moved': 'FORWARD', 'bumped': 'BUMPED', 'harvested': 'HARVEST',
    'gave': 'GIVE', 'rotated': 'ROT', 'blocked': 'BLOCKED',
    'attacked': 'ATTACK', 'countered': 'COUNTER',
}
_ACTION_RANK = {
    'FORWARD': 0, 'BUMPED': 1, 'ATTACK': 2, 'COUNTER': 3,
    'HARVEST': 4, 'GIVE': 5, 'BLOCKED': 6, 'ROT': 7,
}

@dataclass
class SimAdapter:
    """Extracts actual schedules from FoodSimulator output.
    
    Decoupled from QueenCoverage — takes sim + corridor list.
    Sim snapshots (post-step state) are offset +1 so turn 0 = pre-sim.
    """
    sim:       FoodSimulator
    corridors: list              # list[Corridor]
    _names:    dict = field(default_factory=dict)

    def __post_init__(self):
        self._names = {p.id: p.name for p in self.sim.pieces}

    def _primary_action(self, piece_id, sim_turn):
        "Most significant voluntary action for (piece, sim_turn)."
        df = self.sim.df
        evts = df.loc[(df.piece_id == piece_id) & (df.turn == sim_turn), 'event']
        best, best_r = '—', 999
        for ev in evts:
            act = _EVENT_TO_ACTION.get(ev)
            if act and _ACTION_RANK.get(act, 99) < best_r:
                best, best_r = act, _ACTION_RANK[act]
        return best

    def _map_loc(self, hex_idx):
        for cor in self.corridors:
            w = _hex_to_route_w(cor, hex_idx)
            if w is not None:
                return f"{cor.start}→{cor.end}", _route_w_to_seg(cor, w)
        return None, None

    def _has_event(self, pid, sim_turn, ename):
        df = self.sim.df
        return ((df.piece_id == pid) & (df.turn == sim_turn) & (df.event == ename)).any()

    def extract(self, schedule_id='actual', initial_states=None, piece_ids=None):
        """Build Schedule DataFrame.
        
        Args:
            initial_states: dict {piece_id: {location, facing, food, health}}
                            for turn-0 row (pre-simulation).
            piece_ids:      subset of pieces to include (default: all).
        """
        ids  = piece_ids or set(self._names)
        rows = []

        # ── Turn 0 from initial_states ──
        if initial_states:
            for pid, st in initial_states.items():
                if pid not in ids: continue
                cid, sid = self._map_loc(st['location'])
                rows.append(dict(
                    schedule_id=schedule_id, piece_id=pid,
                    piece_name=self._names.get(pid, pid[:8]),
                    turn=0, hex=st['location'], facing=st['facing'],
                    action='—', food=round(st['food'], 1),
                    health=round(st.get('health', 100), 1),
                    segment_id=sid, corridor_id=cid, note='start'))

        # ── Snapshots → turn 1, 2, ... ──
        for snap in self.sim.snapshots:
            t = snap['turn'] + 1          # offset so turn 0 = pre-sim
            for pid, state in snap['piece_states'].items():
                if pid not in ids: continue
                act = self._primary_action(pid, snap['turn'])
                cid, sid = self._map_loc(state['location'])
                note = ''
                if self._has_event(pid, snap['turn'], 'died'):    note = '💀 dead'
                elif self._has_event(pid, snap['turn'], 'starved'): note = 'starving'
                rows.append(dict(
                    schedule_id=schedule_id, piece_id=pid,
                    piece_name=self._names.get(pid, pid[:8]),
                    turn=t, hex=state['location'], facing=state['facing'],
                    action=act, food=round(state['food'], 1),
                    health=round(state['health'], 1),
                    segment_id=sid, corridor_id=cid, note=note))

        return pd.DataFrame(rows, columns=SCHEDULE_COLUMNS) if rows else empty_schedule()


In [ ]:
@dataclass
class VarianceReport:
    "Joins budget + actual schedules and flags divergence."
    budget: pd.DataFrame
    actual: pd.DataFrame

    def compute(self):
        v = pd.merge(self.budget, self.actual,
                     on=['piece_id', 'turn'], suffixes=('_bgt', '_act'),
                     how='outer')
        v['hex_match']    = v.hex_bgt == v.hex_act
        v['facing_match'] = v.facing_bgt == v.facing_act
        v['food_delta']   = v.food_act - v.food_bgt

        def _status(r):
            if pd.notna(r.get('note_act')) and '💀' in str(r.note_act): return '💀 dead'
            if pd.isna(r.get('hex_act')): return 'not_started'
            if pd.isna(r.get('hex_bgt')): return 'unplanned'
            if r.hex_match: return 'on_track'
            return 'off_route'

        v['status'] = v.apply(_status, axis=1)
        return v

    def summary(self):
        v  = self.compute()
        nm = v.piece_name_bgt.fillna(v.piece_name_act)
        out = []
        for pname in nm.unique():
            pv = v[nm == pname]
            on = (pv.status == 'on_track').sum()
            fd = pv.food_delta.mean()
            worst = 'on_track'
            for s in ['💀 dead', 'off_route', 'not_started']:
                if (pv.status == s).any(): worst = s; break
            out.append(dict(piece=pname, on_track=f'{on}/{len(pv)}',
                            avg_food_delta=f'{fd:+.1f}' if pd.notna(fd) else '—',
                            status=worst))
        return pd.DataFrame(out)

    def __ft__(self):
        v  = self.compute()
        nm = v.piece_name_bgt.fillna(v.piece_name_act)
        pieces = nm.unique()
        turns  = sorted(v.turn.unique())

        _col = {'on_track': '#d4edda', 'off_route': '#f8d7da',
                '💀 dead': '#555', 'unplanned': '#fff3cd',
                'not_started': '#e2e3e5'}

        header = Tr(Th('piece'), *[Th(f't{int(t)}') for t in turns])
        trows  = []
        for pname in pieces:
            pv = v[nm == pname].set_index('turn')
            cells = [Th(pname)]
            for t in turns:
                if t in pv.index:
                    r = pv.loc[t]
                    if isinstance(r, pd.DataFrame): r = r.iloc[0]
                    st  = r['status']
                    bg  = _col.get(st, '#fff')
                    ha  = r.get('hex_act', r.get('hex_bgt'))
                    hb  = r.get('hex_bgt')
                    hs  = f'⬡{int(ha)}' if pd.notna(ha) else '—'
                    fd  = r.get('food_delta', 0)
                    tip = (f"bgt=⬡{int(hb) if pd.notna(hb) else '?'} "
                           f"{st} food:{fd:+.1f}" if pd.notna(fd) else st)
                    cells.append(Td(hs, title=tip,
                                    style=f'background:{bg};font-size:0.8em'))
                else:
                    cells.append(Td('—', style='background:#f5f5f5'))
            trows.append(Tr(*cells))

        timetable = Table(Thead(header), Tbody(*trows), cls=TableT.sm)

        # Summary table
        s = self.summary()
        sh = list(s.columns)
        sr = [Tr(*[Td(str(row[c])) for c in sh]) for _, row in s.iterrows()]
        summ = Table(Thead(Tr(*[Th(h) for h in sh])), Tbody(*sr), cls=TableT.sm)

        return Div(
            Style("td[title]{cursor:help}"),
            Card(timetable, header=CardTitle('Variance Timetable')),
            Card(summ,      header=CardTitle('Summary')),
        )


In [ ]:
# ── Bishop solo run: budget vs actual ─────────────────────────────────────

# 1. Teleport bishop to corridor start
bishop = [p for p in myStuff.queenGuard.pieces
          if p.piece_type == PieceType.BISHOP][0]
bishop.location = c.hexes[0]   # ⬡158
bishop.facing   = 0
apply_food_profile([bishop])    # food=140, diet=3.5
print(f"Bishop: {bishop.name} at ⬡{bishop.location}, "
      f"food={bishop.food}, diet={bishop.diet}, "
      f"move_strength={bishop.move_strength}")

# 2. Capture initial state for turn-0 alignment
initial = {bishop.id: dict(
    location=bishop.location, facing=bishop.facing,
    food=bishop.food, health=bishop.health)}

# 3. Budget schedule
planner = PlanAdapter(grid=myStuff.grid, profile=FOOD_PROFILE_V2,
                      elevations=myStuff.terr.elevations)
budget  = planner.build3(bishop, c, schedule_id='bishop_budget')
print(f"\nBudget: {len(budget)} rows, arrives turn {budget.turn.max()}")
show(schedule_timetable(budget, title='Budget'))

# 4. Generate instructions + run simulator
rules, _ = route_to_instructions(c.hexes, myStuff.grid, bishop.facing)
bishop.instructions = InstructionList(rules, cursor=0, patrol=False)

sim_bishop = FoodSimulator(
    grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=fy.tiers,
    pieces=[bishop],
    countries=myStuff.terr.fields.get('country'),
)
sim_bishop.run(num_turns=25)
sim_bishop.summary()

# 5. Actual schedule
adapter = SimAdapter(sim_bishop, corridors=[c])
actual  = adapter.extract(schedule_id='bishop_actual',
                          initial_states=initial)
show(schedule_timetable(actual, title='Actual'))

# 6. Variance
var = VarianceReport(budget, actual)
show(var)

In [ ]:
# ── Bishop solo run: build_simulated budget vs actual ─────────────────────

# 1. Teleport bishop to corridor start
bishop = [p for p in myStuff.queenGuard.pieces
          if p.piece_type == PieceType.BISHOP][0]
bishop.location = c.hexes[0]   # ⬡158
bishop.facing   = 0
apply_food_profile([bishop])
print(f"Bishop: {bishop.name} at ⬡{bishop.location}, "
      f"food={bishop.food}, diet={bishop.diet}, "
      f"move_strength={bishop.move_strength}")

# 2. Capture initial state
initial = {bishop.id: dict(
    location=bishop.location, facing=bishop.facing,
    food=bishop.food, health=bishop.health)}

# 3. Budget via build_simulated (shadow sim, doesn't touch real bishop)
planner = PlanAdapter(
    grid=myStuff.grid,
    profile=FOOD_PROFILE_V2,
    elevations=myStuff.terr.elevations,
    food_tiers=fy.tiers,
    countries=myStuff.terr.fields.get('country'),
)
budget = planner.build_simulated([bishop], c,
                                  schedule_id='bishop_budget_sim',
                                  n_turns=25)
print(f"\nBudget (simulated): {len(budget)} rows")
show(schedule_timetable(budget, title='Budget (simulated)'))

# 4. Real run — generate instructions + run actual simulator
rules, _ = route_to_instructions(c.hexes, myStuff.grid, bishop.facing)
bishop.instructions = InstructionList(rules, cursor=0, patrol=False)

sim_bishop = FoodSimulator(
    grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=fy.tiers,
    pieces=[bishop],
    countries=myStuff.terr.fields.get('country'),
)
sim_bishop.run(num_turns=25)
sim_bishop.summary()

# 5. Actual schedule
adapter = SimAdapter(sim_bishop, corridors=[c])
actual  = adapter.extract(schedule_id='bishop_actual',
                          initial_states=initial)
show(schedule_timetable(actual, title='Actual'))

# 6. Variance
var = VarianceReport(budget, actual)
show(var)


In [ ]:
rebootWorld()

In [ ]:
# ── Leapfrog test: 4 pawns sprint the corridor ──────────────────────────

# 1. Get 4 pawns, stagger along corridor start
pawns_4 = [p for p in myStuff.queenGuard.pieces
           if p.piece_type == PieceType.PAWN][:4]

for i, pawn in enumerate(pawns_4):
    pawn.location = c.hexes[i]
    apply_food_profile([pawn])
    if i < len(c.hexes) - 1:
        f = _exact_facing(myStuff.grid, c.hexes[i], c.hexes[i + 1])
        pawn.facing = f if f is not None else 0

for p in pawns_4:
    print(f"  {p.name}: ⬡{p.location} facing={p.facing} "
          f"food={p.food:.1f} diet={p.diet} speed={p.move_strength}")

# 2. Capture initial states
initial_4 = {p.id: dict(location=p.location, facing=p.facing,
                         food=p.food, health=p.health)
             for p in pawns_4}


In [ ]:
# 3. Budget via build_simulated (all 4 pawns together, handles blocking)
planner = PlanAdapter(
    grid=myStuff.grid,
    profile=FOOD_PROFILE_V2,
    elevations=myStuff.terr.elevations,
    food_tiers=fy.tiers,
    countries=myStuff.terr.fields.get('country'),
)
budget_all = planner.build_simulated(
    pawns_4, c,
    schedule_id='budget_pawns',
    n_turns=25,
)
print(f"Budget (simulated): {len(budget_all)} rows, {len(pawns_4)} pawns")
show(schedule_timetable(budget_all, title='Budget — 4 Pawns (simulated)'))


In [ ]:
# 4. Instructions + simulate
for pawn in pawns_4:
    w = _hex_to_route_w(c, pawn.location)
    sub_route = c.hexes[w:]
    rules, _ = route_to_instructions(sub_route, myStuff.grid, pawn.facing)
    pawn.instructions = InstructionList(rules, cursor=0, patrol=False)

sim_pawns = FoodSimulator(
    grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=fy.tiers,
    pieces=list(pawns_4),
    countries=myStuff.terr.fields.get('country'),
)
sim_pawns.run(num_turns=25)
sim_pawns.summary()


Can we build a test that shows the CrossingPlan connecting the two settlements?. we want to make sure we have a good demo

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

There are a couple of missing pieces before `propose_crossing` will run. Let me build them up.

**Cell 1 — `simple_hex_path`** (needed by `propose_crossing` for bishop pathfinding):

```python
def simple_hex_path(start, end, grid, elevations):
    """BFS shortest path on land hexes (ignores food, just walkability)."""
    if start == end: return [start]
    N = len(elevations)
    visited = {start}
    parent = {start: None}
    queue = deque([start])
    while queue:
        cur = queue.popleft()
        if cur == end:
            path = []
            while cur is not None:
                path.append(cur); cur = parent[cur]
            return path[::-1]
        for nb in grid.neighborsOf(cur):
            if nb < 0 or nb >= N or nb in visited: continue
            if nb in grid.invalidRegion or elevations[nb] <= 0: continue
            visited.add(nb); parent[nb] = cur; queue.append(nb)
    return None
```

**Cell 2 — the crossing plan demo**:

```python
# ── CrossingPlan: queen crosses from city 0 → city 1 ────────────────────

# 1. Fresh world + food
rebootWorld()
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()

cities = []
for country in myStuff.board.kingdoms:
    for city in country.settlements:
        cities.append(city)

c0, c1 = cities[0].location, cities[1].location
print(f"Cities: ⬡{c0} → ⬡{c1}\n")

# 2. Build coverage + corridor
qc = QueenCoverage(
    board=myStuff.board, grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=fy.tiers, profile=FOOD_PROFILE_V2,
    queens=[myStuff.queen],
)

corridor = qc.build_corridor(c0, c1, kind=CorridorType.PERMANENT,
                              n_pawns=5, n_bishops=2)
print(f"\nCorridor: {corridor.length} hexes, "
      f"{corridor.n_stations} stations, {corridor.n_gaps} gaps, "
      f"feasible={'✅' if corridor.feasible else '❌'}")

if not corridor.feasible:
    print("❌ Corridor not feasible — can't test crossing")
else:
    # 3. Teleport queen to corridor start, fill her up
    queen = myStuff.queen
    queen.location = corridor.hexes[0]
    apply_food_profile([queen])
    print(f"\nQueen {queen.name} at ⬡{queen.location}, "
          f"food={queen.food:.1f}, diet={queen.diet}")

    # 4. Propose crossing
    print(f"\n{'='*60}")
    print("Proposing crossing plan...")
    plan = qc.propose_crossing(
        corridor, queen,
        available_pawns=5,
        available_bishops=2,
        n_turns=60,
        max_retries=3,
    )

    # 5. Results
    print(f"\n{'='*60}")
    print(f"Feasible:     {plan.feasible}")
    print(f"Pieces:       {dict(plan.pieces_needed())}")
    print(f"Queen starts: turn {plan.queen_start_turn}")
    print(f"Arrival:      turn {plan.queen_arrival}")
    print(f"Food margin:  {plan.food_margin:.1f}")
    print(f"Bishop fuel:  {plan.bishop_fuel_used:.0f}")

    # 6. Show the plan card + timetable
    show(plan)
```

The key things this test exercises:

- **Corridor building** — PCR + station segmentation between the two settlements
- **Lighthouse placement** — bishops positioned to cover the 10-hex gap at the start
- **Shadow simulation** — throwaway sim with `PieceShadow` clones so originals aren't mutated
- **Retry logic** — if the queen dies on attempt 0 (bishops not ready in time), it adds wait turns / bishops
- **Schedule extraction** — the `SimAdapter` pulls the timetable from the shadow sim
- **`CrossingPlan.__ft__`** — renders the info card + schedule timetable

The most likely failure mode: the queen starves in the gap before bishops can `GIVE` to her. If that happens, you'll see the retry attempts in the output, and the plan may come back with `feasible=False` after 4 tries. If that happens we'd need to tweak either the bishop sprint timing or add a "bishop escorts queen" mode where they walk together instead of parking at lighthouses.

In [ ]:
def simple_hex_path(start, end, grid, elevations):
    """BFS shortest path on land hexes (ignores food, just walkability)."""
    if start == end: return [start]
    N = len(elevations)
    visited = {start}
    parent = {start: None}
    queue = deque([start])
    while queue:
        cur = queue.popleft()
        if cur == end:
            path = []
            while cur is not None:
                path.append(cur); cur = parent[cur]
            return path[::-1]
        for nb in grid.neighborsOf(cur):
            if nb < 0 or nb >= N or nb in visited: continue
            if nb in grid.invalidRegion or elevations[nb] <= 0: continue
            visited.add(nb); parent[nb] = cur; queue.append(nb)
    return None


In [ ]:
# ── CrossingPlan: queen crosses from city 0 → city 1 ────────────────────

# 1. Fresh world + food
rebootWorld()
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()

cities = []
for country in myStuff.board.kingdoms:
    for city in country.settlements:
        cities.append(city)

c0, c1 = cities[0].location, cities[1].location
print(f"Cities: ⬡{c0} → ⬡{c1}\n")

# 2. Build coverage + corridor
qc = QueenCoverage(
    board=myStuff.board, grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=fy.tiers, profile=FOOD_PROFILE_V2,
    queens=[myStuff.queen],
)

corridor = qc.build_corridor(c0, c1, kind=CorridorType.PERMANENT,
                              n_pawns=5, n_bishops=1)
print(f"\nCorridor: {corridor.length} hexes, "
      f"{corridor.n_stations} stations, {corridor.n_gaps} gaps, "
      f"feasible={'✅' if corridor.feasible else '❌'}")

if not corridor.feasible:
    print("❌ Corridor not feasible — can't test crossing")
else:
    # 3. Teleport queen to corridor start, fill her up
    queen = myStuff.queen
    queen.location = corridor.hexes[0]
    apply_food_profile([queen])
    print(f"\nQueen {queen.name} at ⬡{queen.location}, "
          f"food={queen.food:.1f}, diet={queen.diet}")

    # 4. Propose crossing
    print(f"\n{'='*60}")
    print("Proposing crossing plan...")
    plan = qc.propose_crossing(
        corridor, queen,
        available_pawns=5,
        available_bishops=1,
        n_turns=60,
        max_retries=3,
    )

    # 5. Results
    print(f"\n{'='*60}")
    print(f"Feasible:     {plan.feasible}")
    print(f"Pieces:       {dict(plan.pieces_needed())}")
    print(f"Queen starts: turn {plan.queen_start_turn}")
    print(f"Arrival:      turn {plan.queen_arrival}")
    print(f"Food margin:  {plan.food_margin:.1f}")
    print(f"Bishop fuel:  {plan.bishop_fuel_used:.0f}")

    # 6. Show the plan card + timetable
    show(plan)


In [ ]:
@patch
def build_corridor(self: QueenCoverage, start: int, end: int,
                   kind: CorridorType = CorridorType.TEMPORARY,
                   n_pawns: int = 4,
                   n_bishops: int = 1) -> Corridor | None:
    "Build a corridor from start→end, compute metrics, register it."
    existing = self._find_existing_corridor(start, end)
    if existing:
        print(f"build_corridor: reusing ⬡{start}→⬡{end}")
        return existing

    p = self.profile
    # Always constrain PCR to 1 bishop budget so it maximises pawn cover.
    # Full bishop count is used later at lighthouse placement.
    pcr = _pawn_covered_route(
        start, end,
        grid=self.grid, elevations=self.elevations,
        food_tiers=self.food_tiers,
        pawn_harvest=p.get('pawn_harvest', 1.3),
        pawn_diet=p.get('pawn_diet', 0.7),
        queen_diet=p.get('queen_diet', 7.0),
        cone_range=p.get('pawn_sight', 4),
        n_bishops=1,                                    # ← always 1
        bishop_capacity=p.get('bishop_capacity', 140.0),
        bishop_diet=p.get('bishop_diet', 2.5),
        countries=self._countries(),
    )
    if pcr is None:
        print(f"build_corridor: no route ⬡{start}→⬡{end}")
        return None

    segments = _find_stations_v3(pcr, n_pawns=n_pawns,
                                 n_bishops=1, profile=p)  # ← match
    metrics = self._metrics_from_segments(segments, pcr=pcr)

    # Now scale bishop_min_fuel by actual available bishops
    if n_bishops > 1 and metrics.get('fuel_needed', 0) > 0:
        metrics['bishop_min_fuel'] = p.get('bishop_capacity', 140.0) * n_bishops
        metrics['duration'] = int(metrics['bishop_min_fuel'] /
                                  max(metrics['fuel_needed'], 0.1))

    corridor = Corridor(hexes=pcr['route'], kind=kind,
                        segments=segments, **metrics)
    self.corridors.append(corridor)
    print(f"Corridor ⬡{start}→⬡{end} ({kind.value}): "
          f"{corridor.length} hexes, "
          f"{corridor.n_stations} covered, {corridor.n_gaps} gaps, "
          f"fuel={corridor.fuel_needed:.1f} "
          f"{'✅' if corridor.feasible else '❌'}")
    return corridor


In [ ]:
# ── CrossingPlan: queen crosses from city 0 → city 1 ────────────────────

# 1. Fresh world + food
rebootWorld()
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()

cities = []
for country in myStuff.board.kingdoms:
    for city in country.settlements:
        cities.append(city)

c0, c1 = cities[0].location, cities[1].location
print(f"Cities: ⬡{c0} → ⬡{c1}\n")

# 2. Build coverage + corridor
qc = QueenCoverage(
    board=myStuff.board, grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=fy.tiers, profile=FOOD_PROFILE_V2,
    queens=[myStuff.queen],
)

corridor = qc.build_corridor(c0, c1, kind=CorridorType.PERMANENT,
                              n_pawns=5, n_bishops=2)
print(f"\nCorridor: {corridor.length} hexes, "
      f"{corridor.n_stations} stations, {corridor.n_gaps} gaps, "
      f"feasible={'✅' if corridor.feasible else '❌'}")

if not corridor.feasible:
    print("❌ Corridor not feasible — can't test crossing")
else:
    # 3. Teleport queen to corridor start, fill her up
    queen = myStuff.queen
    queen.location = corridor.hexes[0]
    apply_food_profile([queen])
    print(f"\nQueen {queen.name} at ⬡{queen.location}, "
          f"food={queen.food:.1f}, diet={queen.diet}")

    # 4. Propose crossing
    print(f"\n{'='*60}")
    print("Proposing crossing plan...")
    plan = qc.propose_crossing(
        corridor, queen,
        available_pawns=5,
        available_bishops=2,
        n_turns=60,
        max_retries=3,
    )

    # 5. Results
    print(f"\n{'='*60}")
    print(f"Feasible:     {plan.feasible}")
    print(f"Pieces:       {dict(plan.pieces_needed())}")
    print(f"Queen starts: turn {plan.queen_start_turn}")
    print(f"Arrival:      turn {plan.queen_arrival}")
    print(f"Food margin:  {plan.food_margin:.1f}")
    print(f"Bishop fuel:  {plan.bishop_fuel_used:.0f}")

    # 6. Show the plan card + timetable
    show(plan)


Does it look like this plan will work?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

It technically passes — Betty arrives at ⬡345 by turn 14 — but there are some serious warning signs in the timetable:

**🟡 Betty starves mid-crossing.** Look at her food: t0=24 → t2=7 → **t4=0.0** → t6=17. She hits zero food before the bishops can feed her. With `STARVE_MULT=5` and `diet=7`, that's up to **35 HP damage per starving turn**. The plan says `min_food=0.0` which confirms this. She survives, but barely.

**🔴 Both bishops are fully consumed.** `bishop_fuel=280` = 2×140, meaning both bishops drain to zero. You can see it: Bishop_0 hits 0 by ~t30, Bishop_1 by ~t38. These are one-shot resources, not sustainable infrastructure.

**🔴 Betty starves at the destination.** After arriving at ⬡345 with food=17, she burns 7/turn with no food source. By t24 she's at 0.4, then zero forever. The crossing plan doesn't account for arrival logistics.

**🟡 Only 1 pawn for 19 hexes.** The PCR only found 1 viable pawn station (⬡248 T7). That single pawn does great work (food stays at 13.3, giving to Betty when she's in range around t10-14), but it means 10/19 hexes depend entirely on bishop reserves.

The core issue: **the queen starts inside the gap and waits 8 turns burning 7 food/turn before moving**. That's 56 food consumed while stationary — way more than her 24 capacity. The bishop at ⬡175 saves her, but just barely.

To make this robust, I'd suggest:
1. **Start the queen at the first station** instead of the gap start — or have a bishop escort her through the gap
2. **Pre-position a pawn or bishop at the destination** so Betty doesn't starve on arrival
3. **Raise the `min_food` threshold** in the feasibility check — `0.0` shouldn't count as ✅

if we set up optimal camps of pawns at both end for the queen could this work?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes, absolutely. Let me walk through why:

**Origin camp** — Betty currently burns 7 food/turn waiting 8 turns = 56 food eaten before she even moves. Her capacity is only 24. But if 2 pawns are on T7 hexes nearby (which exist — ⬡148, ⬡169 are T7), each exports 8.4/turn. Two pawns = 16.8 export, more than her 7.0 diet. So she'd **depart at full food** instead of starving.

**Destination camp** — Same logic in reverse. Right now Betty arrives at ⬡345 with 17 food and zero income, so she starves in ~2.5 turns. If 2 pawns are pre-positioned at good tiles near ⬡345, she's sustained indefinitely on arrival.

**The corridor itself is fine.** The mid-section (route[9..17]) is covered by the T7 pawn at ⬡248, and Bishop_1 at ⬡175 covers the initial gap. The issue was never the corridor — it was the **endpoints**.

The change to `propose_crossing` would be:

1. Before building the queen's instructions, run `optimal_camp_layout` at `route[0]` and `route[-1]`
2. Add those camp pawns to the shadow roster (HARVEST + GIVE on patrol, facing the route)
3. The queen **waits at the origin camp** until bishops are parked, departing at full food
4. She **arrives into the destination camp**, immediately fed

The pawn budget goes from 1 → maybe 5 (2 origin + 1 mid-station + 2 destination), but you have 5 available. And the bishops carry far less load since Betty isn't running on empty through the gap — she enters it at 24 food and the bishop at ⬡175 just needs to top her off, not rescue her from zero.

Want me to write the updated `propose_crossing` with origin/destination camp support?

In [ ]:
# ── CrossingPlan: queen crosses from city 0 → city 1 ────────────────────

# 1. Fresh world + food
rebootWorld()
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()

cities = []
for country in myStuff.board.kingdoms:
    for city in country.settlements:
        cities.append(city)

c0, c1 = cities[0].location, cities[1].location
print(f"Cities: ⬡{c0} → ⬡{c1}\n")

# 2. Build coverage + corridor
qc = QueenCoverage(
    board=myStuff.board, grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=fy.tiers, profile=FOOD_PROFILE_V2,
    queens=[myStuff.queen],
)

corridor = qc.build_corridor(c0, c1, kind=CorridorType.PERMANENT,
                              n_pawns=5, n_bishops=2)
print(f"\nCorridor: {corridor.length} hexes, "
      f"{corridor.n_stations} stations, {corridor.n_gaps} gaps, "
      f"feasible={'✅' if corridor.feasible else '❌'}")

if not corridor.feasible:
    print("❌ Corridor not feasible — can't test crossing")
else:
    # 3. Teleport queen to corridor start, fill her up
    queen = myStuff.queen
    queen.location = corridor.hexes[0]
    apply_food_profile([queen])
    print(f"\nQueen {queen.name} at ⬡{queen.location}, "
          f"food={queen.food:.1f}, diet={queen.diet}")

    # 4. Propose crossing
    print(f"\n{'='*60}")
    print("Proposing crossing plan...")
    plan = qc.propose_crossing(
        corridor, queen,
        available_pawns=5,
        available_bishops=2,
        n_turns=60,
        max_retries=3,
    )

    # 5. Results
    print(f"\n{'='*60}")
    print(f"Feasible:     {plan.feasible}")
    print(f"Pieces:       {dict(plan.pieces_needed())}")
    print(f"Queen starts: turn {plan.queen_start_turn}")
    print(f"Arrival:      turn {plan.queen_arrival}")
    print(f"Food margin:  {plan.food_margin:.1f}")
    print(f"Bishop fuel:  {plan.bishop_fuel_used:.0f}")

    # 6. Show the plan card + timetable
    show(plan)


can we try this with the food simulator

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Sure! Let's materialize the plan as real pieces and run the full simulator so we can see the charts and food trail.

```python
# ── Materialize the CrossingPlan and run the real FoodSimulator ──────────

route = plan.corridor.hexes
profile = FOOD_PROFILE_V2
grid = myStuff.grid
elevs = myStuff.terr.elevations

# 1. Recreate all pieces from the plan roster
pieces = []
queen_piece = None

for i, r in enumerate(plan.roster):
    if r['role'] == 'queen':
        # Use actual queen, teleported to start
        p = PieceShadow.from_piece(queen)
        p.location = route[0]
        p.food = profile['queen_capacity']
        # Wait + march instructions (same as plan)
        wait = [Instruction.PAUSE.value] * plan.queen_start_turn
        march, _ = route_to_instructions(route, grid, p.facing)
        p.instructions = InstructionList(wait + march, cursor=0, patrol=False)
        queen_piece = p

    elif r['role'] == 'origin_camp':
        p = PieceShadow(
            id=f'real_origin_{i}', name=f'OriginPawn_{i}',
            piece_type=PieceType.PAWN, birth_year=0,
            location=r['hex'], facing=r['facing'],
            food=profile['pawn_capacity'], food_capacity=profile['pawn_capacity'],
            diet=profile['pawn_diet'], health=100, max_health=100,
            move_strength=2, harvest_strength=profile['pawn_harvest'],
            sight=profile['pawn_sight'],
            owner_id=queen.owner_id, flag=queen.flag)
        p.instructions = InstructionList(
            [Instruction.HARVEST.value, Instruction.GIVE.value],
            cursor=0, patrol=True)

    elif r['role'] == 'advance_party':
        # Sprint to dest camp hex, then park
        dest_hex = r['hex']
        sprint_path = simple_hex_path(route[0], dest_hex, grid, elevs)
        if sprint_path:
            sprint_rules, end_f = route_to_instructions(sprint_path, grid, 0)
            want_f = _facing_toward(grid, dest_hex, route[-1])
            diff = (want_f - end_f) % 6
            rot = ([Instruction.ROT_R.value] * diff if diff <= 3
                   else [Instruction.ROT_L.value] * (6 - diff))
            park = [Instruction.HARVEST.value, Instruction.GIVE.value] * 80
            instr = sprint_rules + rot + park
        else:
            instr = [Instruction.HARVEST.value, Instruction.GIVE.value] * 80

        p = PieceShadow(
            id=f'real_advance_{i}', name=f'DestPawn_{i}',
            piece_type=PieceType.PAWN, birth_year=0,
            location=route[0], facing=0,
            food=profile['pawn_capacity'], food_capacity=profile['pawn_capacity'],
            diet=profile['pawn_diet'], health=100, max_health=100,
            move_strength=2, harvest_strength=profile['pawn_harvest'],
            sight=profile['pawn_sight'],
            owner_id=queen.owner_id, flag=queen.flag)
        p.instructions = InstructionList(instr, cursor=0, patrol=False)

    elif r['role'] == 'station':
        p = PieceShadow(
            id=f'real_station_{i}', name=f'StationPawn_{i}',
            piece_type=PieceType.PAWN, birth_year=0,
            location=r['hex'], facing=r['facing'],
            food=profile['pawn_capacity'], food_capacity=profile['pawn_capacity'],
            diet=profile['pawn_diet'], health=100, max_health=100,
            move_strength=2, harvest_strength=profile['pawn_harvest'],
            sight=profile['pawn_sight'],
            owner_id=queen.owner_id, flag=queen.flag)
        p.instructions = InstructionList(
            [Instruction.HARVEST.value, Instruction.GIVE.value],
            cursor=0, patrol=True)

    elif r['role'] == 'lighthouse':
        bishop_hex = r['hex']
        bp = simple_hex_path(route[0], bishop_hex, grid, elevs)
        if bp:
            sprint_rules, end_f = route_to_instructions(bp, grid, 0)
            want_f = r['facing']
            diff = (want_f - end_f) % 6
            rot = ([Instruction.ROT_R.value] * diff if diff <= 3
                   else [Instruction.ROT_L.value] * (6 - diff))
            park = [Instruction.HARVEST.value, Instruction.GIVE.value] * 60
            instr = sprint_rules + rot + park
        else:
            instr = [Instruction.HARVEST.value, Instruction.GIVE.value] * 60

        p = PieceShadow(
            id=f'real_bishop_{i}', name=f'Bishop_{i}',
            piece_type=PieceType.BISHOP, birth_year=0,
            location=route[0], facing=0,
            food=profile['bishop_capacity'], food_capacity=profile['bishop_capacity'],
            diet=profile.get('bishop_diet', 3.5), health=100, max_health=100,
            move_strength=3, harvest_strength=profile.get('bishop_harvest', 0.7),
            sight=profile['pawn_sight'],
            owner_id=queen.owner_id, flag=queen.flag)
        p.instructions = InstructionList(instr, cursor=0, patrol=False)
    else:
        continue

    pieces.append(p)

# 2. Ensure all pieces share owner
for p in pieces:
    p.owner_id = queen.owner_id
    p.flag = queen.flag

# 3. Wrap in squad
real_squad = Squad.squads(1, queen.flag)[0]
real_squad.pieces = list(pieces)

print(f"Roster: {len(pieces)} pieces")
for p in pieces:
    print(f"  {p.name:18s} {p.piece_type.name:6s} ⬡{p.location} "
          f"facing={p.facing} food={p.food:.1f}")

# 4. Run the actual sim
real_sim = FoodSimulator(
    grid=grid, elevations=elevs,
    food_tiers=fy.tiers, pieces=pieces,
    countries=myStuff.terr.fields.get('country'),
    squads=[real_squad],
    callbacks={real_squad.id: food_redistribute_adjacent},
)
real_sim.run(num_turns=60)
real_sim.summary()

# 5. Charts
real_sim.squad_chart(real_squad)
show(real_sim)
```

This recreates the exact plan — same piece positions, same instructions, same timing — but runs it through the full `FoodSimulator` with `food_redistribute_adjacent` enabled via the squad callback. You'll get:

- **`summary()`** — alive count, food/health averages
- **`squad_chart()`** — per-piece health & food lines over 60 turns
- **`show(sim)`** — the full dashboard with alive/health/food by rank + tactical events stacked bar

The key thing to watch: does Betty's food ever dip to 0 mid-crossing? The shadow sim said `min_food=3.0`, so the real sim should match. If it doesn't, we know the shadow sim diverged from reality (different blocking order, different piece move priority, etc.).

In [ ]:
# ── Materialize the CrossingPlan and run the real FoodSimulator ──────────

route = plan.corridor.hexes
profile = FOOD_PROFILE_V2
grid = myStuff.grid
elevs = myStuff.terr.elevations

# 1. Recreate all pieces from the plan roster
pieces = []
queen_piece = None

for i, r in enumerate(plan.roster):
    if r['role'] == 'queen':
        # Use actual queen, teleported to start
        p = PieceShadow.from_piece(queen)
        p.location = route[0]
        p.food = profile['queen_capacity']
        # Wait + march instructions (same as plan)
        wait = [Instruction.PAUSE.value] * plan.queen_start_turn
        march, _ = route_to_instructions(route, grid, p.facing)
        p.instructions = InstructionList(wait + march, cursor=0, patrol=False)
        queen_piece = p

    elif r['role'] == 'origin_camp':
        p = PieceShadow(
            id=f'real_origin_{i}', name=f'OriginPawn_{i}',
            piece_type=PieceType.PAWN, birth_year=0,
            location=r['hex'], facing=r['facing'],
            food=profile['pawn_capacity'], food_capacity=profile['pawn_capacity'],
            diet=profile['pawn_diet'], health=100, max_health=100,
            move_strength=2, harvest_strength=profile['pawn_harvest'],
            sight=profile['pawn_sight'],
            owner_id=queen.owner_id, flag=queen.flag)
        p.instructions = InstructionList(
            [Instruction.HARVEST.value, Instruction.GIVE.value],
            cursor=0, patrol=True)

    elif r['role'] == 'advance_party':
        # Sprint to dest camp hex, then park
        dest_hex = r['hex']
        sprint_path = simple_hex_path(route[0], dest_hex, grid, elevs)
        if sprint_path:
            sprint_rules, end_f = route_to_instructions(sprint_path, grid, 0)
            want_f = _facing_toward(grid, dest_hex, route[-1])
            diff = (want_f - end_f) % 6
            rot = ([Instruction.ROT_R.value] * diff if diff <= 3
                   else [Instruction.ROT_L.value] * (6 - diff))
            park = [Instruction.HARVEST.value, Instruction.GIVE.value] * 80
            instr = sprint_rules + rot + park
        else:
            instr = [Instruction.HARVEST.value, Instruction.GIVE.value] * 80

        p = PieceShadow(
            id=f'real_advance_{i}', name=f'DestPawn_{i}',
            piece_type=PieceType.PAWN, birth_year=0,
            location=route[0], facing=0,
            food=profile['pawn_capacity'], food_capacity=profile['pawn_capacity'],
            diet=profile['pawn_diet'], health=100, max_health=100,
            move_strength=2, harvest_strength=profile['pawn_harvest'],
            sight=profile['pawn_sight'],
            owner_id=queen.owner_id, flag=queen.flag)
        p.instructions = InstructionList(instr, cursor=0, patrol=False)

    elif r['role'] == 'station':
        p = PieceShadow(
            id=f'real_station_{i}', name=f'StationPawn_{i}',
            piece_type=PieceType.PAWN, birth_year=0,
            location=r['hex'], facing=r['facing'],
            food=profile['pawn_capacity'], food_capacity=profile['pawn_capacity'],
            diet=profile['pawn_diet'], health=100, max_health=100,
            move_strength=2, harvest_strength=profile['pawn_harvest'],
            sight=profile['pawn_sight'],
            owner_id=queen.owner_id, flag=queen.flag)
        p.instructions = InstructionList(
            [Instruction.HARVEST.value, Instruction.GIVE.value],
            cursor=0, patrol=True)

    elif r['role'] == 'lighthouse':
        bishop_hex = r['hex']
        bp = simple_hex_path(route[0], bishop_hex, grid, elevs)
        if bp:
            sprint_rules, end_f = route_to_instructions(bp, grid, 0)
            want_f = r['facing']
            diff = (want_f - end_f) % 6
            rot = ([Instruction.ROT_R.value] * diff if diff <= 3
                   else [Instruction.ROT_L.value] * (6 - diff))
            park = [Instruction.HARVEST.value, Instruction.GIVE.value] * 60
            instr = sprint_rules + rot + park
        else:
            instr = [Instruction.HARVEST.value, Instruction.GIVE.value] * 60

        p = PieceShadow(
            id=f'real_bishop_{i}', name=f'Bishop_{i}',
            piece_type=PieceType.BISHOP, birth_year=0,
            location=route[0], facing=0,
            food=profile['bishop_capacity'], food_capacity=profile['bishop_capacity'],
            diet=profile.get('bishop_diet', 3.5), health=100, max_health=100,
            move_strength=3, harvest_strength=profile.get('bishop_harvest', 0.7),
            sight=profile['pawn_sight'],
            owner_id=queen.owner_id, flag=queen.flag)
        p.instructions = InstructionList(instr, cursor=0, patrol=False)
    else:
        continue

    pieces.append(p)

# 2. Ensure all pieces share owner
for p in pieces:
    p.owner_id = queen.owner_id
    p.flag = queen.flag

# 3. Wrap in squad
real_squad = Squad.squads(1, queen.flag)[0]
real_squad.pieces = list(pieces)

print(f"Roster: {len(pieces)} pieces")
for p in pieces:
    print(f"  {p.name:18s} {p.piece_type.name:6s} ⬡{p.location} "
          f"facing={p.facing} food={p.food:.1f}")

# 4. Run the actual sim
real_sim = FoodSimulator(
    grid=grid, elevations=elevs,
    food_tiers=fy.tiers, pieces=pieces,
    countries=myStuff.terr.fields.get('country'),
    squads=[real_squad],
    callbacks={real_squad.id: food_redistribute_adjacent},
)
real_sim.run(num_turns=60)
real_sim.summary()

# 5. Charts
real_sim.squad_chart(real_squad)
show(real_sim)


You can see the bishop drain.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yep — Bishop_6 is the smoking gun. It sprinted to the lighthouse at ⬡388, served its purpose covering route[18] (the final hop to ⬡345), and then just... sat there bleeding out. 140 → ~0 over 60 turns, a straight diagonal line down. It's harvesting on what's probably a T1 hex, pulling maybe 0.7/turn against a 2.5 diet — net loss of ~1.8/turn, so 140/1.8 ≈ 78 turns to empty. The chart confirms it hits zero around turn 55-60.

Bishop_7, by contrast, is parked at ⬡175 on a T2 tile and basically flatlines at 136.5 the entire sim. It barely moved (4 hexes total), ate almost nothing in transit, and its harvest roughly covers its diet. That's the ideal lighthouse behavior.

The good news though: **the crossing itself is rock solid**. Betty arrives at ⬡345 by turn 16 with food ~17, health never dips, and the dest camp pawns keep her fed indefinitely at 17.0. The origin camp pawns stabilize too. 8/8 alive at turn 60.

The Bishop_6 drain is a **post-crossing cleanup problem**, not a crossing problem. Two options:

1. **Recall Bishop_6** after the queen arrives — give it return instructions to march back to a high-tier hex or the origin camp, where pawns can feed it
2. **Accept it as expendable** — it did its job, and by the time it starves Betty is long established at the destination

If you want the corridor to be permanent infrastructure (not just a one-shot crossing), option 1 is the way to go. Otherwise this plan works as-is.

THis is amazing. I can't tell you how many different ideas there was to get piece across the board. Can you document our conversation, because when I git commit these go away.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Here's a document capturing the journey:

````markdown
# Queen Crossing Plan — Development Journal

**Date:** 2026-03-12
**Dialog:** HexMagic crossing plan development (~93 messages)

## The Problem

Move queen Betty safely across 19 hexes of mixed terrain from city ⬡158 → city ⬡345,
through a corridor where 10/19 hexes are barren (no pawn-sustainable food). The queen
has diet=7.0, capacity=24 (only ~3.4 turns of reserves), and cannot self-sustain on
any tile.

## Key Constraints

- **FOOD_PROFILE_V2**: Queen diet=7, cap=30, harvest=0.5 (net@T5 = -5.8 ❌)
- **Pawns**: diet=2, harvest=2, cap=12, net@T5=+3.0 ✅ — 3 pawns needed to sustain queen
- **Bishops**: diet=2.5, harvest=1.0, cap=140 — mobile fuel tanks, break-even at best
- **Available**: 5 pawns, 2 bishops
- **Route**: 19 hexes, only 1 viable pawn station at ⬡248 (T7), 10 gap hexes

## Evolution of the Plan

### Attempt 1: Bare Corridor (no camps)
- PCR found 1 pawn station at ⬡248, covering route[9..17]
- Bishops assigned as mobile caravans escorting queen through gaps
- **Result**: Betty starved mid-crossing (food hit 0.0 at t4), survived but barely
- Both bishops fully consumed (280 fuel burned)
- Betty starved again at destination with no food source
- `min_food=0.0` — technically feasible but extremely fragile

### Key Insight: Endpoint Camps
The corridor itself was fine — the problem was always the **endpoints**:
- **Origin**: Queen burned 56 food waiting 8 turns (7/turn × 8) with only 24 capacity
- **Destination**: Arrived with food=17 and zero income, starving in ~2.5 turns

### Attempt 2: Origin + Destination Pawn Camps
Added `optimal_camp_layout` at both ends of the route:
- **2 origin camp pawns** (⬡179 T7, ⬡159 T6) — feed queen while she waits
- **2 advance party pawns** — sprint ahead to destination (⬡346 T7, ⬡342 T7)
- **1 mid-station pawn** at ⬡248 (T7) — same as before
- **2 lighthouse bishops** — MIP-optimized elevated positions for line-of-sight coverage

### Lighthouse Strategy
Instead of bishops escorting as caravans, they park at elevated hexes with
long sight lines covering the gap:
- **Bishop_0** → ⬡388 (elev=368): covers route[18] (1 hex), fuel=10.5/140
- **Bishop_1** → ⬡175 (elev=361): covers route[0..8] (9 hexes), fuel=94.5/140

This dramatically reduced bishop fuel usage from 280 → 144.

### Final Plan Card
```
Route:       ⬡158→⬡345 (19 hexes)
Roster:      5× PAWN, 2× BISHOP
Queen go:    turn 16
Arrival:     turn 16
Food min:    3.0 (never starves!)
Bishop fuel: 144/280
Status:      ✅
```

## Full Simulator Validation

Materialized plan as real `PieceShadow` objects in `FoodSimulator` with
`food_redistribute_adjacent` squad callback, 60 turns:

- **8/8 alive** at turn 60
- **Health**: avg=102, min=96 (no starvation damage to queen)
- **Betty's food**: never dips below ~3.0 during crossing, stabilizes at 17.0 at destination
- **Origin pawns**: stable at 10-13 food indefinitely
- **Dest pawns**: arrive, park, harvest — sustain Betty on arrival
- **Station pawn** (⬡248): rock solid at 13.3 entire sim
- **Bishop_1** (⬡175): barely moves, flatlines at 136.5 — ideal lighthouse
- **Bishop_0** (⬡388): slow drain from 140→0 over 60 turns (post-crossing bleed, not a crossing issue)

## Remaining Issue

Bishop_0 at ⬡388 drains to zero by ~turn 55-60. It's on a low-tier hex, harvesting
~0.7/turn against 2.5 diet (net -1.8/turn). This is a **post-crossing cleanup problem**,
not a crossing failure. Options:
1. Recall Bishop_0 after queen arrives (return instructions to high-tier hex or origin camp)
2. Accept as expendable — it did its job

## Architecture Used

- `FoodYield` — per-hex food tier computation from terrain
- `QueenCoverage` — corridor building + crossing proposals
- `CorridorType.PERMANENT` — reusable infrastructure corridor
- `PCR` (Pawn Coverage Router) — optimal pawn station placement along route
- `Lighthouse MIP` — integer program for bishop elevation/sight optimization
- `FoodSimulator` — full turn-by-turn simulation with squad food redistribution
- `PieceShadow` — lightweight piece clones for simulation
- `InstructionList` — patrol/march instruction sequences
- `food_redistribute_adjacent` — squad callback for adjacent-hex food sharing

## What We Learned

1. **Endpoints matter more than corridors** — the gap crossing was solvable with
   bishops, but origin/destination starvation was the real killer
2. **Pawns as camps, bishops as lighthouses** — separation of concerns works
3. **Shadow sim matches real sim** — `min_food=3.0` in proposal matched actual simulation
4. **Bishop placement is key** — Bishop_1 on a T2 elevated hex = self-sustaining;
   Bishop_0 on a T1 hex = slow death. Tile selection for lighthouses should factor
   in long-term sustainability, not just sight lines
5. **3 pawns sustain a queen at T5+** — this ratio held throughout all experiments
````

This captures the full arc from the fragile first attempt through the camp insight to the validated final plan. Save it alongside the code and you'll have the "why" behind every design decision. 🏔️👑

I hope it doesn't do isinstance that is a very bad design and I will have to fix.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

I scanned through the simulator code you shared — good news, it's fully duck-typed. It just reads attributes (`piece.health`, `piece.food`, `piece.location`, etc.) and calls methods. No `isinstance` anywhere.

The one method it *does* call on pieces is:

```python
piece.hexes_in_sight(self.grid, self.elevations,
                     elevation_mult=self.elevation_mult, facing_only=True)
```

So `PieceShadow` needs that method too. It's used in `_do_give` (finding nearby hungry allies) and `_combat_phase` (finding enemies in cone). Since the planner doesn't fight, combat doesn't matter — but `_do_give` does if pawns are feeding each other on the march.

You could either:
1. Copy the `hexes_in_sight` method onto `PieceShadow`
2. Import it as a standalone function and delegate: `def hexes_in_sight(self, *a, **kw): return _hexes_in_sight(self, *a, **kw)`

Either way, `PieceShadow` + `FoodSimulator` should just work as long as that method exists. Everything else is attribute access.

# Queen Crossing Plan — Development Journal

**Date:** 2026-03-12
**Dialog:** HexMagic crossing plan development (~93 messages)

## The Problem

Move queen Betty safely across 19 hexes of mixed terrain from city ⬡158 → city ⬡345,
through a corridor where 10/19 hexes are barren (no pawn-sustainable food). The queen
has diet=7.0, capacity=24 (only ~3.4 turns of reserves), and cannot self-sustain on
any tile.

## Key Constraints

- **FOOD_PROFILE_V2**: Queen diet=7, cap=30, harvest=0.5 (net@T5 = -5.8 ❌)
- **Pawns**: diet=2, harvest=2, cap=12, net@T5=+3.0 ✅ — 3 pawns needed to sustain queen
- **Bishops**: diet=2.5, harvest=1.0, cap=140 — mobile fuel tanks, break-even at best
- **Available**: 5 pawns, 2 bishops
- **Route**: 19 hexes, only 1 viable pawn station at ⬡248 (T7), 10 gap hexes

## Evolution of the Plan

### Attempt 1: Bare Corridor (no camps)
- PCR found 1 pawn station at ⬡248, covering route[9..17]
- Bishops assigned as mobile caravans escorting queen through gaps
- **Result**: Betty starved mid-crossing (food hit 0.0 at t4), survived but barely
- Both bishops fully consumed (280 fuel burned)
- Betty starved again at destination with no food source
- `min_food=0.0` — technically feasible but extremely fragile

### Key Insight: Endpoint Camps
The corridor itself was fine — the problem was always the **endpoints**:
- **Origin**: Queen burned 56 food waiting 8 turns (7/turn × 8) with only 24 capacity
- **Destination**: Arrived with food=17 and zero income, starving in ~2.5 turns

### Attempt 2: Origin + Destination Pawn Camps
Added `optimal_camp_layout` at both ends of the route:
- **2 origin camp pawns** (⬡179 T7, ⬡159 T6) — feed queen while she waits
- **2 advance party pawns** — sprint ahead to destination (⬡346 T7, ⬡342 T7)
- **1 mid-station pawn** at ⬡248 (T7) — same as before
- **2 lighthouse bishops** — MIP-optimized elevated positions for line-of-sight coverage

### Lighthouse Strategy
Instead of bishops escorting as caravans, they park at elevated hexes with
long sight lines covering the gap:
- **Bishop_0** → ⬡388 (elev=368): covers route[18] (1 hex), fuel=10.5/140
- **Bishop_1** → ⬡175 (elev=361): covers route[0..8] (9 hexes), fuel=94.5/140

This dramatically reduced bishop fuel usage from 280 → 144.

### Final Plan Card
```
Route:       ⬡158→⬡345 (19 hexes)
Roster:      5× PAWN, 2× BISHOP
Queen go:    turn 16
Arrival:     turn 16
Food min:    3.0 (never starves!)
Bishop fuel: 144/280
Status:      ✅
```

## Full Simulator Validation

Materialized plan as real `PieceShadow` objects in `FoodSimulator` with
`food_redistribute_adjacent` squad callback, 60 turns:

- **8/8 alive** at turn 60
- **Health**: avg=102, min=96 (no starvation damage to queen)
- **Betty's food**: never dips below ~3.0 during crossing, stabilizes at 17.0 at destination
- **Origin pawns**: stable at 10-13 food indefinitely
- **Dest pawns**: arrive, park, harvest — sustain Betty on arrival
- **Station pawn** (⬡248): rock solid at 13.3 entire sim
- **Bishop_1** (⬡175): barely moves, flatlines at 136.5 — ideal lighthouse
- **Bishop_0** (⬡388): slow drain from 140→0 over 60 turns (post-crossing bleed, not a crossing issue)

## Remaining Issue

Bishop_0 at ⬡388 drains to zero by ~turn 55-60. It's on a low-tier hex, harvesting
~0.7/turn against 2.5 diet (net -1.8/turn). This is a **post-crossing cleanup problem**,
not a crossing failure. Options:
1. Recall Bishop_0 after queen arrives (return instructions to high-tier hex or origin camp)
2. Accept as expendable — it did its job

## Architecture Used

- `FoodYield` — per-hex food tier computation from terrain
- `QueenCoverage` — corridor building + crossing proposals
- `CorridorType.PERMANENT` — reusable infrastructure corridor
- `PCR` (Pawn Coverage Router) — optimal pawn station placement along route
- `Lighthouse MIP` — integer program for bishop elevation/sight optimization
- `FoodSimulator` — full turn-by-turn simulation with squad food redistribution
- `PieceShadow` — lightweight piece clones for simulation
- `InstructionList` — patrol/march instruction sequences
- `food_redistribute_adjacent` — squad callback for adjacent-hex food sharing

## What We Learned

1. **Endpoints matter more than corridors** — the gap crossing was solvable with
   bishops, but origin/destination starvation was the real killer
2. **Pawns as camps, bishops as lighthouses** — separation of concerns works
3. **Shadow sim matches real sim** — `min_food=3.0` in proposal matched actual simulation
4. **Bishop placement is key** — Bishop_1 on a T2 elevated hex = self-sustaining;
   Bishop_0 on a T1 hex = slow death. Tile selection for lighthouses should factor
   in long-term sustainability, not just sight lines
5. **3 pawns sustain a queen at T5+** — this ratio held throughout all experiments


We need two things
1. How to generalize this for multiple peices and targets
2. an easy to use api


##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Great questions. Let me sketch both out, and we can refine together.

**Note:** The context window truncated earlier messages (96 → 8), so I'm working from the journal and the code visible here.

---

**1. Generalizing: the key abstractions**

Right now we have one queen, one corridor, one crossing. The natural generalization has three layers:

| Layer | Current | Generalized |
|-------|---------|-------------|
| **Infrastructure** | 1 corridor (⬡158→⬡345) | A `CorridorNetwork` — graph of reusable corridors between cities |
| **Logistics** | 5 pawns, 2 bishops hardcoded | A `ResourcePool` — available pieces, allocated/free tracking |
| **Mission** | 1 queen crossing | A `MissionQueue` — ordered list of crossings, convoys, or patrols |

The key insight from our work: **corridors are reusable infrastructure, crossings are one-shot events that consume corridor capacity**. So:

```python
# A corridor is permanent infrastructure (stations + lighthouses)
# A crossing is a mission that USES a corridor

# Multiple pieces can share a corridor if camps have capacity
# Multiple crossings can be queued sequentially on one corridor
# Multiple corridors can share bishop lighthouses (if sight lines overlap)
```

The hardest generalization is **resource contention** — if you have 8 pawns total and two crossings each want 5, you need to either sequence them or find a thinner plan. That's where the `ResourcePool` earns its keep.

---

**2. API sketch**

```python
# ── Layer 1: Coverage (stateless analysis) ────────────────────
coverage = QueenCoverage(board, grid, elevations, food_tiers,
                         profile=FOOD_PROFILE_V2, queens=my_queens)

# ── Layer 2: Network (build reusable infrastructure) ──────────
network = CorridorNetwork(coverage)

# Add corridors between cities — returns handles
c1 = network.add(city_a, city_b, kind=CorridorType.PERMANENT,
                  n_pawns=5, n_bishops=2)
c2 = network.add(city_b, city_c, kind=CorridorType.TEMPORARY,
                  n_pawns=3, n_bishops=1)

# Network can optimize shared lighthouses across corridors
network.optimize_lighthouses()

# ── Layer 3: Mission (plan a specific crossing) ───────────────
pool = ResourcePool(pawns=all_pawns, bishops=all_bishops)

# Plan one crossing — returns a CrossingPlan
plan = network.plan_crossing(
    piece=queen,          # who's crossing (any piece, not just queens)
    corridor=c1,          # which corridor to use
    pool=pool,            # available resources (marks used ones as allocated)
    n_turns=60,
)

# Plan a convoy — multiple pieces using same corridor
convoy_plan = network.plan_convoy(
    pieces=[queen, knight],
    corridor=c1,
    pool=pool,
    n_turns=60,
)

# ── Validate ──────────────────────────────────────────────────
sim = plan.simulate(grid, elevations, food_tiers)  # returns FoodSimulator
sim.summary()

# ── Execute ───────────────────────────────────────────────────
# Writes InstructionLists onto actual game pieces
plan.apply()              # teleports + sets instructions
plan.apply(dry_run=True)  # just prints what it would do
```

The key API principles:
- **No `isinstance`** — pieces are duck-typed, anything with `.location`, `.food`, `.diet`, `.hexes_in_sight()` works
- **Pool tracks allocation** — `pool.allocate(plan)` / `pool.release(plan)` so two plans can't double-book the same pawn
- **Plans are data** — `CrossingPlan` is inspectable, serializable, `show()`-able before you commit
- **`simulate()` on the plan itself** — one method to go from plan → validated sim with charts
- **`apply()` is the only mutation** — everything else is pure analysis

---

**3. What to build first?**

I'd suggest this order:

1. **`ResourcePool`** — thin wrapper, just tracks which pieces are free vs allocated. Unlocks multi-crossing immediately
2. **`plan_crossing` accepts any piece** — replace the queen-specific logic with profile lookups by piece type
3. **`CorridorNetwork`** — graph of corridors with shared lighthouse optimization
4. **`plan_convoy`** — multiple pieces, staggered timing, shared camps

Steps 1-2 are probably a few hours of refactoring. Steps 3-4 are the meaty generalization.

What feels right as a starting point?

in a different version I had a notion of tatics. the following code isn't functional but being able to describe certain kind of actitivies was intereesting

```python
from dataclasses import dataclass, field
from collections import deque

# ── Ranged food transfer (matches PCR cone_range model) ──
def food_redistribute_ranged(report: TurnReport, sight_range=4):
    """Like adjacent, but transfers food within sight_range (not just neighbors)."""
    alive = [p for p in report.squad.pieces if p.health > 0]
    if len(alive) < 2: return
    grid = report.grid
    
    by_need = sorted(alive, key=lambda p: p.food / max(p.diet, 0.1))
    for receiver in by_need:
        need = receiver.food_capacity - receiver.food
        if need < 0.1: continue
        for donor in alive:
            if donor is receiver: continue
            surplus = _donor_surplus(donor)
            if surplus < 0.1: continue
            hp_d = grid.index_to_hexposition(donor.location)
            hp_r = grid.index_to_hexposition(receiver.location)
            if hp_d.distance(hp_r) > sight_range: continue
            transfer = min(need, surplus)
            donor.food -= transfer
            receiver.food += transfer
            need -= transfer
            if need < 0.1: break

# ── Tactic infrastructure ──
@dataclass
class TacticContext:
    squad: Squad
    grid: object
    terrain: object
    fy: object
    profile: dict
    route: list = field(default_factory=list)
    
    @property
    def pawns(self): return [p for p in self.squad.alive if p.piece_type == PieceType.PAWN]
    @property
    def bishops(self): return [p for p in self.squad.alive if p.piece_type == PieceType.BISHOP]
    @property
    def queens(self): return [p for p in self.squad.alive if p.piece_type == PieceType.QUEEN]

@dataclass
class Assignment:
    piece_id: str = ''
    piece_type: PieceType = PieceType.PAWN
    target_hex: int = -1
    role: str = 'idle'

@dataclass
class TacticSpec:
    name: str = ''
    turn: int = 0
    est_duration: int = 0
    
    @property
    def end_turn(self): return self.turn + self.est_duration
    @property
    def requires(self): return {}
    def plan(self, ctx): return []
    def execute(self, turn, assignments, ctx): pass
    def __repr__(self): return f"{self.name}(t={self.turn}→{self.end_turn})"

@dataclass
class ExecContext:
    queen: object
    queen_w: int
    route: list
    grid: object
    elevations: object
    piece_map: dict
    initialized: dict
    log: list
    turn: int = 0
    
    def march_to(self, piece, target_hex):
        if piece.location == target_hex:
            self.set_harvest_give(piece); return
        path = self.bfs_path(piece.location, target_hex)
        if not path or len(path) < 2:
            self.set_harvest_give(piece); return
        rules, _ = route_to_instructions(path, self.grid, piece.facing)
        rules.extend([Instruction.HARVEST.value, Instruction.GIVE.value])
        piece.instructions = InstructionList(rules, cursor=0, patrol=False)
    
    def set_harvest_give(self, piece):
        piece.instructions = InstructionList(
            [Instruction.HARVEST.value, Instruction.GIVE.value], cursor=0, patrol=True)
    
    def move_queen(self, route_end):
        target_w = min(route_end, len(self.route) - 1)
        if target_w <= self.queen_w: return
        sub = self.route[self.queen_w:target_w + 1]
        if len(sub) > 1:
            rules, _ = route_to_instructions(sub, self.grid, self.queen.facing)
            self.queen.instructions = InstructionList(rules, cursor=0, patrol=False)
            self.log.append(f"t{self.turn}: Queen march route[{self.queen_w}→{target_w}]")
    
    def bfs_path(self, start, end):
        if start == end: return [start]
        parent = {start: None}
        q = deque([start])
        while q:
            cur = q.popleft()
            if cur == end: break
            for nb in self.grid.neighborsOf(cur):
                if nb in parent or nb < 0: continue
                if nb >= len(self.elevations) or self.elevations[nb] <= 0: continue
                parent[nb] = cur
                q.append(nb)
        if end not in parent: return None
        path = []; cur = end
        while cur is not None: path.append(cur); cur = parent[cur]
        path.reverse(); return path
    
    def first_call(self, piece_id, tactic):
        key = id(tactic)
        if self.initialized.get(piece_id) == key: return False
        self.initialized[piece_id] = key; return True

# ── Tactics ──
@dataclass
class CampTactic(TacticSpec):
    name: str = 'camp'
    center_hex: int = -1
    target_food_pct: float = 0.9
    _placements: dict = field(default_factory=dict)
    _done: bool = False
    
    @property
    def requires(self): return {'pawn': 2}
    
    def plan(self, ctx):
        queen = ctx.queens[0] if ctx.queens else None
        if not queen: return []
        pawns = ctx.pawns[:2]
        tmp = Squad.squads(1, ctx.squad.flag)[0]
        tmp.pieces = [queen] + pawns
        camp = optimal_camp_layout(tmp, self.center_hex, ctx.grid,
                                    ctx.terrain.elevations, ctx.fy.tiers, max_ring=3)
        assignments = []
        if camp:
            for piece in tmp.pieces:
                hx = camp['placements'].get(piece.id)
                if hx is not None:
                    self._placements[piece.id] = hx
                    if piece.piece_type == PieceType.PAWN:
                        assignments.append(Assignment(
                            piece_type=PieceType.PAWN, target_hex=hx, role='feeding'))
        return assignments
    
    def execute(self, turn, assignments, ctx):
        if self._done: return
        if ctx.first_call(ctx.queen.id, self):
            qhex = self._placements.get(ctx.queen.id)
            if qhex: ctx.queen.location = qhex
            ctx.set_harvest_give(ctx.queen)
            ctx.log.append(f"t{turn}: Camp opened at ⬡{qhex}")
        for a in assignments:
            piece = ctx.piece_map.get(a.piece_id)
            if not piece: continue
            if ctx.first_call(a.piece_id, self):
                piece.location = a.target_hex
                ctx.set_harvest_give(piece)
        if ctx.queen.food >= ctx.queen.food_capacity * self.target_food_pct:
            self._done = True
            ctx.log.append(f"t{turn}: Camp done, queen food={ctx.queen.food:.1f}")

@dataclass
class PCRTactic(TacticSpec):
    name: str = 'pawn_relay'
    station_hexes: list = field(default_factory=list)
    route_start: int = 0
    route_end: int = 0
    _queen_sent: bool = False
    
    @property
    def requires(self): return {'pawn': len(self.station_hexes)}
    
    def plan(self, ctx):
        return [Assignment(piece_type=PieceType.PAWN, target_hex=h, role='feeding')
                for h in self.station_hexes]
    
    def execute(self, turn, assignments, ctx):
        for a in assignments:
            piece = ctx.piece_map.get(a.piece_id)
            if not piece: continue
            if ctx.first_call(a.piece_id, self):
                ctx.march_to(piece, a.target_hex)
                ctx.log.append(f"t{turn}: {piece.name} → ⬡{a.target_hex} (feeding)")
        
        pawns_ready = all(
            ctx.piece_map[a.piece_id].location == a.target_hex
            for a in assignments if a.piece_id in ctx.piece_map)
        
        if pawns_ready and not self._queen_sent and ctx.queen.food >= ctx.queen.diet * 2:
            ctx.move_queen(self.route_end)
            self._queen_sent = True

@dataclass
class BishopLighthouseTactic(TacticSpec):
    name: str = 'lighthouse'
    lighthouse_hex: int = -1
    sight_range: int = 4
    
    @property
    def requires(self): return {'bishop': 1}
    
    def plan(self, ctx):
        return [Assignment(piece_type=PieceType.BISHOP,
                          target_hex=self.lighthouse_hex, role='lighthouse')]
    
    def execute(self, turn, assignments, ctx):
        for a in assignments:
            piece = ctx.piece_map.get(a.piece_id)
            if not piece: continue
            if ctx.first_call(a.piece_id, self):
                ctx.march_to(piece, a.target_hex)
                ctx.log.append(f"t{turn}: {piece.name} → lighthouse ⬡{a.target_hex}")

# ── Orchestrator ──
class KeyTactics:
    def __init__(self, *tactics, squad, grid, terrain, fy, profile, route=None):
        self.tactics = sorted(tactics, key=lambda t: t.turn)
        self.squad = squad
        self.ctx = TacticContext(squad=squad, grid=grid, terrain=terrain,
                                 fy=fy, profile=profile, route=route or [])
        self._assignments = {}
        self._plan()
    
    def _plan(self):
        claimed = set()
        for i, tactic in enumerate(self.tactics):
            raw = tactic.plan(self.ctx)
            by_type = {}
            for a in raw:
                by_type.setdefault(a.piece_type, []).append(a)
            for ptype, assigns in by_type.items():
                pool = [p for p in self.squad.alive
                        if p.piece_type == ptype and p.id not in claimed]
                for j, a in enumerate(assigns):
                    if j < len(pool):
                        a.piece_id = pool[j].id
                        claimed.add(pool[j].id)
            self._assignments[i] = raw
    
    def at_turn(self, turn):
        active = []
        for i, t in enumerate(self.tactics):
            if t.turn <= turn < t.end_turn:
                active.append((t, self._assignments.get(i, [])))
        return active
    
    @property
    def total_turns(self):
        return max(t.end_turn for t in self.tactics) if self.tactics else 0
    
    def summary(self):
        print(f"🎯 KeyTactics: {len(self.tactics)} tactics, ~{self.total_turns} turns")
        for i, t in enumerate(self.tactics):
            assigns = self._assignments.get(i, [])
            pieces = []
            for a in assigns:
                p = next((p for p in self.squad.alive if p.id == a.piece_id), None)
                pieces.append(f"{p.name if p else '?'}→⬡{a.target_hex}({a.role})")
            print(f"  {i}: {t}  [{', '.join(pieces)}]")

class TacticCallback:
    def __init__(self, kt, grid, elevations, sight_range=4):
        self.kt = kt
        self.queen = kt.ctx.queens[0]
        self.sight_range = sight_range
        self.exec_ctx = ExecContext(
            queen=self.queen, queen_w=0, route=kt.ctx.route,
            grid=grid, elevations=elevations,
            piece_map={p.id: p for p in kt.squad.alive},
            initialized={}, log=[])
        self.turn = 0
    
    def _update_queen_w(self):
        for i, rh in enumerate(self.exec_ctx.route):
            if rh == self.queen.location:
                self.exec_ctx.queen_w = i
    
    def __call__(self, report):
        food_redistribute_ranged(report, self.sight_range)
        self._update_queen_w()
        self.exec_ctx.turn = self.turn
        for tactic, assignments in self.kt.at_turn(self.turn):
            tactic.execute(self.turn, assignments, self.exec_ctx)
        self.turn += 1
```

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

> **Note:** The context window truncated earlier messages (97 → 8), so I'm working from the journal note and this code.

This is a *really* nice pattern. It's essentially the missing middle layer between "here's a corridor" and "here's a running simulator." Let me break down what I think works well and what we could tighten up for the generalized API.

**What's great here:**

- **Declarative `requires` + `plan()`, imperative `execute()`** — clean two-phase lifecycle. Planning is pure, execution mutates.
- **`Assignment` as the contract** — tactics don't hold pieces directly, they request slots by type. The orchestrator does allocation. This is exactly the `ResourcePool` idea.
- **`first_call()` guard** — elegant way to handle one-time init without a separate "setup" phase.
- **`TacticCallback` bridges to `FoodSimulator`** — tactics don't know about the sim, the sim doesn't know about tactics. Clean seam.
- **No `isinstance`** — it compares `piece_type` attributes, which is duck-type friendly. 👍

**What I'd tighten for the generalized version:**

1. **Tactic dependencies** — right now timing is manual (`turn=0`, `turn=8`). The camp *must* finish before the relay starts, but that's implicit. A dependency graph would let tactics declare:

```python
class PCRTactic(TacticSpec):
    depends_on: list[str] = field(default_factory=list)  # ['camp']
```

Then `KeyTactics._plan()` topologically sorts and auto-computes start turns.

2. **Allocation is greedy** — `_plan()` assigns pieces first-come-first-served. For the multi-crossing case, two PCRTactics competing for 5 pawns from a pool of 7 would silently under-allocate. A `ResourcePool` with explicit `allocate/release` + validation would catch this:

```python
pool.allocate(tactic_a)  # claims 3 pawns → ok
pool.allocate(tactic_b)  # wants 5 pawns, only 4 free → raises InsufficientResources
```

3. **Tactic completion** — `CampTactic` has `_done`, but `PCRTactic` doesn't signal completion. A uniform `@property def is_complete(self) -> bool` would let the orchestrator release resources and trigger dependents.

4. **Multi-piece crossings** — `ExecContext.move_queen()` is queen-specific. For convoys, this generalizes to `move_piece(piece, route_end)` and the tactic decides *which* piece moves when.

**How it maps to the API sketch:**

| Tactic | API Layer |
|--------|-----------|
| `CampTactic` | `network.plan_crossing()` auto-generates these for origin/dest |
| `PCRTactic` | `CorridorNetwork.add()` infrastructure — corridor *is* a permanent PCR |
| `BishopLighthouseTactic` | `network.optimize_lighthouses()` — shared across corridors |
| `KeyTactics` | The `MissionPlanner` — sequences tactics for a specific crossing |
| `TacticCallback` | `plan.simulate()` wires this up internally |

**The composable version might look like:**

```python
# Tactics compose into a Mission
mission = Mission("Betty to city B")
mission.add(CampTactic(center=route[0], role='origin'))
mission.add(CampTactic(center=route[-1], role='dest', depends_on=['origin']))
mission.add(PCRTactic(stations=[248], depends_on=['origin']))
mission.add(BishopLighthouseTactic(hex=175))
mission.add(CrossTactic(piece=queen, depends_on=['pawn_relay']))

# Mission resolves dependencies, allocates from pool, simulates
plan = mission.resolve(pool, ctx)
plan.simulate(60).summary()
```

The beautiful thing is your `TacticSpec` base class barely changes — it already has the right shape. The work is in `KeyTactics` becoming smarter about dependency resolution and resource negotiation.

Want to start fleshing out the dependency graph, or tackle `ResourcePool` first?

!cat ../../../HexMagic/overlay.py

The idea would be to have something like the overlay system for tactics. but this get us to a perhaps earlier thing. we want to have overlays on the map for what we have built.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes — that's the right order. Visualize first, then generalize. And the Overlay pattern is a perfect fit because it's already exactly the right shape: **declarative spec + context + composable rendering**.

Here's how corridor/crossing overlays map onto `OverlaySpec`:

```python
def CorridorOverlay(plan, **kw) -> OverlaySpec:
    """Draw corridor route, colored by food safety per hex."""
    def render(ctx: OverlayContext) -> str:
        grid = ctx.grid
        route = plan.corridor.hexes
        parts = []
        
        # Route line — color each segment by min food at that waypoint
        for i in range(len(route) - 1):
            a = grid.hexes[route[i]].center
            b = grid.hexes[route[i+1]].center
            # green=safe, yellow=thin, red=danger
            color = _food_color(plan, i)
            parts.append(
                f'<line x1="{a.x:.1f}" y1="{a.y:.1f}" '
                f'x2="{b.x:.1f}" y2="{b.y:.1f}" '
                f'stroke="{color}" stroke-width="3" opacity="0.8" '
                f'stroke-linecap="round"/>')
        
        # Waypoint dots
        for i, h in enumerate(route):
            c = grid.hexes[h].center
            parts.append(
                f'<circle cx="{c.x:.1f}" cy="{c.y:.1f}" r="3" '
                f'fill="{_food_color(plan, i)}" opacity="0.9"/>')
        
        return '\n'.join(parts)
    
    return OverlaySpec("corridor", render, priority=70)


def CampOverlay(plan, **kw) -> OverlaySpec:
    """Highlight origin/dest camp hexes with pawn icons."""
    def render(ctx: OverlayContext) -> str:
        grid = ctx.grid
        parts = []
        for r in plan.roster:
            if r['role'] not in ('origin_camp', 'advance_party'): continue
            h = r['hex']
            c = grid.hexes[h].center
            color = '#2ecc71' if r['role'] == 'origin_camp' else '#e67e22'
            # Ring marker
            parts.append(
                f'<circle cx="{c.x:.1f}" cy="{c.y:.1f}" r="8" '
                f'fill="none" stroke="{color}" stroke-width="2.5" opacity="0.9"/>')
            # Label
            parts.append(
                f'<text x="{c.x:.1f}" y="{c.y - 12:.1f}" '
                f'text-anchor="middle" font-size="7" fill="{color}" '
                f'font-weight="bold">{r["role"][:3]}</text>')
        return '\n'.join(parts)
    
    return OverlaySpec("camps", render, priority=75)


def LighthouseOverlay(plan, **kw) -> OverlaySpec:
    """Bishop lighthouse positions with sight cone arcs."""
    def render(ctx: OverlayContext) -> str:
        grid = ctx.grid
        parts = []
        for r in plan.roster:
            if r['role'] != 'lighthouse': continue
            h = r['hex']
            c = grid.hexes[h].center
            # Bishop marker — diamond
            s = 6
            parts.append(
                f'<polygon points="{c.x},{c.y-s} {c.x+s},{c.y} '
                f'{c.x},{c.y+s} {c.x-s},{c.y}" '
                f'fill="#9b59b6" opacity="0.85" stroke="white" stroke-width="1"/>')
            
            # Sight coverage — highlight covered route hexes
            covered = r.get('covers', [])
            for ri in covered:
                rh = plan.corridor.hexes[ri]
                rc = grid.hexes[rh].center
                parts.append(
                    f'<line x1="{c.x:.1f}" y1="{c.y:.1f}" '
                    f'x2="{rc.x:.1f}" y2="{rc.y:.1f}" '
                    f'stroke="#9b59b6" stroke-width="1" opacity="0.3" '
                    f'stroke-dasharray="4,3"/>')
        return '\n'.join(parts)
    
    return OverlaySpec("lighthouses", render, priority=80)


def StationOverlay(plan, **kw) -> OverlaySpec:
    """Pawn station hexes along the corridor."""
    def render(ctx: OverlayContext) -> str:
        grid = ctx.grid
        parts = []
        for r in plan.roster:
            if r['role'] != 'station': continue
            c = grid.hexes[r['hex']].center
            parts.append(
                f'<rect x="{c.x-5:.1f}" y="{c.y-5:.1f}" width="10" height="10" '
                f'rx="2" fill="#1abc9c" opacity="0.85" stroke="white" stroke-width="1"/>')
        return '\n'.join(parts)
    
    return OverlaySpec("stations", render, priority=76)


# ── Composite: show everything for a plan ──
def CrossingPlanOverlay(plan, **kw) -> list[OverlaySpec]:
    """Returns all overlays for a crossing plan. Splat into TerrainDisplay."""
    return [
        CorridorOverlay(plan),
        CampOverlay(plan),
        StationOverlay(plan),
        LighthouseOverlay(plan),
    ]
```

Usage would be:

```python
TerrainDisplay(
    CreamOverlay(stylized=True),
    RiverOverlay(),
    *CrossingPlanOverlay(plan),    # splat the list
    terrain=myStuff.terr,
    region=corridor_region,
    radius=12,
)
```

The key parallel to the Tactics idea:

| Overlay System | Tactic System |
|---|---|
| `OverlaySpec(name, renderer, requires, priority)` | `TacticSpec(name, plan, execute, requires, turn)` |
| `OverlayContext(terrain, grid, builder, extras)` | `TacticContext(squad, grid, terrain, fy, profile)` |
| `TerrainDisplay(*overlays, terrain=...)` | `Mission(*tactics, pool=..., ctx=...)` |
| Composable via `*` splat | Composable via dependency graph |
| Pure rendering — no mutation | Two-phase: plan (pure) → execute (mutation) |

The overlay system is **read-only visualization** of the same data structures the tactic system **writes to**. Same shape, different direction.

Want to wire these up against your actual `plan` object and see the corridor on the map?

In [ ]:
def CorridorOverlay(plan, **kw) -> OverlaySpec:
    """Draw corridor route, colored by food safety per hex."""
    def render(ctx: OverlayContext) -> str:
        grid = ctx.grid
        route = plan.corridor.hexes
        parts = []
        
        # Route line — color each segment by min food at that waypoint
        for i in range(len(route) - 1):
            a = grid.hexes[route[i]].center
            b = grid.hexes[route[i+1]].center
            # green=safe, yellow=thin, red=danger
            color = _food_color(plan, i)
            parts.append(
                f'<line x1="{a.x:.1f}" y1="{a.y:.1f}" '
                f'x2="{b.x:.1f}" y2="{b.y:.1f}" '
                f'stroke="{color}" stroke-width="3" opacity="0.8" '
                f'stroke-linecap="round"/>')
        
        # Waypoint dots
        for i, h in enumerate(route):
            c = grid.hexes[h].center
            parts.append(
                f'<circle cx="{c.x:.1f}" cy="{c.y:.1f}" r="3" '
                f'fill="{_food_color(plan, i)}" opacity="0.9"/>')
        
        return '\n'.join(parts)
    
    return OverlaySpec("corridor", render, priority=70)


def CampOverlay(plan, **kw) -> OverlaySpec:
    """Highlight origin/dest camp hexes with pawn icons."""
    def render(ctx: OverlayContext) -> str:
        grid = ctx.grid
        parts = []
        for r in plan.roster:
            if r['role'] not in ('origin_camp', 'advance_party'): continue
            h = r['hex']
            c = grid.hexes[h].center
            color = '#2ecc71' if r['role'] == 'origin_camp' else '#e67e22'
            # Ring marker
            parts.append(
                f'<circle cx="{c.x:.1f}" cy="{c.y:.1f}" r="8" '
                f'fill="none" stroke="{color}" stroke-width="2.5" opacity="0.9"/>')
            # Label
            parts.append(
                f'<text x="{c.x:.1f}" y="{c.y - 12:.1f}" '
                f'text-anchor="middle" font-size="7" fill="{color}" '
                f'font-weight="bold">{r["role"][:3]}</text>')
        return '\n'.join(parts)
    
    return OverlaySpec("camps", render, priority=75)


def LighthouseOverlay(plan, **kw) -> OverlaySpec:
    """Bishop lighthouse positions with sight cone arcs."""
    def render(ctx: OverlayContext) -> str:
        grid = ctx.grid
        parts = []
        for r in plan.roster:
            if r['role'] != 'lighthouse': continue
            h = r['hex']
            c = grid.hexes[h].center
            # Bishop marker — diamond
            s = 6
            parts.append(
                f'<polygon points="{c.x},{c.y-s} {c.x+s},{c.y} '
                f'{c.x},{c.y+s} {c.x-s},{c.y}" '
                f'fill="#9b59b6" opacity="0.85" stroke="white" stroke-width="1"/>')
            
            # Sight coverage — highlight covered route hexes
            covered = r.get('covers', [])
            for ri in covered:
                rh = plan.corridor.hexes[ri]
                rc = grid.hexes[rh].center
                parts.append(
                    f'<line x1="{c.x:.1f}" y1="{c.y:.1f}" '
                    f'x2="{rc.x:.1f}" y2="{rc.y:.1f}" '
                    f'stroke="#9b59b6" stroke-width="1" opacity="0.3" '
                    f'stroke-dasharray="4,3"/>')
        return '\n'.join(parts)
    
    return OverlaySpec("lighthouses", render, priority=80)


def StationOverlay(plan, **kw) -> OverlaySpec:
    """Pawn station hexes along the corridor."""
    def render(ctx: OverlayContext) -> str:
        grid = ctx.grid
        parts = []
        for r in plan.roster:
            if r['role'] != 'station': continue
            c = grid.hexes[r['hex']].center
            parts.append(
                f'<rect x="{c.x-5:.1f}" y="{c.y-5:.1f}" width="10" height="10" '
                f'rx="2" fill="#1abc9c" opacity="0.85" stroke="white" stroke-width="1"/>')
        return '\n'.join(parts)
    
    return OverlaySpec("stations", render, priority=76)


# ── Composite: show everything for a plan ──
def CrossingPlanOverlay(plan, **kw) -> list[OverlaySpec]:
    """Returns all overlays for a crossing plan. Splat into TerrainDisplay."""
    return [
        CorridorOverlay(plan),
        CampOverlay(plan),
        StationOverlay(plan),
        LighthouseOverlay(plan),
    ]


can we build an overlay demo for the queens crossing?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Sure! We need the `_food_color` helper, then we can render it. Let me also handle the case where `covers` might not be on every roster entry.

```python
# ── Helper for corridor coloring ──
def _food_color(plan, waypoint_idx):
    """Color a route waypoint by food tier of its hex."""
    h = plan.corridor.hexes[waypoint_idx]
    tier = fy.tiers.get(h, 0)
    if tier >= 5: return '#2ecc71'   # green — pawn-sustainable
    if tier >= 3: return '#f1c40f'   # yellow — marginal
    return '#e74c3c'                  # red — barren/gap

# ── Render the crossing plan on the map ──
route = plan.corridor.hexes
mid_hex = route[len(route) // 2]

TerrainDisplay(
    CreamOverlay(stylized=True),
    RiverOverlay(),
    *CrossingPlanOverlay(plan),
    terrain=myStuff.terr,
    region=mid_hex,
    radius=14,
)
```

This should give you the full corridor rendered on the terrain map:
- **Colored route line**: green segments where food tier ≥ 5, yellow for marginal, red for barren gaps
- **Green rings**: origin camp pawns (⬡179, ⬡159)
- **Orange rings**: advance party / dest camp pawns
- **Teal squares**: station pawn at ⬡248
- **Purple diamonds**: bishop lighthouses with dashed sight lines to covered route hexes

If `CreamOverlay` or `RiverOverlay` aren't imported, just drop them — the plan overlays will still render on whatever base you have.

In [ ]:
# ── Helper for corridor coloring ──
def _food_color(plan, waypoint_idx):
    """Color a route waypoint by food tier of its hex."""
    h = plan.corridor.hexes[waypoint_idx]
    tier = fy.tiers.get(h, 0)
    if tier >= 5: return '#2ecc71'   # green — pawn-sustainable
    if tier >= 3: return '#f1c40f'   # yellow — marginal
    return '#e74c3c'                  # red — barren/gap

# ── Render the crossing plan on the map ──
route = plan.corridor.hexes
mid_hex = route[len(route) // 2]

TerrainDisplay(
    CreamOverlay(stylized=True),
    RiverOverlay(),
    *CrossingPlanOverlay(plan),
    terrain=myStuff.terr,
    region=mid_hex,
    radius=14,
)


can queen coverage propose a budget of how many pieces it would need for a queen to walk a corridor without dieing the plan should back info on what it takes  I don't have the queen coverage doing the schedule, but it might need a food simulator to check things. Thoughts. The planner should give a schedule based on the resources.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes — the right architecture is **plan deterministically, verify with sim**. Here's how I'd structure it:

```python
@dataclass
class CrossingPlan:
    """Complete resource budget + schedule for a queen to walk a corridor."""
    corridor:         Corridor
    roster:           list[dict]       # [{piece_type, hex, facing, instructions, role}]
    queen_start_turn: int
    schedule:         pd.DataFrame     # validated budget schedule
    feasible:         bool
    queen_arrival:    int              # turn queen reaches end
    food_margin:      float            # min food queen has at any point
    bishop_fuel_used: float
    
    def pieces_needed(self):
        """Summary: how many of each type."""
        from collections import Counter
        return Counter(r['piece_type'] for r in self.roster if r['role'] != 'queen')
    
    def __ft__(self):
        needed = self.pieces_needed()
        summary = ', '.join(f"{v}× {k.name}" for k, v in needed.items())
        status = '✅' if self.feasible else '❌'
        
        info = Table(Tbody(
            Tr(Td(B('Route')),    Td(f'⬡{self.corridor.start}→⬡{self.corridor.end} ({self.corridor.length} hexes)')),
            Tr(Td(B('Roster')),   Td(summary or 'queen only')),
            Tr(Td(B('Queen go')), Td(f'turn {self.queen_start_turn}')),
            Tr(Td(B('Arrival')),  Td(f'turn {self.queen_arrival}')),
            Tr(Td(B('Food min')), Td(f'{self.food_margin:.1f}')),
            Tr(Td(B('Bishop fuel')), Td(f'{self.bishop_fuel_used:.0f}')),
            Tr(Td(B('Status')),   Td(status)),
        ), cls=TableT.sm)
        
        timetable = schedule_timetable(self.schedule, title='Crossing Schedule')
        return Div(
            Card(info, header=CardTitle(f'Crossing Plan {status}')),
            timetable,
        )
```

And the method on `QueenCoverage`:

```python
@patch
def propose_crossing(self: QueenCoverage, corridor: Corridor,
                     queen: Piece,
                     available_pawns: int = 4,
                     available_bishops: int = 2,
                     max_retries: int = 3,
                     n_turns: int = 50) -> CrossingPlan:
    """Propose a resource budget that gets the queen across alive.
    
    Flow:
      1. Allocate pawns to stations (from corridor segments)
      2. Allocate bishops to lighthouse gaps
      3. Compute queen start turn (after bishops are parked)
      4. Build shadow pieces with instructions
      5. Run throwaway sim to verify
      6. If queen dies: retry with +1 bishop or +1 wait turn
      7. Return validated CrossingPlan
    """
    route = corridor.hexes
    segments = corridor.segments
    profile = self.profile
    
    for attempt in range(max_retries + 1):
        roster = []
        shadows = []
        
        # ── 1. Queen shadow (instructions set later) ──
        qs = PieceShadow.from_piece(queen)
        qs.location = route[0]
        qs.food = profile['queen_capacity']
        roster.append(dict(piece_type=PieceType.QUEEN, hex=route[0],
                           facing=qs.facing, role='queen'))
        
        # ── 2. Pawns at stations ──
        pawn_id = 0
        for seg in segments:
            if seg['type'] != 'station': continue
            for hex_idx in seg['hexes'][:available_pawns - pawn_id]:
                mid_w = (seg['min_w'] + seg['max_w']) // 2
                facing = _facing_toward(self.grid, hex_idx, route[mid_w])
                
                ps = PieceShadow(
                    id=f'plan_pawn_{pawn_id}', name=f'Pawn_{pawn_id}',
                    piece_type=PieceType.PAWN, birth_year=0,
                    location=hex_idx, facing=facing,
                    food=profile['pawn_capacity'], food_capacity=profile['pawn_capacity'],
                    diet=profile['pawn_diet'], health=100, max_health=100,
                    move_strength=2, harvest_strength=profile['pawn_harvest'],
                    sight=profile['pawn_sight'],
                )
                ps.instructions = InstructionList(
                    [Instruction.HARVEST.value, Instruction.GIVE.value],
                    cursor=0, patrol=True)
                
                shadows.append(ps)
                roster.append(dict(piece_type=PieceType.PAWN, hex=hex_idx,
                                   facing=facing, role='station'))
                pawn_id += 1
        
        # ── 3. Bishops at lighthouses ──
        all_gap_indices = []
        for seg in segments:
            if seg['type'] == 'caravan':
                all_gap_indices.extend(range(seg['gap_start'], seg['gap_end'] + 1))
        
        n_bishops = min(available_bishops, max(1, attempt + 1))  # retry adds bishops
        bishop_ready_turn = 0
        
        if all_gap_indices:
            lh = optimal_lighthouse_placement(
                all_gap_indices, route, self.grid, self.elevations,
                n_bishops=n_bishops, profile=profile)
            
            if lh and lh['feasible']:
                for pl in lh['placements']:
                    lh_hex = pl['hex']
                    bishop_path = simple_hex_path(route[0], lh_hex,
                                                  self.grid, self.elevations)
                    if not bishop_path:
                        continue
                    
                    sprint_rules, end_facing = route_to_instructions(
                        bishop_path, self.grid, 0)
                    
                    gap_mid = route[(min(pl['serves_route_w']) + max(pl['serves_route_w'])) // 2]
                    face_route = _facing_toward(self.grid, lh_hex, gap_mid)
                    diff = (face_route - end_facing) % 6
                    rot = ([Instruction.ROT_R.value] * diff if diff <= 3
                           else [Instruction.ROT_L.value] * (6 - diff))
                    
                    park = [Instruction.HARVEST.value, Instruction.GIVE.value] * 60
                    
                    bs = PieceShadow(
                        id=f'plan_bishop_{pl["bishop"]}',
                        name=f'Bishop_{pl["bishop"]}',
                        piece_type=PieceType.BISHOP, birth_year=0,
                        location=route[0], facing=0,
                        food=profile['bishop_capacity'],
                        food_capacity=profile['bishop_capacity'],
                        diet=profile['bishop_diet'], health=100, max_health=100,
                        move_strength=3, harvest_strength=profile.get('bishop_harvest', 0.7),
                        sight=profile['pawn_sight'],
                    )
                    bs.instructions = InstructionList(
                        sprint_rules + rot + park, cursor=0, patrol=False)
                    
                    shadows.append(bs)
                    roster.append(dict(piece_type=PieceType.BISHOP, hex=lh_hex,
                                       facing=face_route, role='lighthouse'))
                    
                    arrival = math.ceil(len(bishop_path) / 3) + 2  # +rot +safety
                    bishop_ready_turn = max(bishop_ready_turn, arrival)
        
        # ── 4. Queen start turn + instructions ──
        queen_start = bishop_ready_turn + attempt  # retry adds wait
        wait = [Instruction.PAUSE.value] * queen_start
        march, _ = route_to_instructions(route, self.grid, qs.facing)
        qs.instructions = InstructionList(wait + march, cursor=0, patrol=False)
        shadows.insert(0, qs)
        
        # ── 5. Run shadow sim ──
        initial = {s.id: dict(location=s.location, facing=s.facing,
                               food=s.food, health=s.health)
                   for s in shadows}
        
        sim = FoodSimulator(
            grid=self.grid, elevations=self.elevations,
            food_tiers=self.food_tiers, pieces=shadows,
            countries=self._countries(),
        )
        sim.run(num_turns=n_turns)
        
        # ── 6. Check queen survived + arrived ──
        q_snap = [s['piece_states'].get(qs.id) for s in sim.snapshots]
        arrived = any(s and s['location'] == route[-1] for s in q_snap)
        alive = q_snap[-1] and q_snap[-1]['health'] > 0 if q_snap else False
        min_food = min((s['food'] for s in q_snap if s and s['health'] > 0),
                       default=0)
        
        # Find arrival turn
        arrival_turn = n_turns
        for i, s in enumerate(q_snap):
            if s and s['location'] == route[-1]:
                arrival_turn = i + 1
                break
        
        # Bishop fuel used
        bishop_fuel = sum(
            profile['bishop_capacity'] - (q_snap[-1] or {}).get('food', 0)
            for s_id in [s.id for s in shadows if s.piece_type == PieceType.BISHOP]
            for q_snap_last in [sim.snapshots[-1]['piece_states'].get(s_id)]
            if q_snap_last
        ) if sim.snapshots else 0
        
        # Extract schedule
        adapter = SimAdapter(sim, corridors=[corridor])
        schedule = adapter.extract(schedule_id=f'crossing_v{attempt}',
                                    initial_states=initial)
        
        feasible = arrived and alive
        plan = CrossingPlan(
            corridor=corridor, roster=roster,
            queen_start_turn=queen_start, schedule=schedule,
            feasible=feasible, queen_arrival=arrival_turn,
            food_margin=min_food, bishop_fuel_used=bishop_fuel,
        )
        
        if feasible:
            print(f"✅ Crossing plan (attempt {attempt}): "
                  f"{plan.pieces_needed()}, queen starts t{queen_start}, "
                  f"arrives t{arrival_turn}, min_food={min_food:.1f}")
            return plan
        else:
            reason = 'died' if not alive else 'never arrived'
            print(f"❌ Attempt {attempt}: {reason} "
                  f"(min_food={min_food:.1f}, start=t{queen_start})")
    
    print(f"⚠️ All {max_retries + 1} attempts failed")
    return plan  # return last attempt even if infeasible
```

The key insight: **propose, simulate, verify, retry**. The loop is:

1. **Deterministic allocation** — pawns at stations, bishops at lighthouses, queen waits
2. **Shadow sim** — runs `FoodSimulator` on `PieceShadow`s, originals untouched
3. **Check** — did queen arrive alive? What was her min food?
4. **Retry** — add a bishop or extra wait turns, re-simulate

Each retry is cheap (shadow sim on ~5 pieces, ~50 turns). The `CrossingPlan` comes back with a complete schedule and resource manifest. The caller can `show(plan)` to see the timetable + summary.

The plan doesn't need to be *inside* the corridor — it sits alongside it. Corridor says "what the route looks like", `CrossingPlan` says "what it takes to walk it".

The only question is can we use the same pawns at both ends, but in general yes have the camp support be part of the propose_crossing. We have to be able to live in place while the route is formed and land with an approiate cover.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

*[Response interrupted]*